# PersonaPlex Student — Distillation Training on the Generated Dataset

Trains the distilled PersonaPlex student (`student_ppx_s`: 20 layers, 2048-d, GQA, ~0.8 B own parameters +
the teacher's frozen depth transformer / heads) on the dataset made by
`PersonaPlex_Student_Dataset_Generation.ipynb`. It runs on **one or many GPUs** (PyTorch DDP via `torchrun`).

## How this training works (read once)
Each training sample is a full dual-channel conversation `[17, 1000]`:
- **row 0:** the agent's text;
- **rows 1–8:** the agent's audio codes;
- **rows 9–16:** the user's audio codes;
- frames `[0, prefix_len)` hold the voice + persona prompt, exactly as at inference.

For every batch:
1. **The teacher (PersonaPlex-7B, frozen, bf16)** runs `forward_train` on the sequence and produces its
   predictions (text + audio logits) plus hidden states.
2. **The student** runs the same forward pass and is trained to match the teacher: **KL** on text + the agent's
   8 codebooks, **CE** on the real conversation frames, **bridge** (student output vs. teacher hidden state) and
   **hidden** (layer-to-layer) losses. Loss weights follow the 4-phase schedule:

| Phase | Share of steps | Data | Emphasis |
|---|---|---|---|
| P1 Align | 0–15 % | teacher-forced | bridge + hidden (make the student's internals teacher-like) |
| P2 Behavior | 15–60 % | teacher-forced | KL (imitate the teacher's predictions) |
| P3 On-policy | 60–90 % | student rollouts (share `ONPOLICY_FRACTION`) | KL on states the student itself reaches |
| P4 Polish | 90–100 % | rollouts | + depth transformer unfrozen at 0.1× LR |

In an **on-policy rollout** the student continues a real conversation by itself (sampling like at inference)
while the user side keeps playing the *recorded* user audio. The teacher then scores every frame the student
produced. This is what teaches the student to recover from its own mistakes.

## What was fixed in `distill/train.py` for this dataset (verified against the code)

| # | Old behaviour | Now |
|---|---|---|
| 1 | single GPU | DDP multi-GPU + gradient accumulation |
| 2 | fixed sample order, never shuffled | per-epoch shuffle from (seed, step) → exact resume |
| 3 | rollout started at `chunk_frames//4` = 250, **inside the prompt** (prefix is up to 256 frames) | starts after the longest prefix in the batch + context |
| 4 | rollout fed `SINE_TOKENS` as the user | keeps feeding the **recorded** user channel |
| 5 | rollout dropped frame 0 → everything shifted by one frame | aligned exactly like `LMGen`; forced frames checked against the data |
| 6 | CE on the student's own rollout samples | CE only on real-data frames |
| 7 | CE on prompt frames (forced tokens) | CE excluded on the prefix (KL/bridge/hidden keep it) |
| 8 | KL computed in bf16 over a 32 000-token vocab | fp32 |
| 9 | bf16 weights **and** bf16 AdamW states | fp32 master weights + bf16 autocast (or legacy bf16 on small GPUs) |
| 10 | constant LR; weight decay on norms/embeddings too | warmup + cosine; decay only on matrices |
| 11 | P4 depformer training dropped at save/export | saved + exported |
| 12 | teacher user-stream logits distilled (never used at inference; possibly untrained copied heads) | text + agent codebooks only |
| 13 | no validation, no best checkpoint, CSV only | val loop, best/latest/periodic checkpoints, full logs |

## Run order
1. **Smoke run first** (`SMOKE_TEST = True`, the default). About 12 real steps through every phase, including
   on-policy rollouts, validation, checkpoint, export, and a **resume test**, with *your* batch settings on
   *your* GPUs. It prints PASS/FAIL, peak memory and a **measured estimate of the full training time**.
2. **Full run:** set `SMOKE_TEST = False`, restart the kernel, run from Section 1. Training runs detached; the
   Monitor cell can be re-run anytime, and training resumes automatically after a crash or pod restart.

## Logs (`<OUTPUT_DIR>/`)
| File | Contents |
|---|---|
| `logs/train_rank<N>.log` | full text log of every GPU process |
| `train_log.csv` | per-step losses, LR, gradient norm, step time, throughput, peak memory |
| `logs/val_log.csv` | validation KL / CE / agreement per evaluation |
| `logs/events.jsonl` | phase changes, checkpoints, resume, skipped steps, export |
| `logs/metrics.jsonl` | every logged row as JSON |
| `logs/heartbeat.json` | current step, ETA, peak memory |
| `logs/run_config.json`, `logs/data_report.json` | exact arguments, environment, git revision, dataset analysis |
| `logs/launcher.log` | torchrun output |
| `logs/run_summary.json` | end-of-run summary |

## 1. Environment check

In [ ]:
import os, sys, json, shutil, subprocess, pathlib, platform
print("Python", sys.version.split()[0], "|", platform.platform())
assert sys.version_info >= (3, 10)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. CONFIG
`"auto"` memory settings are **found by the smoke run** (Section 8): it tries a ladder of micro-batch / gradient
checkpointing / precision settings and saves the first that passes every phase with headroom as a memory plan
(`<OUTPUT_ROOT>/memory_plan_<RUN_NAME>.json`), which the full run then uses.

In [ ]:
import os, sys, json, shutil, subprocess, pathlib, math

# ==== WHAT TO RUN ======================================================================================
SMOKE_TEST       = True     # True: ~12-step full-pipeline check (+ resume test). False: the real training.
# =======================================================================================================

WORKSPACE        = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL         = "https://github.com/MoshiHead/personaplex-helium-distillation-v5.git"
REPO_DIR         = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR     = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID       = "nvidia/personaplex-7b-v1"

# ---- data (output of PersonaPlex_Student_Dataset_Generation.ipynb) --------------------------------------
DATASET_ROOT     = os.path.join(WORKSPACE, "ppx_student_dataset_v1")
SMOKE_ON_SMOKE_DATASET = True   # smoke run on the small smoke dataset (fast); False = smoke on the real dataset
if SMOKE_TEST and SMOKE_ON_SMOKE_DATASET and os.path.isdir(os.path.join(WORKSPACE, "ppx_student_dataset_smoke", "train")):
    DATASET_ROOT = os.path.join(WORKSPACE, "ppx_student_dataset_smoke")

# ---- run -------------------------------------------------------------------------------------------------
# student_ppx_s converged at val_agree_text_content 0.459 -- ~2 correct content words in a row, which is
# not enough to carry a 15-word factual answer. student_ppx_m keeps 36.4% of the teacher's FFN memory
# slots instead of 20.8% and pools 4 KV heads instead of 8; projected RTF 0.37 vs the measured 0.33.
STUDENT_CONFIG   = "student_ppx_m"
RUN_NAME         = "student_ppx_m_v1"
OUTPUT_ROOT      = os.path.join(WORKSPACE, "distill_runs")
OUTPUT_DIR       = os.path.join(OUTPUT_ROOT, RUN_NAME + ("_smoke" if SMOKE_TEST else ""))

# ---- hardware / batch --------------------------------------------------------------------------------------
NUM_GPUS         = None     # None = all visible GPUs
MICRO_BATCH      = "auto"   # samples per GPU per forward (auto: the smoke run finds the largest that fits)
GLOBAL_BATCH     = 32       # samples per optimizer step (rounded up to a multiple of GPUs x MICRO_BATCH)
MASTER_DTYPE     = "auto"   # "fp32" (recommended, needs ~70+ GB) | "bf16" (legacy, fits ~32-48 GB) | auto
GRAD_CHECKPOINTING = "auto" # recompute student activations (saves memory, ~25% slower)

# ---- optimisation -------------------------------------------------------------------------------------------
EPOCHS           = 40       # passes over train/ ; TOTAL_STEPS = ceil(EPOCHS * N_train / GLOBAL_BATCH)
                            # was 6 -> 925 steps / ~8.6 GPU-h, which is nowhere near enough for the student to
                            # relearn the teacher's text behaviour after the init's width/depth/FFN truncation.
                            # Every val row of that run was still "best", grad_norm was still clipping at 1.0 at
                            # the final step: the run ended while still steeply descending.
LR               = 2e-4
MIN_LR_RATIO     = 0.1
WEIGHT_DECAY     = 0.01
ONPOLICY_FRACTION = 0.10    # was 0.5. On-policy steps measured 88.1 s/step and 188 frames/s vs 20.6 s and
                            # 1600 frames/s teacher-forced -- they ate ~half the wall clock for ~11% of the
                            # frames, to improve turn-taking, which already worked.
ROLLOUT_FRAMES   = 150      # was 250
ONPOLICY_CONTEXT = 50       # real conversation frames after the prompt before the student takes over
VAL_SAMPLES      = 256
SEED             = 1234

# ---- text-only distillation: where the truncated knowledge actually comes back -----------------------------
# The audio corpus holds well under 1M agent text tokens in total (~90% of its text stream is PAD/EPAD), so it
# cannot teach "what is Bitcoin". Text-only steps feed the SAME temporal transformer one position per token
# with no Mimi and no depformer, so orders of magnitude more text fits in the same budget. The corpus only has
# to COVER the domain -- the teacher supplies the targets, so it needs no answers in it.
# Point TEXT_DATA at .txt/.md/.jsonl/.json files or directories (general prose + crypto/blockchain/finance +
# whatever identity facts your personas assert). Set to None to train audio-only, exactly as before.
TEXT_DATA        = [os.path.join(WORKSPACE, "ppx_text_corpus")]
TEXT_FRACTION    = 0.25     # share of TEACHER-FORCED steps that run as text-only steps (was 0.5)
TEXT_SEQ_LEN     = 512      # text tokens per sequence (must be <= the student's max_frames)
TEXT_CE_WEIGHT   = 0.2
TEXT_MAX_TOKENS  = None     # cap the corpus for a quick experiment

# ---- text-loss weighting (fixes the ~90%-of-the-gradient-is-padding problem) --------------------------------
TEXT_CONTENT_BOOST = 8.0    # extra weight on text frames carrying a real word; 0 = old behaviour
BETA_TEXT          = 2.0    # text-KL weight (phase.beta now weights the AUDIO KL only)

# ---- optional long-context fine-tune on train_long/ (1500 frames = the student's full context) -------------
LONG_FINETUNE    = True
LONG_EPOCHS      = 2
LONG_LR          = 1e-4

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("MODE        ", "SMOKE TEST" if SMOKE_TEST else "FULL TRAINING")
print("DATASET_ROOT", DATASET_ROOT)
print("OUTPUT_DIR  ", OUTPUT_DIR)

## 3. System + Python dependencies
moshi is installed with `--no-deps` (its old pins conflict with newer `huggingface_hub`/`transformers`). Training
imports `moshi` and `distill/` straight from the repo checkout via `PYTHONPATH`.

In [ ]:
SUDO = "" if os.geteuid() == 0 else "sudo "
!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates ffmpeg libopus-dev libsndfile1
if not (pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
assert (pathlib.Path(REPO_DIR) / "distill").is_dir(), "REPO_DIR must contain distill/"

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q --no-deps "{REPO_DIR}/moshi/."
%pip install -q "numpy<2.2" "sphn>=0.1.4,<0.2" einops sentencepiece safetensors huggingface_hub pyyaml
%pip install -q soundfile scipy pyloudnorm jiwer "transformers>=4.40" psutil pandas

In [ ]:
# GPU / PyTorch check -- runs in FRESH subprocesses (that is how every generation worker runs); the kernel itself
# never initializes CUDA. If PyTorch can't use the GPU, the NVIDIA driver is queried directly (cuInit via ctypes,
# no PyTorch) to get the exact CUDA error code, and the one cause that is fixable inside the container -- a
# libcuda on LD_LIBRARY_PATH that doesn't match the host driver -- is repaired automatically.
import glob, json, os, subprocess, sys

TORCH_PROBE = r'''
import json
r = {}
try:
    import torch
    # compile-time kernel list; unlike torch.cuda.get_arch_list() it does not need a working GPU (that returns [])
    flags = torch._C._cuda_getArchFlags() if hasattr(torch._C, "_cuda_getArchFlags") else ""
    r.update(torch=torch.__version__, cuda=torch.version.cuda, arch=sorted(set((flags or "").split())))
    r["available"] = torch.cuda.is_available()
    if r["available"]:
        r["gpus"] = torch.cuda.device_count()
        r["name"] = torch.cuda.get_device_name(0)
        x = torch.randn(1024, 1024, device="cuda", dtype=torch.bfloat16)
        (x @ x).sum().item()
        r["bf16_matmul_ok"] = True
except Exception as e:
    r["error"] = repr(e)
print("PROBE" + json.dumps(r))
'''

DRIVER_PROBE = r'''
import ctypes, json
r = {}
try:
    lib = ctypes.CDLL("libcuda.so.1")
    v = ctypes.c_int()
    lib.cuDriverGetVersion(ctypes.byref(v))
    r["libcuda_api_version"] = v.value                     # e.g. 13010 = CUDA 13.1
    r["cuInit"] = lib.cuInit(0)                            # 0 = success; otherwise a CUresult code
    n = ctypes.c_int()
    r["cuDeviceGetCount"] = lib.cuDeviceGetCount(ctypes.byref(n))
    r["device_count"] = n.value
    r["libcuda_loaded_from"] = sorted({l.split()[-1] for l in open("/proc/self/maps") if "libcuda.so" in l})
except OSError as e:
    r["load_error"] = str(e)
print("CUPROBE" + json.dumps(r))
'''

CU_ERRORS = {
    0: "success", 3: "CUDA_ERROR_NOT_INITIALIZED", 35: "CUDA_ERROR_INSUFFICIENT_DRIVER", 46: "CUDA_ERROR_DEVICES_UNAVAILABLE",
    100: "CUDA_ERROR_NO_DEVICE", 101: "CUDA_ERROR_INVALID_DEVICE", 802: "CUDA_ERROR_SYSTEM_NOT_READY",
    803: "CUDA_ERROR_SYSTEM_DRIVER_MISMATCH", 804: "CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE", 999: "CUDA_ERROR_UNKNOWN",
}


def run_probe(code, tag, env=None):
    p = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, env=env)
    line = [l for l in p.stdout.splitlines() if l.startswith(tag)]
    r = json.loads(line[-1][len(tag):]) if line else {"error": "probe crashed", "stderr_tail": p.stderr[-1500:]}
    if tag == "PROBE" and not r.get("bf16_matmul_ok"):
        r["stderr_tail"] = p.stderr[-800:]
    return r


def smi(q):
    return subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()


cap = float(smi("compute_cap").split()[0] or 0)
need_arch = f"sm_{int(round(cap * 10))}"               # 12.0 -> sm_120 (RTX 5090 / Blackwell)
r = run_probe(TORCH_PROBE, "PROBE")
print("installed torch:", json.dumps({k: v for k, v in r.items() if k != "stderr_tail"}))

if not r.get("bf16_matmul_ok") and r.get("arch") and need_arch not in r["arch"]:
    # Only reinstall when the installed build is KNOWN to lack kernels for this GPU (e.g. moshi's old torch<2.5 pin);
    # an unknown/empty kernel list (CUDA-less build info) never triggers a reinstall.
    # --upgrade is required: without it pip keeps the old build as "already satisfied" and installs nothing.
    print(f"Installed torch has no {need_arch} kernels -> installing the cu130 build")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cu130"], check=True)
    r = run_probe(TORCH_PROBE, "PROBE")

if not r.get("bf16_matmul_ok"):
    # ---------------- driver-level diagnosis (independent of PyTorch)
    d = run_probe(DRIVER_PROBE, "CUPROBE")
    d["cuInit_meaning"] = CU_ERRORS.get(d.get("cuInit"), "see CUDA driver API CUresult docs")
    host_driver = smi("driver_version")
    all_libcuda = sorted({p for pat in ["/usr/lib*/**/libcuda.so*", "/usr/local/**/libcuda.so*", "/opt/**/libcuda.so*"]
                          for p in glob.glob(pat, recursive=True)})
    facts = {
        "host_driver (nvidia-smi)": host_driver, "gpu": smi("name"), "compute_mode": smi("compute_mode"),
        "gpu_processes": subprocess.run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
                                         "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "none",
        "driver_probe": d, "libcuda_files_in_container": all_libcuda,
        "LD_LIBRARY_PATH": os.environ.get("LD_LIBRARY_PATH"),
        "NVIDIA_DRIVER_CAPABILITIES": os.environ.get("NVIDIA_DRIVER_CAPABILITIES"),
    }
    print("\nDRIVER-LEVEL DIAGNOSIS:\n" + json.dumps(facts, indent=1))

    # ---------------- automatic repair: a libcuda that doesn't belong to the host driver shadows the real one
    # (typical: CUDA forward-compat libs from the image, which GeForce/RTX GPUs do not support -> cuInit fails).
    bad_dirs = sorted({os.path.dirname(p) for p in all_libcuda
                       if "compat" in p or (host_driver and ".so." in p and not p.endswith(host_driver)
                                             and os.path.basename(p).count(".") >= 3)})
    lp = [x for x in (os.environ.get("LD_LIBRARY_PATH") or "").split(":") if x]
    clean_lp = [x for x in lp if not any(x.rstrip("/") == b.rstrip("/") for b in bad_dirs) and "compat" not in x]
    if d.get("cuInit") not in (0, None) and clean_lp != lp:
        env = dict(os.environ, LD_LIBRARY_PATH=":".join(clean_lp))
        r2 = run_probe(TORCH_PROBE, "PROBE", env=env)
        print("retry without mismatched libcuda dirs on LD_LIBRARY_PATH:", r2.get("bf16_matmul_ok", False))
        if r2.get("bf16_matmul_ok"):
            os.environ["LD_LIBRARY_PATH"] = ":".join(clean_lp)   # inherited by every worker launched later
            r = r2
            print("FIXED: removed", sorted(set(lp) - set(clean_lp)), "from LD_LIBRARY_PATH for this session "
                  "(re-run this cell after a kernel restart; it re-applies the fix).")

if not r.get("bf16_matmul_ok"):
    code_ = d.get("cuInit")
    advice = {
        999: ("CUDA_ERROR_UNKNOWN from cuInit() itself, below PyTorch, with the right driver library loaded and "
              "/dev/nvidia-uvm present: the GPU or the host driver is in a bad state (e.g. after an Xid error). "
              "Nothing inside the container can fix that. TERMINATE this pod and DEPLOY A NEW ONE (Stop/Start usually "
              "lands on the same machine and GPU). Data on your /workspace network volume is kept."),
        804: ("The container's libcuda is a CUDA forward-compat library; GeForce/RTX GPUs don't support forward "
              "compatibility. Remove the compat directory from LD_LIBRARY_PATH (listed above) or use an image built "
              "for CUDA <= the host driver's version."),
        803: ("libcuda in the container doesn't match the host kernel driver (see libcuda_files_in_container vs "
              "host_driver). Remove the mismatched libcuda from LD_LIBRARY_PATH / the image, or use another template."),
        46: ("GPU busy/unavailable: check gpu_processes and compute_mode above (Exclusive_Process + another process "
             "holding the GPU). Stop that process or restart the pod."),
        100: ("No CUDA device visible to the driver API although nvidia-smi sees one -- the container was started "
              "without GPU compute capability (check NVIDIA_DRIVER_CAPABILITIES includes 'compute'); redeploy the pod."),
    }.get(code_, "Unexpected driver state; send the DRIVER-LEVEL DIAGNOSIS above. Deploying a new pod is the safest fix.")
    raise RuntimeError(f"PyTorch cannot use the GPU: cuInit() = {code_} ({CU_ERRORS.get(code_, '?')}).\n{advice}")

print(f"\nGPU OK in a fresh process: torch {r['torch']} (CUDA {r['cuda']}), {r['gpus']} x {r['name']}, "
      f"{need_arch} kernels present, bf16 matmul ok.")
if "torch" in sys.modules:
    print("NOTE: this kernel imported torch earlier. Restart the kernel (Kernel > Restart) and continue from "
          "Section 2 so later cells don't use a stale module. Workers are separate processes and are unaffected.")

## 4. Install the updated training code into the repo
The cells below write the updated `distill/` files into `REPO_DIR`, so this pod runs exactly the tested code,
whatever the git checkout contains. The same files are in the repository.

In [ ]:
import os
# %%writefile does not create folders, so every directory the cells below write into must exist first.
for _d in ("distill/data", "distill/configs", "tools"):
    os.makedirs(os.path.join(REPO_DIR, _d), exist_ok=True)
print("writing updated training code into", REPO_DIR)


In [ ]:
%%writefile {REPO_DIR}/distill/train.py
# SPDX-License-Identifier: MIT
"""Distillation trainer (v2): P1 Align -> P2 Behavior -> P3 On-policy -> P4 Polish (distill/schedule.py).

Single GPU:  python -m distill.train --student-config student_ppx_s --teacher-checkpoint model.safetensors \
                 --data-dir <dataset>/train --val-dir <dataset>/val --output-dir <run> --chunk-frames 1000 ...
Multi GPU:   torchrun --nproc_per_node=<N> -m distill.train <same args>          (DDP, one process per GPU)

What this version does (and why), relative to the first trainer:
  * Multi-GPU DDP + gradient accumulation. Global batch = world_size x micro_batch x grad_accum.
  * Deterministic per-epoch SHUFFLE derived from (seed, step): resuming continues the exact same data order.
  * fp32 master weights for everything trainable + bf16 autocast forward (the first trainer kept parameters AND
    AdamW states in bf16, where small updates round to zero). KL divergences are computed in fp32.
  * Loss masks match the data: the voice/persona PROMPT PREFIX (forced tokens at inference, prefix_len in the
    manifest) is excluded from hard-label CE; KL / bridge / hidden losses see every frame (the student must encode
    the prompt like the teacher). Only text + the AGENT's 8 codebooks are distilled: the user-stream predictions
    are never used at inference (LMGen overwrites them with the real microphone input), and the teacher loader
    copies depformer heads 0-7 -> 8-15 when missing, so teacher user-stream logits may not be trained predictions.
  * On-policy rollouts (P3/P4) now (a) start after the longest prompt prefix in the batch (the old start,
    chunk_frames//4, fell INSIDE the prompt on this data), (b) keep feeding the RECORDED user channel (the old code
    fed the prompt-phase SINE token as "user"), (c) are frame-aligned exactly like LMGen (the old rollout dropped
    frame 0 and shifted everything by one), (d) are verified against the data on every forced frame, and
    (e) get CE only on real-data frames (the old code applied CE to the student's own samples).
  * Warmup + cosine LR; weight decay only on matrices (not norms / embeddings / L_hidden projections).
  * Phase 4 depformer fine-tuning is kept: checkpoints and the final export include `depformer.*` once trained.
  * DDP-safe skip of non-finite steps (all ranks agree), all-reduced loss normalizers, SIGTERM-safe stop.
  * Validation loop (val split), best / latest / periodic checkpoints, structured logs under <output>/logs.
  * --smoke-test: every phase incl. on-policy rollouts, eval, checkpoint, export in a handful of steps.
"""

import argparse
import contextlib
import csv
import datetime
import json
import logging
import math
import os
import shutil
import signal
import socket
import subprocess
import sys
import time
import traceback
import typing as tp
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP

from moshi.models import loaders
from moshi.models.lm import LMModel, LMGen, AUDIO_TOKENS_PER_STREAM, _delay_sequence, _undelay_sequence
from moshi.utils.compile import no_cuda_graph

from . import checkpoint as ckpt
from .config import load_student_config
from .data.dataset import TeacherTokenDataset, collate_chunks
from .init_from_teacher import initialize_student
from .losses import RunningNorm, HiddenProjection, bridge_loss, frame_weights, _codebook_weights, TEXT_KL_TEMPERATURE
from .schedule import TrainingSchedule, PhaseSpec, PHASES, phase_at
from .student_model import StudentLMModel, build_student_lm, FROZEN_PREFIXES

logger = logging.getLogger("distill.train")
N_AGENT_CODEBOOKS = AUDIO_TOKENS_PER_STREAM          # 8
# Text-stream specials, from moshi/models/lm.py: `existing_text_padding_id=3` (LMGen.zero_text_code) and
# `end_of_text_padding_id=0`. Every other text id is a real SentencePiece piece, i.e. a spoken word.
TEXT_PAD_ID = 3
TEXT_EPAD_ID = 0


def _patch_kv_cache_dtype_for_autocast():
    """Streaming KV caches take their dtype from the attention WEIGHTS. With fp32 master weights + bf16 autocast
    (on-policy rollouts), keys/values are bf16 while the cache would be fp32 -> index_copy_ dtype error. Inside this
    training process only, create the cache in the active autocast dtype (bf16 -- exactly what inference uses).
    Without autocast nothing changes; moshi's source (server / offline / teacher) is untouched."""
    from moshi.modules import transformer as mt
    from . import gqa_attention as ga
    for cls in (mt.StreamingMultiheadAttention, ga.GQAStreamingMultiheadAttention):
        orig = cls._init_streaming_state
        if getattr(orig, "_ppx_autocast_kv", False):
            continue

        def patched(self, batch_size, _orig=orig):
            state = _orig(self, batch_size)
            cache = state.kv_cache.cache
            if torch.is_autocast_enabled(cache.device.type):
                want = torch.get_autocast_dtype(cache.device.type)
                if cache.dtype != want:
                    state.kv_cache.cache = cache.to(want)
            return state
        patched._ppx_autocast_kv = True
        cls._init_streaming_state = patched


_patch_kv_cache_dtype_for_autocast()


# ================================================================================================ forward capture
class ForwardCapture(tp.NamedTuple):
    logits: torch.Tensor
    logits_mask: torch.Tensor
    text_logits: torch.Tensor
    text_logits_mask: torch.Tensor
    raw_hidden: torch.Tensor
    layer_hidden: dict[int, torch.Tensor]


def run_forward_train_with_hooks(model: LMModel, codes: torch.Tensor, layer_indices: list[int]) -> ForwardCapture:
    """`LMModel.forward_train` (delay / undelay glue) with forward hooks that capture the per-layer temporal
    outputs (L_hidden) and the input of `out_norm` (L_bridge: the teacher's raw transformer output / the student's
    Bridge output). Codes [B, 17, T] -> logits aligned with codes (no extra shift needed for CE / KL)."""
    B, K, T = codes.shape
    initial = model._get_initial_token().expand(B, -1, -1)
    delayed_codes = _delay_sequence(model.delays, codes, initial)
    delayed_codes = torch.cat([initial, delayed_codes], dim=2)

    captured_layers: dict[int, torch.Tensor] = {}
    raw_hidden_box: dict[str, torch.Tensor] = {}
    handles = []

    def make_layer_hook(i):
        def hook(module, inputs, output):
            captured_layers[i] = output
        return hook

    for idx in layer_indices:
        handles.append(model.transformer.layers[idx].register_forward_hook(make_layer_hook(idx)))

    def out_norm_hook(module, inputs, output):
        raw_hidden_box["raw"] = inputs[0]

    handles.append(model.out_norm.register_forward_hook(out_norm_hook))
    try:
        transformer_out, text_logits = model.forward_embeddings(model.embed_codes(delayed_codes[:, :, :-1]))
        logits = model.forward_depformer_training(delayed_codes[:, :, 1:], transformer_out)
    finally:
        for h in handles:
            h.remove()

    logits, logits_mask = _undelay_sequence(
        model.delays[model.audio_offset:model.audio_offset + model.dep_q], logits, fill_value=float("nan"))
    logits_mask &= (codes[:, model.audio_offset:model.audio_offset + model.dep_q] != model.zero_token_id)
    text_logits, text_logits_mask = _undelay_sequence(model.delays[:1], text_logits, fill_value=float("nan"))
    text_logits_mask &= (codes[:, :1] != model.zero_token_id)
    return ForwardCapture(logits, logits_mask, text_logits, text_logits_mask, raw_hidden_box["raw"], captured_layers)


class StudentWithProjections(nn.Module):
    """What DDP wraps: the student + the L_hidden projection heads, so ONE forward call covers every trainable
    parameter (DDP needs the forward to go through the wrapped module to synchronize gradients)."""

    def __init__(self, student: StudentLMModel, projections: nn.ModuleList):
        super().__init__()
        self.student = student
        self.projections = projections

    capture_layers: bool = True          # set False when phase.delta == 0: skips the L_hidden capture+projection

    def forward(self, codes: torch.Tensor, text_only: bool = False) -> dict:
        # `text_only` goes through the SAME wrapped module (rather than calling `self.student` directly) so
        # DDP still sees the forward and synchronizes gradients on the text steps. The audio embeddings and
        # the depformer get no gradient on those steps, which is why DDP is built with
        # find_unused_parameters=True (see `wrap_ddp`).
        if text_only:
            return {"text_logits": text_only_text_logits(self.student, codes)}
        want = list(range(len(self.projections))) if self.capture_layers else []
        cap = run_forward_train_with_hooks(self.student, codes, want)
        projected = [self.projections[i](cap.layer_hidden[i].float()) for i in want]
        return {"logits": cap.logits, "logits_mask": cap.logits_mask, "text_logits": cap.text_logits,
                "text_logits_mask": cap.text_logits_mask, "raw_hidden": cap.raw_hidden, "projected": projected}


# ================================================================================================ text-only mode
def text_only_text_logits(model: LMModel, text_in: torch.Tensor) -> torch.Tensor:
    """Pure text-LM forward through `model`'s temporal path. `text_in` [B, T] text token ids -> text logits
    [B, T, text_card], where position t predicts the token that FOLLOWS `text_in[:, t]`.

    The 16 audio rows are filled with `zero_token_id` (-1), for which `ScaledEmbedding.forward` returns
    exactly zero (`y = torch.where(is_zero, 0, y)` in moshi/models/lm.py) -- so `embed_codes` reduces to the
    text embedding alone and no audio information enters or is expected. The depformer is never called: this
    goes through `forward_codes` -> `forward_embeddings` only, which for the student is
    transformer -> bridge -> frozen out_norm -> frozen text_linear. Both the student's transformer and its
    bridge are therefore trained by this objective, in the same space inference uses.
    """
    B, T = text_in.shape
    seq = torch.full((B, model.num_codebooks, T), model.zero_token_id, dtype=torch.long, device=text_in.device)
    seq[:, 0] = text_in
    _, text_logits = model.forward_codes(seq)        # text_logits [B, 1, T, text_card]
    return text_logits[:, 0]


def text_only_terms(s_text_logits: torch.Tensor, t_text_logits: torch.Tensor, targets: torch.Tensor,
                    text_kl_temperature: float) -> dict[str, torch.Tensor]:
    """KL(teacher || student) + hard-label CE on a text-only window. Every position carries a real token, so
    unlike the audio path there is no padding to weight away -- these are plain means."""
    mask = torch.ones(targets.shape, dtype=torch.bool, device=targets.device)
    kl = kl_fp32(s_text_logits.unsqueeze(1), t_text_logits.unsqueeze(1), mask.unsqueeze(1), text_kl_temperature)
    ce = masked_ce(s_text_logits.unsqueeze(1), targets.unsqueeze(1), mask.unsqueeze(1))
    return {"kl_text": kl, "ce": ce}


# ================================================================================================ losses
def kl_fp32(student_logits, teacher_logits, mask, temperature, weight=1.0):
    """KL(teacher || student) at `temperature` (x tau^2), computed in fp32, as a WEIGHTED MEAN over valid
    positions. NaNs in the undelay tail are masked out (and scrubbed so nan*0 can't leak).

    The normalizer is `(weight * mask).sum()`, not `mask.sum()`. With the previous `mask.sum()` denominator a
    per-position `weight` inflated the term's MAGNITUDE instead of redistributing its mean -- harmless while
    `RunningNorm` absorbed the scale, but it made per-position weighting useless as a way to shift emphasis
    between positions (the whole point of the text content weighting in `loss_terms`): doubling the weight on
    10% of positions doubled the loss rather than moving gradient mass onto those positions."""
    s = torch.nan_to_num(student_logits.float(), nan=0.0) / temperature
    with torch.no_grad():
        t = torch.nan_to_num(teacher_logits.float(), nan=0.0) / temperature
        log_t = F.log_softmax(t, dim=-1)
    log_s = F.log_softmax(s, dim=-1)
    kl = F.kl_div(log_s, log_t, reduction="none", log_target=True).sum(dim=-1) * (temperature ** 2)
    m = mask.to(kl.dtype)
    if weight is not None and (not isinstance(weight, (int, float)) or weight != 1.0):
        m = m * weight
    return (kl * m).sum() / m.sum().clamp_min(1.0)


def masked_ce(logits, targets, mask, weight=None):
    """Hard-label CE as a weighted mean over mask==True positions; exactly 0 (not NaN) when no position is
    valid. `weight` (broadcastable to `mask`) redistributes emphasis across positions -- the normalizer is
    `(weight * mask).sum()`, matching `kl_fp32`."""
    lg = torch.nan_to_num(logits.float(), nan=0.0)
    tg = targets.masked_fill(~mask, -100)
    per_pos = F.cross_entropy(lg.reshape(-1, lg.shape[-1]), tg.reshape(-1), ignore_index=-100, reduction="none")
    if weight is None:
        return per_pos.sum() / mask.sum().clamp_min(1)
    w = (mask.to(per_pos.dtype) * weight).reshape(-1)
    return (per_pos * w).sum() / w.sum().clamp_min(1e-6)


def hidden_term(projected: torch.Tensor, teacher_hidden: torch.Tensor) -> torch.Tensor:
    """== distill.losses.hidden_loss on already-projected student states (L2-normalize, MSE + (1 - cosine)), in a
    memory-lean form. For unit vectors p, t in R^D: MSE = mean_pos(|p - t|^2) / D = 2 (1 - mean_pos(p.t)) / D and
    cosine = p.t, so the loss is (1 + 2/D) * (1 - mean_pos(p.t)). Same value, far fewer fp32 [B, T, 4096]
    intermediates kept for backward (x 20 layers this was a large share of activation memory)."""
    p = F.normalize(projected.float(), dim=-1)
    with torch.no_grad():
        t = F.normalize(teacher_hidden.float(), dim=-1)
    dot = (p * t).sum(dim=-1).mean()
    return (1.0 + 2.0 / p.shape[-1]) * (1.0 - dot)


def loss_terms(s_out: dict, t_cap: ForwardCapture, codes: torch.Tensor, transition_mask: torch.Tensor,
               prefix_lens: torch.Tensor, ce_end: tp.Optional[torch.Tensor], selected_layers: list[int],
               args, initial_audio_id: int, text_initial_id: int) -> dict[str, torch.Tensor]:
    B, K, T = codes.shape
    dev = codes.device
    frames = torch.arange(T, device=dev)[None, :]
    conv = frames >= prefix_lens[:, None].to(dev)                    # [B, T] conversation (non-prompt) frames
    ce_frames = conv if not args.ce_on_prefix else torch.ones_like(conv)
    if ce_end is not None:                                           # on-policy: CE only on real-data frames
        ce_frames = ce_frames & (frames < ce_end[:, None].to(dev))
    kl_frames = torch.ones_like(conv) if args.kl_on_prefix else conv

    na = 2 * N_AGENT_CODEBOOKS if args.distill_user_stream else N_AGENT_CODEBOOKS
    s_audio, t_audio = s_out["logits"][:, :na], t_cap.logits[:, :na]
    a_mask = s_out["logits_mask"][:, :na]
    t_mask = s_out["text_logits_mask"]                               # [B, 1, T]

    cb_w = _codebook_weights(na, dev, torch.float32).view(1, -1, 1)
    cb1 = torch.nan_to_num(t_cap.logits[:, 0].float(), nan=0.0)      # teacher's first agent codebook
    fw = frame_weights(transition_mask.to(dev), cb1).unsqueeze(1)    # [B, 1, T]
    kl_audio = kl_fp32(s_audio, t_audio, a_mask & kl_frames[:, None], args.audio_kl_temperature, weight=cb_w * fw)

    # ---- text: weight the frames that carry an actual WORD ------------------------------------------------
    # The text stream is ~87-90% TEXT_PAD(3)/TEXT_EPAD(0) plus the forced persona tokens inside the prompt
    # prefix. Unweighted, the text KL/CE is therefore a padding-and-timing objective: ~90% of its gradient goes
    # to positions whose answer is fully determined by the prompt or by the turn-taking pattern, which is
    # exactly what the student learned (fluent, correctly-timed, content-free speech). Boosting content frames
    # by `--text-content-boost` moves gradient mass onto the words that carry the answer, at zero extra compute.
    text_tgt = codes[:, :1]
    content = (text_tgt != TEXT_PAD_ID) & (text_tgt != TEXT_EPAD_ID) & conv[:, None]
    text_w = 1.0 + args.text_content_boost * content.to(torch.float32)

    kl_text = kl_fp32(s_out["text_logits"], t_cap.text_logits, t_mask & kl_frames[:, None],
                      TEXT_KL_TEMPERATURE, weight=text_w)

    text_ok = t_mask & ce_frames[:, None] & (text_tgt != text_initial_id)
    audio_tgt = codes[:, 1:1 + na]
    audio_ok = a_mask & ce_frames[:, None] & (audio_tgt != initial_audio_id)
    ce = masked_ce(s_out["text_logits"], text_tgt, text_ok, weight=text_w) \
        + masked_ce(s_audio, audio_tgt, audio_ok)

    br = bridge_loss(s_out["raw_hidden"], t_cap.raw_hidden)
    if len(s_out.get("projected", [])) > 0 and len(selected_layers) > 0 and len(t_cap.layer_hidden) > 0:
        hid = torch.stack([hidden_term(s_out["projected"][i], t_cap.layer_hidden[selected_layers[i]])
                           for i in range(len(selected_layers))]).mean()
    else:
        hid = torch.zeros((), device=dev)
    # `kl_audio` and `kl_text` are returned SEPARATELY (there is no combined "kl" any more): they used to be
    # summed into one term normalized by one RunningNorm, so there was no way to raise text pressure without
    # also raising audio pressure. The training loop weights them with `phase.beta` and `--beta-text`.
    return {"ce": ce, "kl_audio": kl_audio, "kl_text": kl_text, "bridge": br, "hidden": hid,
            "ce_frames": ce_frames.float().sum(), "kl_frames": kl_frames.float().sum(),
            "content_frames": content.float().sum()}


class SyncedNorm:
    """RunningNorm whose statistic is updated with the all-reduced value, so every rank scales identically."""

    def __init__(self):
        self.norm = RunningNorm()

    def scale(self, local_value: float) -> float:
        m = self.norm._mean
        return (m if m is not None else max(abs(local_value), 1e-6)) + self.norm.eps

    def update(self, global_value: float):
        self.norm(torch.tensor(global_value))


# ================================================================================================ distributed
class DistInfo(tp.NamedTuple):
    rank: int
    world: int
    local_rank: int
    device: torch.device

    @property
    def main(self):
        return self.rank == 0


def setup_distributed(device_arg: str) -> DistInfo:
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        rank, world = int(os.environ["RANK"]), int(os.environ["WORLD_SIZE"])
        local = int(os.environ.get("LOCAL_RANK", "0"))
        if device_arg.startswith("cuda"):
            torch.cuda.set_device(local)
            device = torch.device("cuda", local)
            backend = "nccl"
        else:
            device, backend = torch.device("cpu"), "gloo"
        # generous timeout: rank 0 alone runs the student initialization while the others wait
        dist.init_process_group(backend=backend, timeout=datetime.timedelta(minutes=90))
        return DistInfo(rank, world, local, device)
    device = torch.device(device_arg)
    if device.type == "cuda":
        torch.cuda.set_device(device.index or 0)
    return DistInfo(0, 1, 0, device)


def barrier(di: DistInfo):
    if di.world > 1:
        dist.barrier()


def all_reduce(t: torch.Tensor, di: DistInfo, op=None) -> torch.Tensor:
    if di.world > 1:
        dist.all_reduce(t, op=op or dist.ReduceOp.SUM)
    return t


# ================================================================================================ data
class GlobalBatchSampler:
    """Global step s consumes positions [s*G, (s+1)*G) of an infinite stream made of per-epoch permutations
    (numpy RNG seeded with (seed, epoch)). Fully determined by the step -> exact resume, no sampler state."""

    def __init__(self, n: int, global_batch: int, seed: int):
        self.n, self.g, self.seed = n, global_batch, seed
        self._cache: dict[int, np.ndarray] = {}

    def perm(self, epoch: int) -> np.ndarray:
        if epoch not in self._cache:
            if len(self._cache) > 3:
                self._cache.pop(min(self._cache))
            self._cache[epoch] = np.random.default_rng([self.seed, epoch]).permutation(self.n)
        return self._cache[epoch]

    def global_indices(self, step: int) -> list[int]:
        out = []
        for p in range(step * self.g, (step + 1) * self.g):
            e, o = divmod(p, self.n)
            out.append(int(self.perm(e)[o]))
        return out

    def epoch_of(self, step: int) -> float:
        return step * self.g / self.n


def load_batch(ds: TeacherTokenDataset, idxs: list[int], chunk: int, device) -> dict:
    b = collate_chunks([ds[i] for i in idxs], chunk)
    if len(b["sample_ids"]) != len(idxs):
        raise RuntimeError(f"{len(idxs) - len(b['sample_ids'])} sample(s) shorter than --chunk-frames {chunk} "
                           "(collate_chunks drops them) -- the dataset analysis at startup should have caught this")
    return {"codes": b["codes"].to(device, non_blocking=True), "transition_mask": b["transition_mask"].to(device),
            "prefix_lens": b["prefix_lens"].to(device), "sample_ids": b["sample_ids"]}


@torch.no_grad()
def init_fidelity_report(student, teacher, batches: list[torch.Tensor], max_batches: int = 4) -> dict:
    """How much of the teacher's TEXT behaviour the initialization actually preserved, measured on content
    frames only (real words -- not PAD/EPAD).

    This is the number that decides whether a run is worth starting, and nothing in the pipeline reported it
    before: `tests/test_init_sanity.py` only verifies the weight-transform algebra in the IDENTITY case (same
    width, all layers, all heads), and `sanity_check_student` only catches total collapse (all-silence or
    all-identical tokens). A student can pass both while retaining almost none of the teacher's text
    behaviour -- which is what produced fluent, correctly-timed, content-free speech.

    Read `top1_agree`:
      > 0.5  the init preserved the text pathway; the remaining gap is a training-budget / loss-weighting
             problem, so the content weighting and the text-only steps should close it.
      < 0.2  the init destroyed the text pathway. More steps will not fix that -- widen the student where
             knowledge lives (distill/configs/student_ppx_m.yaml keeps 36.4% of the teacher's FFN memory
             slots instead of 20.8%) and re-check this number.
    """
    student.eval(); teacher.eval()
    kls, t1s, t5s, n = [], [], [], 0
    for b in batches[:max_batches]:
        t_cap = run_forward_train_with_hooks(teacher, b, [])
        s_cap = run_forward_train_with_hooks(student, b, [])
        tl = torch.nan_to_num(t_cap.text_logits.float(), nan=0.0)[:, 0]        # [B, T, text_card]
        sl = torch.nan_to_num(s_cap.text_logits.float(), nan=0.0)[:, 0]
        txt = b[:, 0]
        content = (txt != TEXT_PAD_ID) & (txt != TEXT_EPAD_ID)
        if int(content.sum()) == 0:
            continue
        tlc, slc = tl[content], sl[content]
        kls.append(float(F.kl_div(F.log_softmax(slc, -1), F.softmax(tlc, -1), reduction="batchmean")))
        t_arg = tlc.argmax(-1)
        t1s.append(float((slc.argmax(-1) == t_arg).float().mean()))
        k = min(5, slc.shape[-1])
        t5s.append(float((slc.topk(k, -1).indices == t_arg[:, None]).any(-1).float().mean()))
        n += int(content.sum())
    if not kls:
        return {"content_frames": 0, "note": "no content frames in the calibration batches"}
    rep = {"content_frames": n, "kl_text_content": round(sum(kls) / len(kls), 4),
           "top1_agree": round(sum(t1s) / len(t1s), 4), "top5_agree": round(sum(t5s) / len(t5s), 4)}
    # This number is a ZERO-SHOT reading of an init that was never meant to be zero-shot functional:
    # P1_align exists precisely to repair the layer-boundary mismatch the reduction creates. An earlier
    # version of this function declared "init DESTROYED the text pathway -- widen the student" below 0.2
    # top-1, which on the real 7B teacher fired at 0.0092 while the run went on to halve kl_text and cut
    # bridge_raw from 1.32 to 0.66 in 120 steps. Absolute top-1 thresholds do not transfer across scales,
    # so the verdict no longer prescribes an architecture change. Decide from TRAINABILITY instead:
    # `bridge_lstsq_residual` (logged next to this), and whether bridge_raw / kl_text_raw fall over P1.
    rep["verdict"] = (
        "informational only -- do NOT change architecture on this number alone. The init is expected to be "
        "far from the teacher zero-shot; P1_align repairs it. Decide from the first validations instead: "
        "val_kl_text_content and val_agree_text_content must improve, and train_log.csv's bridge_raw must "
        "fall (bridge_raw < 1.0 implies cosine > 0, i.e. the pathway is intact and trainable). Only if "
        "bridge_raw stays flat near its starting value AND val_kl_text_content stalls across several "
        "evaluations should you widen the student (distill/configs/student_ppx_m.yaml)."
    )
    if rep["top1_agree"] > 0.5:
        rep["verdict"] = ("init already reproduces the teacher's text argmax zero-shot -- unusually good; "
                          "the remaining gap is budget/weighting, not architecture.")
    return rep


def build_calibration_batches(ds: TeacherTokenDataset, args, device) -> list[torch.Tensor]:
    """Calibration batches for `init_from_teacher.initialize_student`, taken from frames where the teacher is
    ANSWERING -- not from the prompt prefix and greeting.

    This matters more than it looks. The activations collected here decide, irreversibly:
      * the 4096 -> hidden_size residual SVD basis   (`compute_activation_projection`)
      * which FFN hidden channels survive 11264 -> ~3754 (`select_ffn_channels`, ranked by activation RMS)
      * which teacher layers are kept               (`compute_layer_scores`)
      * the Bridge least-squares fit                (`init_bridge_least_squares`)

    The previous version called `collate_chunks(..., calib_frames)`, which takes `codes[:, :calib_frames]` --
    the FIRST frames of each sample. With `prefix_len = 136 + persona_tokens` (136-256 here) and
    `calib_frames=500`, 27-51% of every calibration sequence was the voice-prompt + persona-prompt prefix, and
    the rest was the greeting and at most the first question. So the init kept the subspace and the FFN
    channels that carry the most activation energy while encoding a voice clip, reading a persona prompt and
    saying hello -- and discarded the sparse, low-RMS channels that only fire when the teacher recalls a fact.
    That is a direct mechanism for "keeps voice/persona/turn-taking, loses knowledge".

    Here each sample is cropped to start `--calib-skip-frames` frames PAST its own prefix, so every calibration
    frame is mid-answer. Samples too short for that crop are skipped; `--calib-prefer-scenario` biases selection
    toward the knowledge-heavy scenarios (multi-turn Q&A) when the manifest records a scenario.
    """
    want_frames = args.calib_frames
    prefer = {s for s in (args.calib_prefer_scenario or "").split(",") if s}
    order = list(np.random.default_rng([args.seed, 12345]).permutation(len(ds)))
    if prefer:
        scen = [str(e.get("scenario", "")) for e in ds.entries]
        order.sort(key=lambda i: 0 if scen[i] in prefer else 1)        # stable: keeps the shuffle inside groups

    batches, group, used, skipped = [], [], 0, 0
    for i in order:
        if len(batches) >= args.calib_batches:
            break
        s = ds[int(i)]
        start = int(s.prefix_len) + args.calib_skip_frames
        if s.codes.shape[-1] - start < want_frames:
            skipped += 1
            continue                                                    # too short to crop past the prompt
        s.codes = s.codes[:, start:start + want_frames].contiguous()
        s.transition_frames = [f - start for f in s.transition_frames if 0 <= f - start < want_frames]
        s.prefix_len = 0                                                # the crop contains no prompt frames
        group.append(s)
        used += 1
        if len(group) == 4:
            batches.append(collate_chunks(group, want_frames)["codes"].to(device))
            group = []
    if not batches:
        raise SystemExit(
            f"Could not build any calibration batch: no sample has {want_frames} frames left after skipping "
            f"its prefix + --calib-skip-frames {args.calib_skip_frames}. Lower --calib-frames or "
            f"--calib-skip-frames (prefix_len here is {max(int(e.get('prefix_len', 0)) for e in ds.entries)} "
            f"at most, samples are {min(e['num_frames'] for e in ds.entries)} frames at least).")
    logger.info("calibration: %d batches x 4 samples x %d answer-phase frames (skipped %d too-short samples, "
                "prefer_scenario=%s)", len(batches), want_frames, skipped, sorted(prefer) or "-")
    return batches


def analyze_dataset(ds: TeacherTokenDataset, chunk: int, name: str) -> dict:
    ents = ds.entries
    frames = np.array([e["num_frames"] for e in ents])
    pls = np.array([int(e.get("prefix_len", 0)) for e in ents])
    too_short = int((frames < chunk).sum())
    rep = {"name": name, "samples": len(ents), "frames_min": int(frames.min()), "frames_max": int(frames.max()),
           "chunk_frames": chunk, "samples_shorter_than_chunk": too_short,
           "hours_total_in_chunk": round(float(np.minimum(frames, chunk).sum()) / 12.5 / 3600, 2),
           "conversation_hours_in_chunk": round(float(np.clip(np.minimum(frames, chunk) - pls, 0, None).sum()) / 12.5 / 3600, 2),
           "prefix_len_min": int(pls.min()), "prefix_len_max": int(pls.max()), "prefix_len_mean": round(float(pls.mean()), 1),
           "has_prefix_len": bool(pls.max() > 0),
           "scenarios": {k: int(v) for k, v in zip(*np.unique([e.get("scenario", "?") for e in ents], return_counts=True))}}
    return rep


# ================================================================================================ on-policy rollout
@torch.no_grad()
def student_rollout(student: StudentLMModel, codes: torch.Tensor, start: int, n_total: int, autocast_ctx,
                    sampling: dict) -> torch.Tensor:
    """Returns [B, 17, n_total]: frames < start are the data (teacher-forced on every channel); from `start` on the
    student samples text + agent audio while the user channel keeps receiving the RECORDED user codes.

    LMGen overwrites call 0 with the initial token and returns frame f at call f+1 (max_delay=1), exactly as in
    data generation where recorded codes[:, j] == call j+1. So call 0 is a dummy and call j+1 carries codes[:, j];
    n_total + 2 calls return exactly frames 0 .. n_total-1, aligned with `codes`."""
    B, K, T = codes.shape
    u = student.audio_offset + AUDIO_TOKENS_PER_STREAM             # first user-stream row (9)
    was_training = student.training
    student.eval()
    lm_gen = LMGen(student, device=codes.device, use_sampling=True, **sampling)
    out = []
    try:
        with no_cuda_graph(), autocast_ctx, lm_gen.streaming(B):
            for c in range(n_total + 2):
                j = min(max(c - 1, 0), T - 1)                     # frame provided at this call (call 0: dummy)
                user = codes[:, u:, j:j + 1]
                if c - 1 < start:
                    tok = lm_gen.step(input_tokens=user, moshi_tokens=codes[:, 1:u, j:j + 1], text_token=codes[:, 0, j])
                else:
                    tok = lm_gen.step(input_tokens=user)
                if tok is not None:
                    out.append(tok[:, :, 0])
    finally:
        student.train(was_training)
    return torch.stack(out, dim=-1)[:, :, :n_total].contiguous()


# ================================================================================================ trainer
TRAIN_CSV_COLUMNS = ["step", "phase", "total", "ce_raw", "kl_audio_raw", "kl_text_raw", "bridge_raw", "hidden_raw",
                     "seconds_per_step", "lr", "grad_norm", "onpolicy", "textstep", "frames_per_s", "peak_mem_gb",
                     "skipped_total", "epoch", "text_tokens_total"]
VAL_CSV_COLUMNS = ["step", "phase", "val_kl_text", "val_kl_audio", "val_ce_text", "val_ce_audio", "val_bridge",
                   "val_hidden", "val_agree_text", "val_agree_cb0",
                   # content-vs-padding split of the text stream: `val_agree_text_content` is the number to
                   # watch for "can it answer the question". `val_agree_text` alone cannot distinguish a model
                   # that learned the answer from one that only learned when to stay silent.
                   "val_agree_text_content", "val_agree_text_pad", "val_ce_text_content", "val_kl_text_content",
                   "val_content_frac", "val_samples", "best"]


class RunLogger:
    """<output>/logs: train_rank<r>.log (every rank), metrics.jsonl, events.jsonl, val_log.csv, heartbeat.json;
    <output>/train_log.csv (same columns as before + extras). Only rank 0 writes the shared files."""

    def __init__(self, out_dir: Path, di: DistInfo):
        self.di = di
        self.dir = out_dir / "logs"
        self.dir.mkdir(parents=True, exist_ok=True)
        fmt = logging.Formatter(f"%(asctime)s %(levelname)s [rank{di.rank}] %(message)s")
        root = logging.getLogger()
        root.setLevel(logging.INFO)
        for h in list(root.handlers):
            root.removeHandler(h)
        fh = logging.FileHandler(self.dir / f"train_rank{di.rank}.log", encoding="utf-8")
        fh.setFormatter(fmt)
        root.addHandler(fh)
        if di.main:
            sh = logging.StreamHandler(sys.stdout)
            sh.setFormatter(fmt)
            root.addHandler(sh)
        logging.captureWarnings(True)
        self.train_csv = out_dir / "train_log.csv"
        self.val_csv = self.dir / "val_log.csv"

    def _csv(self, path: Path, cols: list[str], row: dict):
        new = not path.exists()
        if not new:
            with open(path, encoding="utf-8") as f:
                header = f.readline().strip().split(",")
            if header != cols:                     # a log from an older trainer version: keep it, start fresh
                path.replace(path.with_suffix(f".old{int(time.time())}.csv"))
                new = True
        with open(path, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
            if new:
                w.writeheader()
            w.writerow(row)

    def train_row(self, row: dict):
        if self.di.main:
            self._csv(self.train_csv, TRAIN_CSV_COLUMNS, row)
            self.jsonl("metrics.jsonl", row)

    def val_row(self, row: dict):
        if self.di.main:
            self._csv(self.val_csv, VAL_CSV_COLUMNS, row)
            self.jsonl("metrics.jsonl", {"type": "val", **row})

    def jsonl(self, name: str, obj: dict):
        if self.di.main:
            with open(self.dir / name, "a", encoding="utf-8") as f:
                f.write(json.dumps({"time": time.strftime("%Y-%m-%dT%H:%M:%S"), **obj}) + "\n")

    def event(self, kind: str, **info):
        logger.info("EVENT %s %s", kind, json.dumps(info)[:2000])
        self.jsonl("events.jsonl", {"event": kind, **info})

    def write_json(self, name: str, obj: dict):
        if self.di.main:
            tmp = self.dir / f".{name}.tmp"
            tmp.write_text(json.dumps(obj, indent=1), encoding="utf-8")
            tmp.replace(self.dir / name)


def load_teacher(args, device) -> LMModel:
    if args.teacher_kwargs_json:                 # tests only: a tiny synthetic teacher
        from safetensors.torch import load_file
        kw = json.loads(args.teacher_kwargs_json)
        teacher = LMModel(device=device, dtype=torch.bfloat16, **kw)
        teacher.load_state_dict(load_file(args.teacher_checkpoint, device="cpu"), strict=True)
        # LMModel.__init__ builds text_linear/out_norm/depformer_in/linears without a dtype; get_moshi_lm fixes
        # that with a trailing .to() -- do the same here.
        teacher = teacher.to(device=device, dtype=torch.bfloat16)
    else:
        # Load on CPU, then move to THIS rank's GPU. get_moshi_lm(device=cuda:N) calls safetensors'
        # load_file(device="cuda") without the index, which safetensors maps to cuda:0 -- under DDP every rank would
        # first put the whole 16.7 GB teacher on GPU 0 (and the caching allocator keeps it there): OOM on rank 0.
        teacher = loaders.get_moshi_lm(args.teacher_checkpoint, device="cpu", dtype=torch.bfloat16)
        teacher = teacher.to(device)
        if device.type == "cuda":
            torch.cuda.empty_cache()
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False
    return teacher


TRAINABLE_TOP = ("transformer", "emb", "text_emb", "bridge")


def param_groups(student: StudentLMModel, projections: nn.ModuleList, weight_decay: float, depth_mult: float):
    """Decay only matrices; never norms, biases, embeddings or the L_hidden projections. The depformer group
    exists from the start (params frozen until Phase 4, so AdamW simply skips them) -> stable group layout."""
    dec, nodec, dep_dec, dep_nodec = [], [], [], []
    for n, p in student.named_parameters():
        top = n.split(".")[0]
        if n.startswith("depformer."):
            (dep_dec if p.ndim >= 2 else dep_nodec).append(p)
        elif n.startswith(FROZEN_PREFIXES):
            continue
        elif top in ("emb", "text_emb") or p.ndim < 2:
            nodec.append(p)
        else:
            dec.append(p)
    groups = [{"name": "decay", "params": dec, "weight_decay": weight_decay, "lr_mult": 1.0},
              {"name": "no_decay", "params": nodec, "weight_decay": 0.0, "lr_mult": 1.0},
              {"name": "hidden_proj", "params": list(projections.parameters()), "weight_decay": 0.0, "lr_mult": 1.0},
              {"name": "depformer_decay", "params": dep_dec, "weight_decay": weight_decay, "lr_mult": depth_mult},
              {"name": "depformer_no_decay", "params": dep_nodec, "weight_decay": 0.0, "lr_mult": depth_mult}]
    return [g for g in groups if g["params"]]


def lr_factor(step: int, total: int, warmup: int, min_ratio: float) -> float:
    if warmup > 0 and step < warmup:
        return (step + 1) / warmup
    prog = min(1.0, max(0.0, (step - warmup) / max(1, total - warmup)))
    return min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * prog))


def phase_for(step: int, total: int, fixed: tp.Optional[str]) -> PhaseSpec:
    if fixed:
        return next(p for p in PHASES if p.name == fixed)
    return phase_at(min(1.0, step / max(1, total)))


def git_rev(path: Path) -> str:
    try:
        return subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"], capture_output=True, text=True,
                              timeout=10).stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def train(args):
    di = setup_distributed(args.device)
    out = Path(args.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    rl = RunLogger(out, di)
    torch.manual_seed(args.seed + di.rank)
    np.random.seed(args.seed + di.rank)
    if di.device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    # ---------------------------------------------------------------- smoke overrides
    if args.smoke_test:
        args.total_steps = args.smoke_steps
        args.eval_every = max(2, args.smoke_steps // 3)
        args.save_every = max(2, args.smoke_steps // 3)
        args.keep_every = 0
        args.val_samples = min(args.val_samples, 4 * di.world)
        args.warmup_steps = min(args.warmup_steps, 2)
        args.log_every = 1
        args.calib_batches = min(args.calib_batches, 2)
        args.calib_frames = min(args.calib_frames, 300)
        # the smoke dataset's samples are the same 1000 frames, but keep the crop reachable on short data
        args.calib_skip_frames = min(args.calib_skip_frames, 50)
        args.onpolicy_fraction = 1.0          # every P3/P4 smoke step exercises a rollout
        if di.device.type == "cpu":           # CPU tests only; on GPU keep the REAL rollout length so the smoke
            args.rollout_frames = min(args.rollout_frames, 24)          # run measures true on-policy step time
            args.onpolicy_context_frames = min(args.onpolicy_context_frames, 8)

    # ---------------------------------------------------------------- batch geometry
    micro = args.micro_batch
    if args.batch_size % (di.world * micro) != 0:
        raise SystemExit(f"--batch-size {args.batch_size} must be a multiple of world_size x --micro-batch "
                         f"= {di.world} x {micro}")
    accum = args.batch_size // (di.world * micro)

    # ---------------------------------------------------------------- data
    train_ds = TeacherTokenDataset(args.data_dir)
    val_ds = TeacherTokenDataset(args.val_dir) if args.val_dir and Path(args.val_dir, "manifest.jsonl").exists() else None
    rep = {"train": analyze_dataset(train_ds, args.chunk_frames, args.data_dir)}
    if val_ds is not None:
        rep["val"] = analyze_dataset(val_ds, args.chunk_frames, args.val_dir)
    if rep["train"]["samples_shorter_than_chunk"]:
        raise SystemExit(f"{rep['train']['samples_shorter_than_chunk']} training samples are shorter than "
                         f"--chunk-frames {args.chunk_frames}; use the dataset's frame count (main: 1000, long: 1500)")
    if val_ds is not None and rep["val"]["samples_shorter_than_chunk"]:
        raise SystemExit("validation samples shorter than --chunk-frames")
    if not rep["train"]["has_prefix_len"]:
        logger.warning("manifest has no prefix_len (older dataset format): the prompt prefix can't be masked and "
                       "on-policy rollouts start after --onpolicy-context-frames only")
    student_cfg = load_student_config(args.student_config)
    if args.chunk_frames + 2 > student_cfg.max_frames:
        raise SystemExit(f"--chunk-frames {args.chunk_frames} exceeds the student's context ({student_cfg.max_frames})")
    sampler = GlobalBatchSampler(len(train_ds), args.batch_size, args.seed)

    # ---------------------------------------------------------------- text-only corpus (optional)
    # A share `--text-fraction` of the teacher-forced steps run as pure text steps instead. This is where the
    # knowledge the init truncated actually comes back: the audio corpus holds well under 1M agent text tokens
    # in total, while a text corpus can push orders of magnitude more through the same temporal transformer at
    # one position per token (no Mimi, no depformer). Mixed in -- never run alone -- so the student keeps the
    # 17-stream input distribution it sees at inference.
    text_stream = None
    if args.text_data:
        if not args.tokenizer:
            raise SystemExit("--text-data requires --tokenizer (the teacher's tokenizer_spm_32k_3.model)")
        if args.text_seq_len > student_cfg.max_frames:
            raise SystemExit(f"--text-seq-len {args.text_seq_len} exceeds the student's attention context "
                             f"({student_cfg.max_frames})")
        from .data.text_dataset import build_token_cache, TextTokenStream
        cache = args.text_cache or str(out / "text_tokens.npy")
        if di.main:
            build_token_cache(args.text_data, args.tokenizer, cache, args.text_max_tokens)
        barrier(di)                                      # other ranks wait for rank 0 to write the cache
        tokens = np.load(cache, mmap_mode="r")
        text_stream = TextTokenStream(tokens, args.text_seq_len, args.seed)
        rep["text"] = {"cache": cache, "tokens": int(tokens.shape[0]), "seq_len": args.text_seq_len,
                       "windows": len(text_stream), "fraction_of_steps": args.text_fraction,
                       "tokens_per_step": di.world * micro * accum * args.text_seq_len,
                       "text_token_updates_planned": int(args.total_steps * args.text_fraction * di.world
                                                         * micro * accum * args.text_seq_len)}
        if di.main:
            logger.info("text-only distillation: %s", json.dumps(rep["text"]))

    rep["geometry"] = {"world_size": di.world, "micro_batch": micro, "grad_accum": accum,
                       "global_batch": args.batch_size, "steps_per_epoch": round(len(train_ds) / args.batch_size, 2),
                       "epochs_total": round(args.total_steps * args.batch_size / len(train_ds), 2)}
    rl.write_json("data_report.json", rep)
    logger.info("data: %s", json.dumps(rep))

    # ---------------------------------------------------------------- models
    teacher = load_teacher(args, di.device)
    tkw = json.loads(args.teacher_kwargs_json) if args.teacher_kwargs_json else None
    student = build_student_lm(student_cfg, args.teacher_checkpoint, device=di.device, dtype=torch.bfloat16,
                               teacher_kwargs=tkw)
    ckpt_path = out / "student_checkpoint.pt"
    payload = None
    start_step = 0
    warm_has_depformer = False
    if args.resume and ckpt_path.exists():
        payload = ckpt.load_training_payload(str(ckpt_path))
        if payload.get("format") != "ppx_student_train_v2":
            raise SystemExit(f"{ckpt_path} was written by the old trainer; use --init-from {ckpt_path} to warm-start "
                             "from its weights in a NEW --output-dir")
        start_step = int(payload["step"])
        selected_layers = payload["selected_teacher_layers"]
        rl.event("resume", step=start_step, checkpoint=str(ckpt_path))
    elif args.init_from:
        warm = ckpt.load_training_payload(args.init_from)
        warm_has_depformer = bool(warm.get("depformer_trained"))
        selected_layers = warm["selected_teacher_layers"]
        ckpt.load_trainable_state_dict(student, warm["student_state_dict"])
        rl.event("init_from", path=args.init_from, source_step=warm.get("step"),
                 depformer_included=bool(warm.get("depformer_trained")))
    else:
        # --------- data-driven init from the teacher on rank 0 only; DDP then broadcasts rank 0's weights
        init_path = out / "init_student.pt"
        if di.main:
            if init_path.exists():
                ini = ckpt.load_training_payload(str(init_path))
                ckpt.load_trainable_state_dict(student, ini["student_state_dict"])
                selected_layers = ini["selected_teacher_layers"]
                rl.event("init_reused", path=str(init_path))
            else:
                t0 = time.time()
                calib = build_calibration_batches(train_ds, args, di.device)
                rl.event("calibration_built", batches=len(calib),
                         frames_per_batch=int(calib[0].shape[-1]) if calib else 0,
                         skip_frames=args.calib_skip_frames)
                diag = initialize_student(student, teacher, calib, student_cfg.init.keep_first,
                                          student_cfg.init.keep_last,
                                          spread_layers=(args.layer_selection == "spread"))
                selected_layers = [int(x) for x in diag["selected_layers"]]
                torch.save({"student_state_dict": ckpt.trainable_state_dict(student),
                            "selected_teacher_layers": selected_layers, "diag": {k: v for k, v in diag.items()
                                                                                 if k != "layer_scores"}},
                           init_path)
                fid = init_fidelity_report(student, teacher, calib)
                rl.event("student_initialized", seconds=round(time.time() - t0, 1), selected_layers=selected_layers,
                         bridge_residual=diag.get("bridge_lstsq_residual"), init_fidelity=fid)
                rl.write_json("init_fidelity.json", fid)
                logger.info("INIT FIDELITY (content text frames only): %s", json.dumps(fid))
                logger.info("INIT VERDICT: %s", fid.get("verdict", "n/a"))
                del calib
        obj = [selected_layers if di.main else None]
        if di.world > 1:
            dist.broadcast_object_list(obj, src=0)
        selected_layers = obj[0]
    if len(selected_layers) != student_cfg.num_hidden_layers:
        raise SystemExit(f"selected_teacher_layers has {len(selected_layers)} entries, student has "
                         f"{student_cfg.num_hidden_layers} layers")

    # ---------------------------------------------------------------- precision, depformer state, projections
    def unfreeze_depformer():
        student.unfreeze_depth_transformer()
        if args.master_dtype == "fp32":
            student.depformer.float()          # in-place dtype change keeps the Parameter objects (optimizer refs)

    # include_depformer: the depformer weights differ from the teacher's (trained in P4 now or in the run we
    #   warm-started from) -> they must be saved in checkpoints and shipped in the export.
    # depformer_unfrozen: the depformer is trainable in THIS run (Phase 4 of this schedule).
    include_depformer = bool((payload or {}).get("depformer_trained")) or warm_has_depformer
    phase0 = phase_for(start_step, args.total_steps, args.fixed_phase)
    depformer_unfrozen = False
    if phase0.depth_transformer_unfrozen:
        unfreeze_depformer()
        depformer_unfrozen = include_depformer = True
    if args.master_dtype == "fp32":
        for name in TRAINABLE_TOP:
            getattr(student, name).float()
    if payload is not None:
        ckpt.load_trainable_state_dict(student, payload["student_state_dict"])
    student.transformer.gradient_checkpointing = bool(args.grad_checkpointing)
    student.train()

    projections = nn.ModuleList([HiddenProjection(student_cfg.hidden_size, student_cfg.teacher_reference.dim)
                                 for _ in selected_layers]).to(device=di.device, dtype=torch.float32)
    if payload is not None:
        projections.load_state_dict(payload["hidden_projections_state_dict"])

    groups = param_groups(student, projections, args.weight_decay, 0.1)
    optimizer = torch.optim.AdamW(groups, lr=args.lr, betas=(0.9, 0.95), eps=1e-8,
                                  fused=(di.device.type == "cuda"))
    for g in optimizer.param_groups:
        g["base_lr"] = args.lr * g["lr_mult"]
    if payload is not None:
        optimizer.load_state_dict(payload["optimizer_state_dict"])
        for g in optimizer.param_groups:                 # base lr follows the CURRENT --lr
            g["base_lr"] = args.lr * g["lr_mult"]

    # `txt_kl` / `txt_ce` are the TEXT-ONLY step's own normalizers. They must not share the audio path's
    # `kl_text` / `ce` statistics: the audio-mode text KL is a content-weighted mean over ~1000 mostly-padding
    # frames, while the text-only KL is a plain mean over real tokens -- different magnitudes, and one
    # RunningNorm cannot track both without mis-scaling whichever step type ran least recently.
    norms = {k: SyncedNorm() for k in ("ce", "kl_audio", "kl_text", "bridge", "hidden", "txt_kl", "txt_ce")}
    if payload is not None:
        for k, st in (payload.get("running_norm_states") or {}).items():
            if k in norms and st:
                norms[k].norm.load_state_dict(st)

    wrapper = StudentWithProjections(student, projections)

    def wrap_ddp(find_unused: bool):
        if di.world == 1:
            return wrapper
        return DDP(wrapper, device_ids=[di.local_rank] if di.device.type == "cuda" else None,
                   find_unused_parameters=find_unused, broadcast_buffers=False)
    # After Phase 4 unfreezes the depformer, its steps for the (undistilled) user codebooks get no gradient ->
    # find_unused_parameters is required from then on. Text-only steps need it too, and from step 0: they feed
    # zero_token_id on all 16 audio rows and never call the depformer, so emb.* and depformer.* receive no
    # gradient on those steps. Without find_unused_parameters, DDP aborts the first time that happens.
    def ddp_needs_find_unused() -> bool:
        return (depformer_unfrozen or bool(args.text_data)) and not args.distill_user_stream

    model = wrap_ddp(find_unused=ddp_needs_find_unused())

    if di.device.type == "cuda":
        free_b, total_b = torch.cuda.mem_get_info(di.device)
        static = {"device": str(di.device), "allocated_gb": round(torch.cuda.memory_allocated(di.device) / 1e9, 2),
                  "reserved_gb": round(torch.cuda.memory_reserved(di.device) / 1e9, 2),
                  "device_free_gb": round(free_b / 1e9, 2), "device_total_gb": round(total_b / 1e9, 2)}
        logger.info("static GPU memory after model setup: %s", json.dumps(static))
        rl.event("static_memory_rank0", **static)
    trainable = [p for g in optimizer.param_groups for p in g["params"]]
    n_params = {"student_trainable": sum(p.numel() for p in student.parameters() if p.requires_grad),
                "hidden_projections": sum(p.numel() for p in projections.parameters()),
                "teacher": sum(p.numel() for p in teacher.parameters())}
    autocast_ctx = (lambda: torch.autocast(device_type=di.device.type, dtype=torch.bfloat16)) \
        if args.master_dtype == "fp32" else contextlib.nullcontext
    sampling = {"temp": 0.8, "temp_text": 0.7, "top_k": 250, "top_k_text": 25}

    run_cfg = {"args": vars(args), "world_size": di.world, "grad_accum": accum, "host": socket.gethostname(),
               "torch": torch.__version__, "cuda": torch.version.cuda, "git": git_rev(Path(__file__).parents[1]),
               "selected_teacher_layers": selected_layers, "params": n_params, "start_step": start_step,
               "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
               if torch.cuda.is_available() else []}
    rl.write_json("run_config.json", run_cfg)
    rl.event("start", **{k: run_cfg[k] for k in ("world_size", "grad_accum", "start_step", "params")},
             total_steps=args.total_steps)

    stop = {"flag": False}

    def on_signal(signum, frame):
        stop["flag"] = True
        logger.warning("signal %s received: saving a checkpoint and stopping after this step", signum)
    signal.signal(signal.SIGTERM, on_signal)
    signal.signal(signal.SIGINT, on_signal)

    def check_ranks_in_sync(step: int):
        """All DDP ranks must hold identical weights. Compare a float64 checksum of every trainable tensor."""
        with torch.no_grad():
            chk = torch.zeros(1, device=di.device, dtype=torch.float64)
            for p in trainable:
                chk += p.detach().double().sum()
            if di.world > 1:
                lo, hi = chk.clone(), chk.clone()
                dist.all_reduce(lo, op=dist.ReduceOp.MIN)
                dist.all_reduce(hi, op=dist.ReduceOp.MAX)
                ok = bool((hi - lo).abs().item() <= 1e-6 * max(1.0, abs(hi.item())))
            else:
                ok, lo, hi = True, chk, chk
        logger.info("param checksum step %d: %.12e (ranks in sync: %s)", step, chk.item(), ok)
        if not ok:
            rl.event("rank_desync", step=step, min=lo.item(), max=hi.item())
        return ok

    def save(step: int, tag: str = "latest"):
        check_ranks_in_sync(step)
        if di.main:
            ckpt.save_training_state(
                str(ckpt_path), step=step, student=student, hidden_projections=projections, optimizer=optimizer,
                lr_scheduler=_LRState(step), schedule_state={"phase": phase_for(max(0, step - 1), args.total_steps,
                                                                                 args.fixed_phase).name},
                running_norm_states={k: v.norm.state_dict() for k, v in norms.items()},
                selected_teacher_layers=selected_layers, depformer_trained=include_depformer,
                train_args=vars(args), extra={"best_val": best["value"], "tag": tag})
            if tag == "keep":
                (out / "checkpoints").mkdir(exist_ok=True)
                shutil.copy2(ckpt_path, out / "checkpoints" / f"step_{step:07d}.pt")
            if tag == "best":
                shutil.copy2(ckpt_path, out / "best.pt")
            rl.event("checkpoint", step=step, tag=tag)
        barrier(di)

    best = {"value": (payload or {}).get("extra", {}).get("best_val") if payload else None}

    @torch.no_grad()
    def evaluate(step: int, phase: PhaseSpec):
        if val_ds is None:
            return
        wrapper.eval()
        # a P4 training step leaves capture_layers False (phase.delta == 0); validation always reports
        # val_hidden, so re-enable the L_hidden capture for the duration of the eval loop.
        prev_capture = wrapper.capture_layers
        wrapper.capture_layers = True
        idx = list(range(min(args.val_samples, len(val_ds))))[di.rank::di.world]
        # `agree_text` averages over EVERY valid text frame, ~87-90% of which are PAD/EPAD -- a model that
        # nails the padding/timing pattern and gets every word wrong already scores ~0.8-0.87 on it. The
        # `*_content` / `*_pad` split below is what actually tracks the goal (answering the question): it
        # reports word-frame agreement/CE separately from padding-frame agreement/CE.
        keys = ["kl_text", "kl_audio", "ce_text", "ce_audio", "bridge", "hidden", "agree_text", "agree_cb0",
                "agree_text_content", "agree_text_pad", "ce_text_content", "kl_text_content", "content_frac"]
        sums = torch.zeros(len(keys) + 1, device=di.device, dtype=torch.float64)
        for k in range(0, len(idx), micro):
            b = load_batch(val_ds, idx[k:k + micro], args.chunk_frames, di.device)
            codes, pl = b["codes"], b["prefix_lens"]
            t_cap = run_forward_train_with_hooks(teacher, codes, selected_layers)
            with autocast_ctx():
                s_out = wrapper(codes)
            T = codes.shape[-1]
            conv = (torch.arange(T, device=di.device)[None] >= pl[:, None])
            tm = s_out["text_logits_mask"] & conv[:, None]
            am = s_out["logits_mask"][:, :N_AGENT_CODEBOOKS] & conv[:, None]
            s_a, t_a = s_out["logits"][:, :N_AGENT_CODEBOOKS], t_cap.logits[:, :N_AGENT_CODEBOOKS]
            vals = [kl_fp32(s_out["text_logits"], t_cap.text_logits, tm, TEXT_KL_TEMPERATURE),
                    kl_fp32(s_a, t_a, am, args.audio_kl_temperature),
                    masked_ce(s_out["text_logits"], codes[:, :1], tm & (codes[:, :1] != student.text_initial_token_id)),
                    masked_ce(s_a, codes[:, 1:1 + N_AGENT_CODEBOOKS], am & (codes[:, 1:1 + N_AGENT_CODEBOOKS] != student.initial_token_id)),
                    bridge_loss(s_out["raw_hidden"], t_cap.raw_hidden),
                    torch.stack([hidden_term(s_out["projected"][i], t_cap.layer_hidden[selected_layers[i]])
                                 for i in range(len(selected_layers))]).mean()]
            st = torch.nan_to_num(s_out["text_logits"].float(), nan=0).argmax(-1)
            tt = torch.nan_to_num(t_cap.text_logits.float(), nan=0).argmax(-1)
            sa = torch.nan_to_num(s_a[:, 0].float(), nan=0).argmax(-1)
            ta = torch.nan_to_num(t_a[:, 0].float(), nan=0).argmax(-1)
            vals.append(((st == tt) & tm).sum().float() / tm.sum().clamp_min(1))
            vals.append(((sa == ta) & am[:, 0]).sum().float() / am[:, 0].sum().clamp_min(1))
            # ---- content vs padding split on the text stream (the metric the goal is defined on)
            txt = codes[:, :1]
            is_content = (txt != TEXT_PAD_ID) & (txt != TEXT_EPAD_ID)
            cm, pm = tm & is_content, tm & ~is_content
            vals.append(((st == tt) & cm).sum().float() / cm.sum().clamp_min(1))
            vals.append(((st == tt) & pm).sum().float() / pm.sum().clamp_min(1))
            vals.append(masked_ce(s_out["text_logits"], txt, cm & (txt != student.text_initial_token_id)))
            vals.append(kl_fp32(s_out["text_logits"], t_cap.text_logits, cm, TEXT_KL_TEMPERATURE))
            vals.append(cm.sum().float() / tm.sum().clamp_min(1))
            n = codes.shape[0]
            for i, v in enumerate(vals):
                sums[i] += float(v) * n
            sums[-1] += n
        all_reduce(sums, di)
        wrapper.capture_layers = prev_capture
        wrapper.train()
        if sums[-1] == 0:
            return
        res = {k: float(sums[i] / sums[-1]) for i, k in enumerate(keys)}
        # `best` is selected on the CONTENT text KL (plus audio), not on the padding-dominated text KL: the old
        # criterion `kl_text + kl_audio` is minimized by fitting the silence/timing pattern, so it could call a
        # checkpoint "best" purely for getting better at staying quiet.
        score = args.best_text_weight * res["kl_text_content"] + res["kl_audio"]
        is_best = best["value"] is None or score < best["value"]
        row = {"step": step, "phase": phase.name, **{f"val_{k}": round(v, 5) for k, v in res.items()},
               "val_samples": int(sums[-1]), "best": int(is_best)}
        rl.val_row(row)
        logger.info("VAL step %d %s", step, json.dumps(row))
        if is_best:
            best["value"] = score
            save(step, "best")

    # ---------------------------------------------------------------- training loop
    skipped_total, consecutive_skips = 0, 0
    text_tokens_total = 0        # cumulative text tokens distilled over all ranks (text-only steps)
    phase_times: dict[str, list[float]] = {}
    peak_all = 0.0
    last_phase = None
    step = start_step
    initial_audio_id, text_initial_id = student.initial_token_id, student.text_initial_token_id
    t_run = time.time()
    while step < args.total_steps:
        phase = phase_for(step, args.total_steps, args.fixed_phase)
        if phase.name != last_phase:
            rl.event("phase", step=step, phase=phase.name, weights={k: getattr(phase, k) for k in
                                                                   ("alpha", "beta", "gamma", "delta", "epsilon")},
                     on_policy=phase.on_policy, depformer_unfrozen=phase.depth_transformer_unfrozen)
            last_phase = phase.name
        if phase.depth_transformer_unfrozen and not depformer_unfrozen:
            unfreeze_depformer()
            depformer_unfrozen = include_depformer = True
            trainable = [p for g in optimizer.param_groups for p in g["params"]]
            model = wrap_ddp(find_unused=ddp_needs_find_unused())
            rl.event("depformer_unfrozen", step=step, lr_mult=0.1)
        onpolicy = phase.on_policy and args.onpolicy_fraction > 0 and \
            ((step * 2654435761) % 1000) / 1000.0 < args.onpolicy_fraction      # same decision on every rank
        # Text-only steps replace teacher-forced steps only (an on-policy rollout is about speech behaviour,
        # which text cannot exercise). A different multiplier than `onpolicy` so the two decisions are
        # independent; both are pure functions of `step`, so every rank agrees and a resume reproduces them.
        textstep = (text_stream is not None and not onpolicy and args.text_fraction > 0
                    and ((step * 40503 + 17) % 1000) / 1000.0 < args.text_fraction)
        f = lr_factor(step, args.total_steps, args.warmup_steps, args.min_lr_ratio)
        for g in optimizer.param_groups:
            g["lr"] = g["base_lr"] * f
        if di.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        gidx = sampler.global_indices(step)
        mine = gidx[di.rank * micro * accum:(di.rank + 1) * micro * accum]
        optimizer.zero_grad(set_to_none=True)
        acc = {k: 0.0 for k in ("total", "ce", "kl_audio", "kl_text", "bridge", "hidden")}
        local_ok = True
        n_frames = 0
        # ---- ON-POLICY: Fast batched rollout for all micro-batches of this step.
        # Runs a single streaming rollout for the whole step, moves output to CPU, and frees GPU caches
        # before entering the forward-backward accumulation loop.
        rollout_info = None
        t_roll = 0.0
        if onpolicy:
            all_samples = [train_ds[i] for i in mine]
            all_b = collate_chunks(all_samples, args.chunk_frames)
            all_codes = all_b["codes"].to(di.device, non_blocking=True)
            all_pl = all_b["prefix_lens"]
            start = int(all_pl.max().item()) + args.onpolicy_context_frames
            n_total = min(all_codes.shape[-1], start + args.rollout_frames)
            if n_total - start >= 8:
                if di.device.type == "cuda":
                    torch.cuda.synchronize()
                t_r0 = time.time()
                torch.manual_seed(args.seed * 1_000_003 + step * 997 + di.rank * 31)
                rb = args.rollout_batch_size if args.rollout_batch_size > 0 else all_codes.shape[0]
                gens = [student_rollout(student, all_codes[i:i + rb], start, n_total, autocast_ctx(), sampling)
                        for i in range(0, all_codes.shape[0], rb)]
                gen = gens[0] if len(gens) == 1 else torch.cat(gens, dim=0)
                if not (torch.equal(gen[:, :, :start], all_codes[:, :, :start])
                        and torch.equal(gen[:, 9:, :n_total], all_codes[:, 9:, :n_total])):
                    raise RuntimeError("on-policy rollout is misaligned with the data (forced frames differ) -- "
                                       "this is a bug; refusing to train on it")
                gen_cpu = gen.cpu()
                del gen, gens, all_codes, all_pl, all_b, all_samples
                if di.device.type == "cuda":
                    torch.cuda.empty_cache()
                t_roll = time.time() - t_r0
                rollout_info = (gen_cpu, start, n_total)

        # ---- TEXT-ONLY STEP: distill the teacher's text distribution, one position per token -----------------
        if textstep:
            tb = torch.from_numpy(text_stream.batch(step, di.rank, di.world, micro, accum)).to(di.device)
            n_text_tokens = 0
            for m in range(accum):
                w = tb[m * micro:(m + 1) * micro]                     # [micro, seq_len + 1]
                text_in, targets = w[:, :-1].contiguous(), w[:, 1:].contiguous()
                n_text_tokens += targets.numel()
                sync = model.no_sync() if (di.world > 1 and m < accum - 1) else contextlib.nullcontext()
                with sync:
                    with torch.no_grad():
                        t_text = text_only_text_logits(teacher, text_in)
                    with autocast_ctx():
                        s_text = model(text_in, text_only=True)["text_logits"]
                    tt = text_only_terms(s_text, t_text, targets, TEXT_KL_TEMPERATURE)
                    total = (tt["kl_text"] / norms["txt_kl"].scale(tt["kl_text"].item())
                             + args.text_ce_weight * tt["ce"] / norms["txt_ce"].scale(tt["ce"].item()))
                    if not torch.isfinite(total):
                        local_ok = False
                    (total / accum).backward()
                acc["kl_text"] += float(tt["kl_text"].detach()) / accum
                acc["ce"] += float(tt["ce"].detach()) / accum
                acc["total"] += float(total.detach()) / accum
                del t_text, s_text, tt, total
            n_frames = n_text_tokens        # reported as frames_per_s: text tokens/s on a text step
            del tb

        for m in (range(accum) if not textstep else ()):
            b = load_batch(train_ds, mine[m * micro:(m + 1) * micro], args.chunk_frames, di.device)
            codes, tmask, pl = b["codes"], b["transition_mask"], b["prefix_lens"]
            ce_end = None
            if rollout_info is not None:
                gen_cpu, start, n_total = rollout_info
                codes = gen_cpu[m * micro:(m + 1) * micro, :, :n_total].to(di.device, non_blocking=True)
                # `transition_frames` came from the RECORDED conversation's turn-taking events. Past `start`
                # the agent channel is the student's own rollout, so those frames no longer mark anything that
                # happened -- keeping them made `frame_weights` apply its 4x transition boost to arbitrary
                # frames. Only the teacher-forced prefix region still has valid transitions.
                tmask = tmask[:, :n_total].clone()
                tmask[:, start:] = False
                ce_end = torch.full((codes.shape[0],), start, device=di.device, dtype=torch.long)
            n_frames += codes.numel() // codes.shape[1]
            sync = model.no_sync() if (di.world > 1 and m < accum - 1) else contextlib.nullcontext()
            need_hidden = phase.delta > 0
            wrapper.capture_layers = need_hidden
            with sync:
                with torch.no_grad():
                    t_cap = run_forward_train_with_hooks(teacher, codes, selected_layers if need_hidden else [])
                with autocast_ctx():
                    s_out = model(codes)
                terms = loss_terms(s_out, t_cap, codes, tmask, pl, ce_end, selected_layers, args,
                                   initial_audio_id, text_initial_id)
                total = (phase.alpha * terms["ce"] / norms["ce"].scale(terms["ce"].item())
                         + phase.beta * terms["kl_audio"] / norms["kl_audio"].scale(terms["kl_audio"].item())
                         + args.beta_text * terms["kl_text"] / norms["kl_text"].scale(terms["kl_text"].item())
                         + phase.gamma * terms["bridge"] / norms["bridge"].scale(terms["bridge"].item())
                         + phase.delta * terms["hidden"] / norms["hidden"].scale(terms["hidden"].item()))
                # L_speaker (phase.epsilon) is a validation metric only in this repo (see losses.py) -> not trained.
                if not torch.isfinite(total):
                    local_ok = False
                (total / accum).backward()
            for k in ("ce", "kl_audio", "kl_text", "bridge", "hidden"):
                acc[k] += float(terms[k].detach()) / accum
            acc["total"] += float(total.detach()) / accum
            del t_cap, s_out, terms, total, b, codes, tmask, pl
        del rollout_info

        grad_norm = torch.nn.utils.clip_grad_norm_([p for p in trainable if p.grad is not None], args.max_grad_norm)
        flag = torch.tensor([1.0 if (local_ok and torch.isfinite(grad_norm)) else 0.0], device=di.device)
        all_reduce(flag, di, dist.ReduceOp.MIN if di.world > 1 else None)
        if flag.item() > 0:
            optimizer.step()
            consecutive_skips = 0
        else:
            optimizer.zero_grad(set_to_none=True)
            skipped_total += 1
            consecutive_skips += 1
            rl.event("skipped_nonfinite_step", step=step, local_ok=local_ok, grad_norm=float(grad_norm))
            if consecutive_skips >= args.max_consecutive_skips:
                save(step, "latest")
                raise RuntimeError(f"{consecutive_skips} consecutive non-finite steps -- aborting (checkpoint saved)")

        # ---- metrics: all-reduce means, update the shared loss normalizers
        vec = torch.tensor([acc[k] for k in ("total", "ce", "kl_audio", "kl_text", "bridge", "hidden")] + [n_frames],
                           device=di.device, dtype=torch.float64)
        all_reduce(vec, di)
        vals = (vec[:6] / di.world).tolist()
        g_frames = float(vec[6])
        if flag.item() > 0:
            # Only the norms whose term actually ran this step. A text step leaves kl_audio/bridge/hidden at
            # 0.0, so feeding those zeros into their RunningNorms would drag the statistics toward zero and
            # then blow those terms up on the next audio step.
            if textstep:
                norms["txt_kl"].update(vals[3])
                norms["txt_ce"].update(vals[1])
            else:
                norms["ce"].update(vals[1])
                norms["kl_audio"].update(vals[2])
                norms["kl_text"].update(vals[3])
                norms["bridge"].update(vals[4])
                norms["hidden"].update(vals[5])
        dt = time.time() - t0
        peak = torch.cuda.max_memory_allocated() / 1e9 if di.device.type == "cuda" else 0.0
        pk = torch.tensor([peak], device=di.device)
        all_reduce(pk, di, dist.ReduceOp.MAX if di.world > 1 else None)
        peak_all = max(peak_all, float(pk))
        if textstep:
            text_tokens_total += int(g_frames)        # g_frames is the all-reduced token count on a text step
        key = f"{phase.name}{'_onpolicy' if onpolicy else '_text' if textstep else ''}"
        phase_times.setdefault(key, []).append(dt)
        step += 1
        if step % args.log_every == 0 or step == args.total_steps or step == start_step + 1:
            row = {"step": step, "phase": phase.name, "total": round(vals[0], 5), "ce_raw": round(vals[1], 5),
                   "kl_audio_raw": round(vals[2], 5), "kl_text_raw": round(vals[3], 5), "bridge_raw": round(vals[4], 5),
                   "hidden_raw": round(vals[5], 5), "seconds_per_step": round(dt, 3), "lr": f"{args.lr * f:.3e}",
                   "grad_norm": round(float(grad_norm), 4), "onpolicy": int(onpolicy), "textstep": int(textstep),
                   "text_tokens_total": text_tokens_total,
                   "frames_per_s": round(g_frames / max(dt, 1e-6), 1), "peak_mem_gb": round(float(pk), 2),
                   "skipped_total": skipped_total, "epoch": round(sampler.epoch_of(step), 3)}
            rl.train_row(row)
            logger.info("step %d/%d %s total=%.4f ce=%.4f kl=%.4f+%.4f bridge=%.4f hidden=%.4f gn=%.3f lr=%s "
                        "%.2fs/step peak=%.1fGB%s", step, args.total_steps, phase.name, vals[0], vals[1], vals[2],
                        vals[3], vals[4], vals[5], float(grad_norm), row["lr"], dt, float(pk),
                        (" [on-policy, rollout %.1fs = %.0f%% of step]" % (t_roll, 100 * t_roll / max(dt, 1e-6)))
                        if onpolicy else (" [text-only, %d text tokens so far]" % text_tokens_total)
                        if textstep else "")
            # ETA from measured per-phase step times
            eta = 0.0
            for s2 in range(step, args.total_steps, max(1, (args.total_steps - step) // 200 or 1)):
                ph = phase_for(s2, args.total_steps, args.fixed_phase).name
                cand = [v for k2, v in phase_times.items() if k2.startswith(ph)]
                mean = np.mean([x for v in cand for x in v[-50:]]) if cand else dt
                eta += mean * max(1, (args.total_steps - step) // 200 or 1)
            rl.write_json("heartbeat.json", {"time": time.strftime("%Y-%m-%dT%H:%M:%S"), "step": step,
                                             "total_steps": args.total_steps, "phase": phase.name,
                                             "eta_hours": round(eta / 3600, 2), "skipped_total": skipped_total,
                                             "peak_mem_gb": round(peak_all, 2), "best_val": best["value"]})
        stop_t = torch.tensor([1.0 if stop["flag"] else 0.0], device=di.device)
        all_reduce(stop_t, di, dist.ReduceOp.MAX if di.world > 1 else None)
        if args.eval_every and (step % args.eval_every == 0 or step == args.total_steps):
            evaluate(step, phase)
        if stop_t.item() > 0:
            save(step, "latest")
            rl.event("stopped_by_signal", step=step)
            break
        if step % args.save_every == 0 or step == args.total_steps:
            save(step, "latest")
        if args.keep_every and step % args.keep_every == 0:
            save(step, "keep")

    # ---------------------------------------------------------------- export + summary
    finished = step >= args.total_steps
    if finished and di.main:
        exp_dir = out / "export"
        exp_dir.mkdir(exist_ok=True)
        from .export import export_bf16
        export_bf16(student, str(exp_dir / f"{student_cfg.name}_bf16.safetensors"), selected_layers,
                    include_depformer=include_depformer)
        if (out / "best.pt").exists():
            # the final weights are already exported above, so the live model can be overwritten with the best
            # checkpoint's weights (no second 17 GB teacher load into CPU RAM). If best.pt predates Phase 4 it has
            # no depformer keys -> exported without depformer -> inference uses the teacher's original one, exactly
            # the state that checkpoint was evaluated in.
            bp = ckpt.load_training_payload(str(out / "best.pt"))
            ckpt.load_trainable_state_dict(student, bp["student_state_dict"])
            export_bf16(student, str(exp_dir / f"{student_cfg.name}_best_bf16.safetensors"), selected_layers,
                        include_depformer=bool(bp.get("depformer_trained")))
        rl.event("exported", dir=str(exp_dir), depformer_included=include_depformer)
    summary = {"finished": finished, "steps_done": step, "start_step": start_step, "skipped_nonfinite": skipped_total,
               "peak_mem_gb": round(peak_all, 2), "best_val_kl": best["value"], "world_size": di.world,
               "seconds": round(time.time() - t_run, 1),
               "mean_step_seconds_by_phase": {k: round(float(np.mean(v)), 3) for k, v in phase_times.items()},
               "depformer_included": include_depformer, "depformer_trained_this_run": depformer_unfrozen}
    rl.write_json("smoke_summary.json" if args.smoke_test else "run_summary.json", summary)
    rl.event("end", **summary)
    barrier(di)
    if di.world > 1:
        dist.destroy_process_group()


class _LRState:
    """The LR is a pure function of the step (lr_factor), so the 'scheduler state' is just the step."""

    def __init__(self, step):
        self.step = step

    def state_dict(self):
        return {"step": self.step}


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    # --- original arguments (unchanged meaning, except --batch-size is now the GLOBAL batch)
    ap.add_argument("--student-config", required=True, help="e.g. student_ppx_s")
    ap.add_argument("--teacher-checkpoint", required=True)
    ap.add_argument("--data-dir", required=True, help="training split dir (manifest.jsonl + codes/)")
    ap.add_argument("--output-dir", required=True)
    ap.add_argument("--total-steps", type=int, default=200_000, help="optimizer steps")
    ap.add_argument("--batch-size", type=int, default=8, help="GLOBAL batch = world x micro-batch x grad-accum")
    ap.add_argument("--chunk-frames", type=int, default=1000, help="must equal the dataset's frames (main 1000 / long 1500)")
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--audio-kl-temperature", type=float, default=1.0)
    ap.add_argument("--text-content-boost", type=float, default=8.0,
                    help="extra weight on text frames carrying a real word (not PAD/EPAD, not the prompt "
                         "prefix) in the text KL and the text CE. 0 = the old unweighted behaviour, where "
                         "~90%% of the text gradient trains padding/timing. 8.0 puts roughly half the text "
                         "gradient on content words at no extra compute.")
    # ---- text-only distillation (the stage that restores the knowledge the init truncated) ----------------
    ap.add_argument("--text-data", nargs="*", default=None,
                    help="text files / directories / globs (.txt .md .jsonl .json) to distill the teacher's "
                         "TEXT distribution on, interleaved with the audio steps. The corpus only has to "
                         "COVER the domain -- the teacher supplies the targets, so no answers are needed in "
                         "it. Omit to train exactly as before (audio only).")
    ap.add_argument("--tokenizer", default=None,
                    help="teacher SentencePiece model (tokenizer_spm_32k_3.model); required with --text-data")
    ap.add_argument("--text-cache", default=None,
                    help="path for the tokenized uint16 .npy cache (default <output-dir>/text_tokens.npy). "
                         "Reused if it exists -- delete it to retokenize.")
    ap.add_argument("--text-fraction", type=float, default=0.25,
                    help="share of TEACHER-FORCED steps that run as text-only steps instead. On-policy steps "
                         "are never replaced. 0 = audio only. Lowered from 0.5: at 0.5 half of every update "
                         "fed zero_token_id on all 16 audio rows, which halved the audio-conditioned "
                         "training (and left emb[0..15] with gradient on only half the steps) for a corpus "
                         "small enough to be seen ~100 times. A three-way probe (question / different "
                         "question / silence) confirmed conditioning itself survived -- silence produced "
                         "only a greeting -- so this is a budget-allocation setting, not a correctness one.")
    ap.add_argument("--text-seq-len", type=int, default=512,
                    help="text tokens per sequence in a text-only step (must be <= the student's max_frames)")
    ap.add_argument("--text-max-tokens", type=int, default=None,
                    help="cap the tokenized corpus at N tokens (for quick experiments)")
    ap.add_argument("--text-ce-weight", type=float, default=0.2,
                    help="hard-label CE weight on text-only steps (the KL term carries weight 1.0 there)")
    ap.add_argument("--best-text-weight", type=float, default=1.0,
                    help="weight of the CONTENT text KL in the best-checkpoint score "
                         "(score = w * val_kl_text_content + val_kl_audio)")
    ap.add_argument("--beta-text", type=float, default=2.0,
                    help="weight of the text KL term. Separate from --beta/phase.beta (which now weights the "
                         "AUDIO KL only), so text pressure can be raised without changing audio pressure.")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--save-every", type=int, default=500)
    ap.add_argument("--smoke-test", action="store_true", help="all phases (incl. on-policy), eval, checkpoint, export in --smoke-steps steps")
    # --- new
    ap.add_argument("--val-dir", default=None)
    ap.add_argument("--micro-batch", type=int, default=1, help="samples per GPU per forward")
    ap.add_argument("--master-dtype", choices=["fp32", "bf16"], default="fp32",
                    help="fp32 = fp32 master weights + bf16 autocast (recommended); bf16 = legacy, lower memory")
    ap.add_argument("--grad-checkpointing", action="store_true", help="recompute student layer activations in backward")
    ap.add_argument("--weight-decay", type=float, default=0.01)
    ap.add_argument("--warmup-steps", type=int, default=1000)
    ap.add_argument("--min-lr-ratio", type=float, default=0.1)
    ap.add_argument("--max-grad-norm", type=float, default=1.0)
    ap.add_argument("--eval-every", type=int, default=1000)
    ap.add_argument("--val-samples", type=int, default=256)
    ap.add_argument("--keep-every", type=int, default=5000, help="also keep checkpoints/step_N.pt every N steps (0 = off)")
    ap.add_argument("--log-every", type=int, default=10)
    ap.add_argument("--seed", type=int, default=1234)
    ap.add_argument("--onpolicy-fraction", type=float, default=0.5, help="share of P3/P4 steps that use student rollouts")
    ap.add_argument("--rollout-frames", type=int, default=250, help="frames the student generates per rollout (20 s)")
    ap.add_argument("--rollout-batch-size", type=int, default=0,
                    help="samples rolled out together in ONE streaming pass (0 = the rank's whole per-step batch, "
                         "micro_batch x grad_accum). Rollout cost is per-call (launch/latency bound), not per-sample, "
                         "so a bigger rollout batch is ~free. Lower it only if the rollout OOMs.")
    ap.add_argument("--onpolicy-context-frames", type=int, default=50,
                    help="real conversation frames after the longest prefix before the student takes over")
    ap.add_argument("--ce-on-prefix", action="store_true", help="also apply hard-label CE on prompt-prefix frames")
    ap.add_argument("--no-kl-on-prefix", dest="kl_on_prefix", action="store_false",
                    help="exclude prompt-prefix frames from KL too (default: included)")
    ap.add_argument("--distill-user-stream", action="store_true",
                    help="also distill the teacher's user-stream codebooks (off: never used at inference, see docstring)")
    ap.add_argument("--calib-batches", type=int, default=64,
                    help="init calibration batches (4 samples each). The residual SVD fits a 4096x4096 "
                         "covariance and the bridge solves for hidden_size unknowns per output column, so 16 "
                         "batches (32k frames) was thin; 64 gives ~128k answer-phase frames.")
    ap.add_argument("--calib-frames", type=int, default=500, help="frames per calibration sample")
    ap.add_argument("--calib-skip-frames", type=int, default=150,
                    help="frames to skip PAST each sample's prompt prefix before taking calibration frames, so "
                         "the init's SVD basis / FFN channel ranking / layer scores are measured while the "
                         "teacher is answering rather than while it loads the voice prompt and says hello. "
                         "~150 frames = 12 s at 12.5 Hz. 0 still starts at the end of the prefix (the prompt "
                         "is excluded either way) -- it just includes the greeting.")
    ap.add_argument("--layer-selection", choices=["spread", "score"], default="spread",
                    help="how init_from_teacher picks which teacher layers to keep. 'score' is pure "
                         "1-cos(h[l],h[l+1]) ranking, which on the real teacher dropped layers 17-28 -- a "
                         "contiguous block of 12 -- because late-middle layers all rank last together. "
                         "'spread' spaces the drops and takes the lowest-scoring layer in each bucket.")
    ap.add_argument("--calib-prefer-scenario", default="multi_turn,single_followup,repeat_clarify",
                    help="comma-separated scenarios to prefer for calibration (knowledge-heavy Q&A rather than "
                         "backchannel/distractor/casual turn-taking). Empty = no preference.")
    ap.add_argument("--init-from", default=None, help="warm-start weights from a checkpoint (new run, new optimizer)")
    ap.add_argument("--fixed-phase", default=None, choices=[p.name for p in PHASES],
                    help="use one phase for the whole run (e.g. P2_behavior for the long-context fine-tune)")
    ap.add_argument("--max-consecutive-skips", type=int, default=50)
    ap.add_argument("--smoke-steps", type=int, default=12)
    ap.add_argument("--teacher-kwargs-json", default=None, help=argparse.SUPPRESS)   # tests only
    args = ap.parse_args()
    try:
        train(args)
    except Exception:
        logging.getLogger("distill.train").error("FATAL:\n%s", traceback.format_exc())
        raise


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {REPO_DIR}/distill/checkpoint.py
# SPDX-License-Identifier: MIT
"""Resumable checkpointing for student training.

The saved checkpoint holds ONLY the trained delta: the student's transformer,
embeddings, and bridge (plus the L_hidden projection heads, optimizer, and
schedule/RNG state). It never re-saves the frozen depformer/heads/Mimi weights
that are reused verbatim from the teacher checkpoint at load time -- "don't
duplicate them" (see PersonaPlex_Distill_RunPod.ipynb export step and
moshi/moshi/models/loaders.get_student_lm).
"""

from dataclasses import dataclass
from pathlib import Path
import random
import typing as tp

import numpy as np
import torch

from .student_model import StudentLMModel, FROZEN_PREFIXES


def trainable_state_dict(model: StudentLMModel, include_depformer: bool = False) -> dict[str, torch.Tensor]:
    """The student's own weights (transformer, embeddings, bridge). With include_depformer=True also the depth
    transformer (`depformer.*`), which Phase 4 fine-tunes -- without it those P4 updates would be lost. The other
    frozen teacher parts (depformer_in/emb, linears, text_linear, out_norm) are never trained and never saved."""
    full = model.state_dict()
    return {k: v for k, v in full.items()
            if not k.startswith(FROZEN_PREFIXES) or (include_depformer and k.startswith("depformer."))}


def load_trainable_state_dict(model: StudentLMModel, state_dict: dict[str, torch.Tensor]):
    result = model.load_state_dict(state_dict, strict=False, assign=False)
    unexpected_non_frozen = [k for k in result.unexpected_keys if not k.startswith(FROZEN_PREFIXES)]
    assert not unexpected_non_frozen, unexpected_non_frozen
    missing_non_frozen = [k for k in result.missing_keys if not k.startswith(FROZEN_PREFIXES)]
    assert not missing_non_frozen, f"Checkpoint is missing trainable keys: {missing_non_frozen}"


@dataclass
class TrainingCheckpoint:
    step: int
    student_config_name: str
    selected_teacher_layers: list[int]


def save(
    path: str,
    step: int,
    student: StudentLMModel,
    hidden_projections: torch.nn.ModuleList,
    optimizer: torch.optim.Optimizer,
    schedule_state: dict,
    running_norm_states: dict,
    selected_teacher_layers: list[int],
    extra: tp.Optional[dict] = None,
):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "step": step,
        "student_config_name": student.student_config.name,
        "student_state_dict": trainable_state_dict(student),
        "hidden_projections_state_dict": hidden_projections.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "schedule_state": schedule_state,
        "running_norm_states": running_norm_states,
        "selected_teacher_layers": selected_teacher_layers,
        "rng_state": {
            "torch": torch.get_rng_state(),
            "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "numpy": np.random.get_state(),
            "python": random.getstate(),
        },
        "extra": extra or {},
    }
    tmp_path = f"{path}.tmp"
    torch.save(payload, tmp_path)
    Path(tmp_path).replace(path)


def load(
    path: str,
    student: StudentLMModel,
    hidden_projections: torch.nn.ModuleList,
    optimizer: tp.Optional[torch.optim.Optimizer] = None,
    map_location: str = "cpu",
) -> dict:
    # weights_only=False: the payload holds numpy/Python RNG state alongside the tensors, which
    # torch>=2.6's default weights_only=True refuses to unpickle. These are our own checkpoints.
    payload = torch.load(path, map_location=map_location, weights_only=False)
    assert payload["student_config_name"] == student.student_config.name, (
        f"Checkpoint was trained with config {payload['student_config_name']!r}, "
        f"but model was built with {student.student_config.name!r}."
    )
    load_trainable_state_dict(student, payload["student_state_dict"])
    hidden_projections.load_state_dict(payload["hidden_projections_state_dict"])
    if optimizer is not None:
        optimizer.load_state_dict(payload["optimizer_state_dict"])

    rng = payload["rng_state"]
    torch.set_rng_state(rng["torch"])
    if rng["torch_cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(rng["torch_cuda"])
    np.random.set_state(rng["numpy"])
    random.setstate(rng["python"])

    return payload


# --------------------------------------------------------------------------------------------------------------
# Full training state (used by distill/train.py). Everything needed to resume bit-for-bit in data order and
# optimizer state; RNG streams are re-derived from (seed, step, rank) by the trainer, so they need no saving.
# --------------------------------------------------------------------------------------------------------------
def save_training_state(path: str, *, step: int, student: StudentLMModel, hidden_projections: torch.nn.Module,
                        optimizer: torch.optim.Optimizer, lr_scheduler, schedule_state: dict,
                        running_norm_states: dict, selected_teacher_layers: list[int], depformer_trained: bool,
                        train_args: dict, extra: tp.Optional[dict] = None):
    """Atomic save (write to .tmp, then rename) so a crash mid-save never corrupts the last good checkpoint."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "format": "ppx_student_train_v2",
        "step": step,
        "student_config_name": student.student_config.name,
        # stored in the TRAINING dtype (fp32 master weights when --master-dtype fp32) for an exact resume
        "student_state_dict": trainable_state_dict(student, include_depformer=depformer_trained),
        "depformer_trained": depformer_trained,
        "hidden_projections_state_dict": hidden_projections.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "lr_scheduler_state_dict": lr_scheduler.state_dict(),
        "schedule_state": schedule_state,
        "running_norm_states": running_norm_states,
        "selected_teacher_layers": selected_teacher_layers,
        "train_args": train_args,
        "extra": extra or {},
    }
    tmp_path = f"{path}.tmp"
    torch.save(payload, tmp_path)
    Path(tmp_path).replace(path)


def load_training_payload(path: str) -> dict:
    # weights_only=False: our own checkpoints (python dicts / numbers alongside tensors)
    return torch.load(path, map_location="cpu", weights_only=False)


In [ ]:
%%writefile {REPO_DIR}/distill/export.py
# SPDX-License-Identifier: MIT
"""Export a trained student to inference-ready checkpoints:
  1. BF16 checkpoint: the trainable delta only (transformer, embeddings,
     bridge) -- frozen depformer/heads/Mimi are never duplicated, they're
     always loaded from the teacher checkpoint at inference time
     (moshi/models/loaders.get_student_lm).
  2. AWQ INT4 checkpoint: same, but the temporal transformer's Linear weights
     (q_proj/k_proj/v_proj/out_proj/linear_in/linear_out) are additionally
     quantized to INT4 (W4A16) via distill/awq_quant.py. Everything else in
     the exported file (bridge, embeddings) stays BF16 -- per the quantization
     policy, INT4 error is only acceptable in the temporal transformer, never
     in the bridge/heads/Mimi (those aren't even part of this export; heads
     and Mimi are always the teacher's, loaded separately).

Note on the plan's "FP16" wording: this repo uses BF16 uniformly for the LM
(see moshi/models/loaders.get_moshi_lm's default dtype) -- there is no FP16
path anywhere in the existing code to match, so exports here use BF16 for
every non-INT4 tensor rather than introducing a third dtype.
"""

from pathlib import Path
import json
import typing as tp

import torch
from safetensors.torch import save_file, load_file

from .awq_quant import quantize_student_temporal_transformer, TEMPORAL_LINEAR_NAMES, QuantizedLinear
from .checkpoint import trainable_state_dict
from .student_model import StudentLMModel


def export_bf16(student: StudentLMModel, output_path: str, selected_teacher_layers: list[int],
                include_depformer: bool = False):
    """include_depformer=True when Phase 4 fine-tuned the depth transformer: those weights then ship in the export
    and `load_export` (strict=False load into a model built from the teacher checkpoint) overrides the teacher's
    frozen copy with them. Without it the P4 depformer updates would silently be lost."""
    state_dict = {k: v.to(torch.bfloat16)
                  for k, v in trainable_state_dict(student, include_depformer=include_depformer).items()}
    metadata = {
        "student_config_name": student.student_config.name,
        "selected_teacher_layers": json.dumps(selected_teacher_layers),
        "quantization": "none",
        "depformer_included": str(bool(include_depformer)),
    }
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    save_file(state_dict, output_path, metadata=metadata)


def export_awq_int4(
    student: StudentLMModel,
    calibration_inputs: tp.Sequence[torch.Tensor],
    output_path: str,
    selected_teacher_layers: list[int],
    group_size: int = 128,
    include_depformer: bool = False,
):
    quantized = quantize_student_temporal_transformer(student, calibration_inputs, group_size=group_size)

    state_dict: dict[str, torch.Tensor] = {}
    for name, q in quantized.items():
        prefix = name.removesuffix(".weight")
        state_dict[f"{prefix}.qweight"] = q.packed
        state_dict[f"{prefix}.scales"] = q.scales.to(torch.bfloat16)
        state_dict[f"{prefix}.zeros"] = q.zeros
        state_dict[f"{prefix}.awq_scale"] = q.awq_scale.to(torch.bfloat16)

    non_quantized_prefixes = tuple(f".{n}.weight" for n in TEMPORAL_LINEAR_NAMES)
    for k, v in trainable_state_dict(student, include_depformer=include_depformer).items():
        if k.startswith("transformer.") and k.endswith(non_quantized_prefixes):
            continue  # replaced by the quantized tensors above
        state_dict[k] = v.to(torch.bfloat16)

    metadata = {
        "student_config_name": student.student_config.name,
        "selected_teacher_layers": json.dumps(selected_teacher_layers),
        "quantization": "awq_int4_temporal_linears_only",
        "group_size": str(group_size),
        "depformer_included": str(bool(include_depformer)),
    }
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    save_file(state_dict, output_path, metadata=metadata)


def load_export_metadata(path: str) -> dict:
    with open(path, "rb") as f:
        header_len = int.from_bytes(f.read(8), "little")
        header = json.loads(f.read(header_len))
    return header.get("__metadata__", {})


def load_export(path: str, student: StudentLMModel):
    """Loads a checkpoint produced by `export_bf16` or `export_awq_int4` into an
    already-built `StudentLMModel` (i.e. after `build_student_lm`, so the frozen
    teacher submodules are already in place). For an AWQ export, the temporal
    transformer's Linear weights are dequantized to the student's working dtype
    at load time (no custom INT4 GEMM kernel -- see awq_quant.py docstring): the
    on-disk file is 4-bit, the in-memory weights used for matmul are not.
    """
    metadata = load_export_metadata(path)
    tensors = load_file(path, device="cpu")
    working_dtype = student.text_linear.weight.dtype

    if metadata.get("quantization", "none") == "none":
        state_dict = {k: v.to(working_dtype) for k, v in tensors.items()}
        student.load_state_dict(state_dict, strict=False, assign=False)
        return

    assert metadata["quantization"] == "awq_int4_temporal_linears_only", metadata["quantization"]
    group_size = int(metadata["group_size"])

    quantized_prefixes = set()
    for key in tensors:
        if key.endswith(".qweight"):
            quantized_prefixes.add(key.removesuffix(".qweight"))

    plain_state_dict = {
        k: v.to(working_dtype) for k, v in tensors.items()
        if not any(k.startswith(p + ".") for p in quantized_prefixes)
    }
    student.load_state_dict(plain_state_dict, strict=False, assign=False)

    for prefix in quantized_prefixes:
        packed = tensors[f"{prefix}.qweight"]
        scales = tensors[f"{prefix}.scales"].float()
        zeros = tensors[f"{prefix}.zeros"]
        awq_scale = tensors[f"{prefix}.awq_scale"].float()
        # prefix looks like "transformer.layers.<i>.self_attn.q_proj"
        module = student
        for part in prefix.split("."):
            module = getattr(module, part) if not part.isdigit() else module[int(part)]
        out_features, in_features = module.weight.shape
        q = QuantizedLinear(packed=packed, scales=scales, zeros=zeros, group_size=group_size,
                             in_features=in_features, out_features=out_features)
        weight = q.dequantize(working_dtype) / awq_scale.to(working_dtype).unsqueeze(0)
        module.weight.data.copy_(weight)


In [ ]:
%%writefile {REPO_DIR}/distill/data/dataset.py
# SPDX-License-Identifier: MIT
"""Reads the manifest + token-sequence files produced by `generate_teacher.py`.

Primary path: teacher logits are computed on the fly at train time from the
resident teacher model (see distill/train.py) -- this dataset only serves
token sequences and prompt/scenario metadata, matching the "store only token
sequences + prompts" storage policy.

Fallback path: `TopKLogitCache` lets a training run consume pre-computed
top-64 teacher logits instead, for setups where decoupling generation from
training is worth the extra disk (this is NOT the default).
"""

from dataclasses import dataclass
from pathlib import Path
import json
import typing as tp

import torch
from torch.utils.data import Dataset


@dataclass
class Sample:
    sample_id: str
    voice_prompt: str
    persona_text: str
    scenario: str
    codes: torch.Tensor  # [num_codebooks, T]
    transition_frames: list[int]
    # Frames [0, prefix_len) are the voice + persona prompt (forced tokens at inference). Present in datasets made
    # by PersonaPlex_Student_Dataset_Generation.ipynb (prefix_len = 136 + persona tokens); 0 for older datasets
    # generated by distill/data/generate_teacher.py, whose stored sequences start AFTER the prompt.
    prefix_len: int = 0

    def transition_mask(self) -> torch.Tensor:
        mask = torch.zeros(self.codes.shape[-1], dtype=torch.bool)
        for f in self.transition_frames:
            if 0 <= f < mask.shape[0]:
                mask[f] = True
        return mask


class TeacherTokenDataset(Dataset):
    """One item per generated sample. `__getitem__` returns a `Sample`; batching
    (padding to a common length, chunking long sequences into training windows)
    is left to a `collate_fn` / sampler in `distill/train.py` since the right
    chunking policy depends on the training phase (teacher-forced windows vs.
    full-episode on-policy rollouts).
    """

    def __init__(self, data_dir: str):
        self.data_dir = Path(data_dir)
        manifest_path = self.data_dir / "manifest.jsonl"
        if not manifest_path.exists():
            raise FileNotFoundError(
                f"No manifest.jsonl in {data_dir} -- run distill/data/generate_teacher.py first."
            )
        self.entries = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines()]

    def __len__(self) -> int:
        return len(self.entries)

    def __getitem__(self, idx: int) -> Sample:
        entry = self.entries[idx]
        codes = torch.load(self.data_dir / "codes" / f"{entry['sample_id']}.pt", map_location="cpu")
        return Sample(
            sample_id=entry["sample_id"],
            voice_prompt=entry["voice_prompt"],
            persona_text=entry["persona_text"],
            scenario=entry["scenario"],
            codes=codes,
            transition_frames=entry["transition_frames"],
            prefix_len=int(entry.get("prefix_len", 0)),
        )

    def total_frames(self) -> int:
        return sum(e["num_frames"] for e in self.entries)


def collate_chunks(samples: list[Sample], chunk_frames: int) -> dict[str, torch.Tensor]:
    """Pads/truncates a list of `Sample`s to `chunk_frames` and stacks them into
    a batch. Samples shorter than `chunk_frames` are dropped (a training run
    should pick `chunk_frames` well below the shortest generated sample)."""
    usable = [s for s in samples if s.codes.shape[-1] >= chunk_frames]
    if not usable:
        raise ValueError(f"No samples have >= {chunk_frames} frames in this batch.")
    codes = torch.stack([s.codes[:, :chunk_frames] for s in usable], dim=0)
    transition_mask = torch.stack([s.transition_mask()[:chunk_frames] for s in usable], dim=0)
    prefix_lens = torch.tensor([min(s.prefix_len, chunk_frames) for s in usable], dtype=torch.long)
    return {"codes": codes, "transition_mask": transition_mask, "prefix_lens": prefix_lens,
            "sample_ids": [s.sample_id for s in usable]}


class TopKLogitCache:
    """Fallback storage for pre-computed top-64 teacher logits, keyed by
    `sample_id`. Layout on disk: `<cache_dir>/<sample_id>.pt` holding a dict with
    `text_topk_ids/text_topk_vals` [T, 64] and `audio_topk_ids/audio_topk_vals`
    [K, T, 64]. NOT populated by default -- see module docstring.
    """

    TOPK = 64

    def __init__(self, cache_dir: str):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def _topk(logits: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        vals, ids = torch.topk(logits.float(), TopKLogitCache.TOPK, dim=-1)
        return ids.to(torch.int32), vals.to(torch.float16)

    def write(self, sample_id: str, text_logits: torch.Tensor, audio_logits: torch.Tensor):
        text_ids, text_vals = self._topk(text_logits)
        audio_ids, audio_vals = self._topk(audio_logits)
        torch.save(
            {"text_topk_ids": text_ids, "text_topk_vals": text_vals,
             "audio_topk_ids": audio_ids, "audio_topk_vals": audio_vals},
            self.cache_dir / f"{sample_id}.pt",
        )

    def read(self, sample_id: str) -> tp.Optional[dict[str, torch.Tensor]]:
        path = self.cache_dir / f"{sample_id}.pt"
        if not path.exists():
            return None
        return torch.load(path, map_location="cpu")

    @staticmethod
    def to_dense(ids: torch.Tensor, vals: torch.Tensor, card: int, fill_value: float = -1e4) -> torch.Tensor:
        """Reconstructs a dense (very negative outside top-k) logits tensor from a
        sparse top-k cache, so it can be dropped into `losses.kl_distillation_loss`
        as if it were the real teacher logits."""
        shape = ids.shape[:-1] + (card,)
        dense = torch.full(shape, fill_value, dtype=vals.dtype)
        dense.scatter_(-1, ids.long(), vals)
        return dense


In [ ]:
%%writefile {REPO_DIR}/distill/gqa_attention.py
# SPDX-License-Identifier: MIT
"""GQA (grouped-query attention) streaming transformer for the distilled student.

The teacher (`moshi.modules.transformer.StreamingTransformer`) only implements
plain multi-head attention: one shared head count for Q, K and V, with a single
fused `in_proj_weight`. There is no GQA anywhere in the teacher codebase.

This module is a parallel implementation, used only by the student's temporal
transformer, that supports `num_kv_heads < num_heads`. The teacher's
`StreamingTransformer` and `StreamingMultiheadAttention` are not modified or
imported into the forward path here (only a few stateless helpers are reused).
It mirrors the teacher's streaming/KV-cache/RoPE/CUDA-graph-ability contract so
it can be dropped into `LMGen` exactly like the teacher's transformer.
"""

from contextlib import ExitStack
from dataclasses import dataclass
import math
import typing as tp

import torch
import torch.nn as nn
import torch.utils.checkpoint
from torch.nn import functional as F

from moshi.modules.streaming import StreamingModule
from moshi.modules.transformer import (
    KVCacheResult,
    RingKVCache,
    LayerScale,
    create_norm_fn,
)
from moshi.utils.compile import no_compile
from moshi.modules.gating import make_gating


def _apply_rope_single(x: torch.Tensor, offset: torch.Tensor, max_period: float) -> torch.Tensor:
    """RoPE for a single tensor `x`: [B, H, T, D] (always `time_before_heads=False`
    layout here). A deliberate reimplementation of the rotation math in
    `moshi.modules.rope.apply_rope`, NOT a call into it -- that function is
    decorated `@torch_compile_lazy`, giving it one process-wide compiled-function
    cache shared with the teacher's `StreamingMultiheadAttention`. Two problems
    with routing through it here: (1) it takes a `(q, k)` pair, and this module's
    Q and K have different head counts (GQA), so calling it as `apply_rope(q, q,
    ...)` / `apply_rope(k, k, ...)` aliases its two arguments -- a pattern the
    teacher's code never produces, since its own q and k are always distinct
    tensors; (2) the student's shapes differ from the teacher's, so a shape-
    triggered recompile of that SHARED compiled function can occur on a call
    made from inside `LMGen`'s `CUDAGraphed` capture window (`torch.cuda.graph()`
    requires no new CUDA work -- allocation, compilation, kernel autotuning --
    during capture). Both produced a real `torch.cuda.graphs.py` capture failure
    ("operation failed due to a previous error") on an actual GPU. Plain, eager,
    uncompiled ops here sidestep all of that; RoPE is cheap next to the rest of
    attention, so there's no real performance cost to not compiling it.
    """
    B, H, T, D = x.shape
    assert D % 2 == 0
    ds = torch.arange(D // 2, device=x.device, dtype=torch.float32)
    freqs = torch.exp(ds * (-math.log(max_period) * 2 / D))
    ts = (offset.float() + torch.arange(T, device=x.device, dtype=torch.float32)).view(1, 1, -1, 1)
    xr = x[..., 0::2].float()
    xi = x[..., 1::2].float()
    rotr = torch.cos(freqs * ts)
    roti = torch.sin(freqs * ts)
    xor = xr * rotr - xi * roti
    xoi = xr * roti + xi * rotr
    return torch.stack([xor, xoi], dim=-1).reshape(B, H, T, D).to(x.dtype)


@dataclass
class _GQAMHAState:
    kv_cache: RingKVCache
    offset: torch.Tensor
    offset_cpu: int

    def reset(self):
        self.kv_cache.reset()
        self.offset.zero_()
        self.offset_cpu = 0


class GQAStreamingMultiheadAttention(StreamingModule[_GQAMHAState]):
    """Causal streaming attention with `num_kv_heads <= num_heads` (GQA/MQA).

    Unlike the teacher's `StreamingMultiheadAttention`, this uses separate
    `q_proj` / `k_proj` / `v_proj` linears (instead of one fused `in_proj_weight`)
    since Q and KV live at different widths. `num_heads` must be divisible by
    `num_kv_heads`.

    Args:
        embed_dim (int): model dimension (d_model).
        num_heads (int): number of query heads.
        num_kv_heads (int): number of key/value heads. `num_heads % num_kv_heads == 0`.
        causal (bool): causal mask applied automatically.
        context (int, optional): number of past time steps visible to attention.
        max_period (float, optional): RoPE base period; None disables RoPE.
        device, dtype: passed to factory kwargs.
    """

    _fsdp_final = True

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        num_kv_heads: int,
        causal: bool = True,
        context: tp.Optional[int] = None,
        max_period: tp.Optional[float] = None,
        device=None,
        dtype=None,
    ):
        super().__init__()
        assert embed_dim % num_heads == 0, (embed_dim, num_heads)
        assert num_heads % num_kv_heads == 0, (num_heads, num_kv_heads)
        factory_kwargs = {"device": device, "dtype": dtype}

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = embed_dim // num_heads
        self.num_groups = num_heads // num_kv_heads
        self.causal = causal
        self.context = context
        self.max_period = max_period

        kv_dim = num_kv_heads * self.head_dim
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False, **factory_kwargs)
        self.k_proj = nn.Linear(embed_dim, kv_dim, bias=False, **factory_kwargs)
        self.v_proj = nn.Linear(embed_dim, kv_dim, bias=False, **factory_kwargs)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False, **factory_kwargs)

    def _init_streaming_state(self, batch_size: int) -> _GQAMHAState:
        if self.context is None:
            raise RuntimeError(
                "Cannot create a streaming KVCache without a context to estimate capacity."
            )
        device = self.q_proj.weight.device
        dtype = self.k_proj.weight.dtype
        kv_cache = RingKVCache(
            batch_size, self.num_kv_heads, self.head_dim, self.context, device, dtype
        )
        return _GQAMHAState(
            kv_cache,
            offset=torch.zeros(1, device=device, dtype=torch.long),
            offset_cpu=0,
        )

    def _complete_kv(self, k, v) -> KVCacheResult:
        state = self._streaming_state
        if state is None:
            return KVCacheResult.from_kv(k, v)
        else:
            return state.kv_cache.complete(k, v)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor):
        assert query is key and key is value, "self-attention only"
        state = self._streaming_state
        B, T, _ = query.shape

        if state is None:
            offset = torch.zeros(1, device=query.device, dtype=torch.long)
        else:
            assert self.causal, "Streaming only available for causal"
            offset = state.offset

        q = self.q_proj(query).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(key).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(value).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        if self.max_period is not None:
            # See _apply_rope_single's docstring: deliberately not moshi.modules.rope.apply_rope
            # (shared, torch.compile-cached, and shaped for a paired q/k of equal head count).
            q = _apply_rope_single(q, offset, self.max_period)
            k = _apply_rope_single(k, offset, self.max_period)

        k, v, pos_k = self._complete_kv(k, v)
        k = k.repeat_interleave(self.num_groups, dim=1)
        v = v.repeat_interleave(self.num_groups, dim=1)

        if self.causal:
            pos_k = pos_k.view(1, -1)
            pos_q = offset + torch.arange(T, device=q.device, dtype=torch.long).view(-1, 1)
            delta = pos_q - pos_k
            attn_bias = (pos_k >= 0) & (delta >= 0)
            if self.context is not None:
                attn_bias = attn_bias & (delta < self.context)
        else:
            attn_bias = None

        x = F.scaled_dot_product_attention(q, k, v, attn_bias, dropout_p=0.0)
        x = x.transpose(1, 2).reshape(B, T, self.embed_dim)
        x = self.out_proj(x)

        if state is not None:
            state.offset.add_(T)
            state.offset_cpu += T
        return x


@dataclass
class _GQALayerState:
    offset_cpu: int

    def reset(self):
        self.offset_cpu = 0


class GQAStreamingTransformerLayer(StreamingModule[_GQALayerState]):
    """Pre-norm transformer layer with GQA attention and a gated FFN.

    Structurally mirrors `moshi.modules.transformer.StreamingTransformerLayer`
    (same norm placement, same gated-FFN helper), swapping in
    `GQAStreamingMultiheadAttention` for the teacher's plain MHA.
    """

    _fsdp_final = True

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        num_kv_heads: int,
        dim_feedforward: int,
        causal: bool = True,
        context: tp.Optional[int] = None,
        max_period: tp.Optional[float] = None,
        norm: str = "rms_norm_f32",
        layer_scale: tp.Optional[float] = None,
        gating: str = "silu",
        device=None,
        dtype=None,
    ):
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        self.self_attn = GQAStreamingMultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            num_kv_heads=num_kv_heads,
            causal=causal,
            context=context,
            max_period=max_period,
            **factory_kwargs,
        )
        self.norm1 = create_norm_fn(norm, d_model, **factory_kwargs)
        self.norm2 = create_norm_fn(norm, d_model, **factory_kwargs)
        self.gating = make_gating(gating, d_model, dim_feedforward, **factory_kwargs)

        if layer_scale is None:
            self.layer_scale_1: nn.Module = nn.Identity()
            self.layer_scale_2: nn.Module = nn.Identity()
        else:
            self.layer_scale_1 = LayerScale(d_model, layer_scale, **factory_kwargs)
            self.layer_scale_2 = LayerScale(d_model, layer_scale, **factory_kwargs)

    def _init_streaming_state(self, batch_size: int) -> _GQALayerState:
        return _GQALayerState(offset_cpu=0)

    def _sa_block(self, x: torch.Tensor) -> torch.Tensor:
        x_orig = x
        x = self.norm1(x)
        update = self.self_attn(x, x, x)
        return x_orig + self.layer_scale_1(update)

    def _ff_block(self, x: torch.Tensor) -> torch.Tensor:
        x_orig = x
        x = self.norm2(x)
        update = self.gating(x)
        return x_orig + self.layer_scale_2(update)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with ExitStack() as stack:
            if x.device.type != "cuda":
                stack.enter_context(no_compile())
            x = self._sa_block(x)
            x = self._ff_block(x)
            state = self._streaming_state
            if state:
                state.offset_cpu += x.shape[1]
            return x


@dataclass
class _GQATransformerState:
    offset: torch.Tensor

    def reset(self):
        self.offset.zero_()


class GQAStreamingTransformer(StreamingModule[_GQATransformerState]):
    """Stack of `GQAStreamingTransformerLayer`, with RoPE shared across layers.

    Drop-in replacement for `moshi.modules.transformer.StreamingTransformer`
    from the point of view of `LMModel.forward_codes` / `LMGen`: same
    `forward(x) -> x` contract, same `.streaming()` / `.reset_streaming()` API.
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        num_kv_heads: int,
        num_layers: int,
        dim_feedforward: int,
        causal: bool = True,
        context: tp.Optional[int] = None,
        max_period: float = 10_000,
        positional_scale: float = 1.0,
        norm: str = "rms_norm_f32",
        layer_scale: tp.Optional[float] = None,
        gating: str = "silu",
        device=None,
        dtype=None,
    ):
        super().__init__()
        self.max_period = max_period
        self.positional_scale = positional_scale

        self.layers = nn.ModuleList(
            [
                GQAStreamingTransformerLayer(
                    d_model=d_model,
                    num_heads=num_heads,
                    num_kv_heads=num_kv_heads,
                    dim_feedforward=dim_feedforward,
                    causal=causal,
                    context=context,
                    max_period=max_period,
                    norm=norm,
                    layer_scale=layer_scale,
                    gating=gating,
                    device=device,
                    dtype=dtype,
                )
                for _ in range(num_layers)
            ]
        )

    def _init_streaming_state(self, batch_size: int) -> _GQATransformerState:
        device = next(self.parameters()).device
        return _GQATransformerState(offset=torch.zeros(1, device=device, dtype=torch.long))

    # Set by distill/train.py (--grad-checkpointing). Only used for full-sequence training forwards: never while
    # streaming (LMGen / inference), never in eval mode.
    gradient_checkpointing: bool = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        use_ckpt = self.gradient_checkpointing and self.training and self._streaming_state is None \
            and torch.is_grad_enabled()
        for layer in self.layers:
            if use_ckpt:
                # non-reentrant: forward hooks (train.py captures per-layer outputs for L_hidden) and autocast
                # behave exactly as without checkpointing; activations are recomputed in backward.
                x = torch.utils.checkpoint.checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        state = self._streaming_state
        if state is not None:
            state.offset.add_(x.shape[1])
        return x


In [ ]:
%%writefile {REPO_DIR}/distill/student_model.py
# SPDX-License-Identifier: MIT
"""The distilled student LM: small GQA temporal transformer + Bridge, with the
depth transformer, output heads, embeddings-for-the-frozen-path and Mimi all
reused verbatim (frozen) from the teacher checkpoint.

`StudentLMModel` subclasses `moshi.models.lm.LMModel` so that `LMGen` (the
streaming generation driver used by both `server.py` and `offline.py`) can
drive it exactly like the teacher: `forward_codes`, `forward_depformer`,
`forward_depformer_training`, `.depformer`, `.dep_q`, `.delays`, etc. are all
inherited unchanged. Only `forward_embeddings` is overridden, to route the
student's transformer output through the Bridge before it reaches the frozen
`out_norm` / `text_linear` / depformer chain.
"""

from pathlib import Path
import typing as tp

import torch
from safetensors.torch import load_file

from moshi.models.lm import LMModel, ScaledEmbedding
from moshi.models.loaders import _lm_kwargs as _TEACHER_LM_KWARGS

from .config import StudentConfig, load_student_config
from .bridge import Bridge
from .gqa_attention import GQAStreamingTransformer

# Module-name prefixes that are frozen and reused verbatim from the teacher
# checkpoint. Everything else (transformer.*, emb.*, text_emb.*, bridge.*) is
# new and trainable.
FROZEN_PREFIXES = (
    "out_norm.",
    "text_linear.",
    "depformer_in.",
    "depformer_emb.",
    "depformer_text_emb.",
    "depformer.",
    "linears.",
)


def teacher_lm_kwargs() -> dict:
    """The exact kwargs `moshi.models.loaders.get_moshi_lm` uses to build the teacher,
    with the same `dep_q=16` patch applied. Used to build a teacher-shaped meta
    skeleton whose frozen submodules line up 1:1 with the teacher checkpoint."""
    kwargs = dict(_TEACHER_LM_KWARGS)
    kwargs["dep_q"] = 16
    return kwargs


class StudentLMModel(LMModel):
    def __init__(
        self,
        student_config: StudentConfig,
        device: tp.Union[str, torch.device] = "cpu",
        dtype: torch.dtype = torch.bfloat16,
        teacher_kwargs: tp.Optional[dict] = None,
    ):
        # Build a teacher-shaped skeleton on the meta device: near-zero cost, and it
        # guarantees the frozen submodules (depformer*, linears, text_linear, out_norm)
        # come out with exactly the shapes the teacher checkpoint expects.
        # `teacher_kwargs` defaults to the real 7B teacher's construction kwargs;
        # it's overridable so tests/test_init_sanity.py can exercise the full
        # init/wiring pipeline against a tiny synthetic teacher instead of
        # requiring a multi-GB download.
        super().__init__(device="meta", dtype=dtype, **(teacher_kwargs or teacher_lm_kwargs()))
        # `LMModel.__init__` builds `text_linear`, `out_norm`, `depformer_in`, and `linears`
        # via plain `torch.nn.Linear(...)` / `create_norm_fn(...)` calls with NO device/dtype
        # kwargs (see moshi/models/lm.py) -- they always land on default CPU/float32 regardless
        # of the `device="meta", dtype=dtype` passed above. `get_moshi_lm` (loaders.py) papers
        # over this with a blanket `model.to(device=device, dtype=dtype)` after construction;
        # without the same fix here these four submodule groups would be real (non-meta, wrong
        # dtype) tensors, defeating the "near-zero cost" meta build AND making
        # `load_frozen_from_teacher` below cast teacher weights to float32 (via
        # `self.text_linear.weight.dtype`) instead of the model's real working dtype.
        self.to(device="meta", dtype=dtype)

        cfg = student_config
        self.student_config = cfg

        # Replace the teacher-scale trainable path with the student-scale one.
        del self.transformer
        self.transformer = GQAStreamingTransformer(
            d_model=cfg.hidden_size,
            num_heads=cfg.num_attention_heads,
            num_kv_heads=cfg.num_key_value_heads,
            num_layers=cfg.num_hidden_layers,
            dim_feedforward=cfg.intermediate_size,
            causal=cfg.causal,
            context=cfg.max_frames,
            max_period=cfg.rope_theta,
            layer_scale=cfg.layer_scale,
            norm=cfg.norm,
            gating=cfg.gating,
            device=device,
            dtype=dtype,
        )

        del self.emb
        self.emb = torch.nn.ModuleList(
            [
                ScaledEmbedding(self.card + 1, cfg.hidden_size, zero_idx=self.zero_token_id,
                                 device=device, dtype=dtype)
                for _ in range(self.n_q)
            ]
        )
        del self.text_emb
        self.text_emb = ScaledEmbedding(
            self.text_card + 1, cfg.hidden_size, zero_idx=self.zero_token_id,
            device=device, dtype=dtype,
        )

        self.bridge = Bridge(cfg.bridge.in_dim, cfg.bridge.out_dim, device=device, dtype=dtype)
        self._student_device = torch.device(device)

    def load_frozen_from_teacher(self, teacher_state_dict: dict[str, torch.Tensor]):
        """Fill in the frozen submodules (depformer*, linears, text_linear, out_norm)
        from a teacher `LMModel` state dict, and freeze them. `transformer.*`,
        `emb.*`, `text_emb.*` keys in `teacher_state_dict` are ignored (unexpected) by
        design -- those modules were already replaced with student-scale ones above.
        """
        state_dict = {
            k: v.to(device=self._student_device, dtype=self.text_linear.weight.dtype)
            for k, v in teacher_state_dict.items()
            if k.startswith(FROZEN_PREFIXES)
        }
        result = self.load_state_dict(state_dict, strict=False, assign=True)
        missing_frozen = [k for k in result.missing_keys if k.startswith(FROZEN_PREFIXES)]
        assert not missing_frozen, f"Teacher checkpoint is missing frozen keys: {missing_frozen}"
        assert not result.unexpected_keys, (
            f"Unexpected keys after filtering to frozen prefixes: {result.unexpected_keys}"
        )
        for meta_name, module in self.named_modules():
            for p in module.parameters(recurse=False):
                assert p.device.type != "meta", (
                    f"{meta_name} still has meta-device parameters after loading frozen "
                    "teacher weights -- this module was expected to be either replaced "
                    "with a student-scale module or filled in from the teacher checkpoint."
                )
        self.freeze_teacher_components()

    def freeze_teacher_components(self):
        for name, p in self.named_parameters():
            if name.startswith(FROZEN_PREFIXES):
                p.requires_grad = False

    def unfreeze_depth_transformer(self):
        """Phase 4 ("Polish") only: unfreeze the depth transformer at 0.1x LR."""
        for p in self.depformer.parameters():
            p.requires_grad = True

    def trainable_parameters(self) -> tp.Iterator[torch.nn.Parameter]:
        return (p for p in self.parameters() if p.requires_grad)

    def forward_embeddings(self, input: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        transformer_out = self.transformer(input)
        transformer_out = self.bridge(transformer_out)
        if self.out_norm:
            transformer_out = self.out_norm(transformer_out)
        text_logits = self.text_linear(transformer_out)
        text_logits = text_logits[:, None]
        return transformer_out, text_logits

    def forward_depformer_training(
        self,
        sequence: torch.Tensor,
        transformer_out: torch.Tensor,
    ) -> torch.Tensor:
        B, K, T = sequence.shape
        Ka = self.dep_q
        assert (
            K == self.num_codebooks
        ), f"Codebooks for Depformer training should be passed all at once, got {K,}."
        depformer_inputs = []
        for cb_index in range(Ka):
            if self.depformer_multi_linear:
                linear_index = cb_index
                if self.depformer_weights_per_step_schedule is not None:
                    linear_index = self.depformer_weights_per_step_schedule[cb_index]
                transformer_in = self.depformer_in[linear_index](transformer_out)
            else:
                transformer_in = self.depformer_in[0](transformer_out)
            if cb_index == 0:
                token_in = self.depformer_text_emb(sequence[:, 0])
            else:
                token_in = self.depformer_emb[cb_index - 1](sequence[:, cb_index + self.audio_offset - 1])
            depformer_inputs.append(token_in + transformer_in)
        depformer_input = torch.stack(depformer_inputs, 2)
        # depformer_input is [B, T, K, depformer_dim], reshaping to [B * T, K, D]
        depformer_input = depformer_input.view(B * T, Ka, -1)
        if getattr(self.transformer, "gradient_checkpointing", False) and self.training and torch.is_grad_enabled():
            depformer_output = torch.utils.checkpoint.checkpoint(
                self.depformer, depformer_input, use_reentrant=False
            )
        else:
            depformer_output = self.depformer(depformer_input)
        all_logits = []
        for cb_index in range(Ka):
            logits = self.linears[cb_index](depformer_output[:, cb_index])
            all_logits.append(logits.view(B, T, -1))
        logits = torch.stack(all_logits, 1)
        assert logits.dim() == 4, logits.shape  # [B, Ka, T, card]
        return logits

    def raw_temporal_output(self, sequence: torch.Tensor) -> torch.Tensor:
        """Student transformer output before the Bridge (used by L_hidden, which
        distills at the temporal-transformer level, not the bridged/teacher level)."""
        return self.transformer(self.embed_codes(sequence))

    def bridged_conditioning(self, sequence: torch.Tensor) -> torch.Tensor:
        """Bridge output (pre teacher out_norm) -- used as the L_bridge prediction,
        compared against the teacher's raw (pre out_norm) transformer output."""
        return self.bridge(self.raw_temporal_output(sequence))


def build_student_lm(
    student_config: tp.Union[str, StudentConfig],
    teacher_checkpoint: tp.Union[str, Path],
    device: tp.Union[str, torch.device] = "cpu",
    dtype: torch.dtype = torch.bfloat16,
    teacher_kwargs: tp.Optional[dict] = None,
) -> StudentLMModel:
    """Build a `StudentLMModel` with freshly-initialized (untrained) student-scale
    modules and the frozen teacher submodules loaded in. This is the *structural*
    build only -- see `distill/init_from_teacher.py` for the actual data-driven
    initialization (layer selection, SVD width reduction, GQA KV pooling, bridge
    least-squares fit) that should be run before this model produces anything
    resembling coherent speech.
    """
    if isinstance(student_config, str):
        student_config = load_student_config(student_config)

    # teacher_kwargs: only for tests with a tiny synthetic teacher; None = the real PersonaPlex-7B layout.
    model = StudentLMModel(student_config, device=device, dtype=dtype, teacher_kwargs=teacher_kwargs)

    teacher_checkpoint = str(teacher_checkpoint)
    if teacher_checkpoint.endswith(".safetensors"):
        teacher_state_dict = load_file(teacher_checkpoint, device="cpu")
    else:
        teacher_state_dict = torch.load(teacher_checkpoint, map_location="cpu")

    model.load_frozen_from_teacher(teacher_state_dict)
    model.eval()
    return model


In [ ]:
%%writefile {REPO_DIR}/distill/data/text_dataset.py
# SPDX-License-Identifier: MIT
"""Plain-text token stream for TEXT-ONLY distillation (`distill/train.py --text-data ...`).

Why this exists
---------------
The student's temporal transformer is the only part of PersonaPlex that holds world knowledge (the frozen
depformer / `linears` / `text_linear` / Mimi are the acoustic rendering stack), and the distillation init
reduces it to a fraction of the teacher's width, depth, heads and FFN channels. Recovering the text behaviour
it lost needs text tokens -- a lot of them.

An audio conversation is a terrible carrier for those tokens. At 12.5 Hz, a sample of 1000 frames costs 1000
transformer positions *plus* a 16-codebook depformer forward at every position, and carries only ~100 real
text tokens (the other ~90% of the text stream is PAD/EPAD and the forced persona prompt). A 100 h corpus of
such conversations contains well under 1M agent text tokens in total.

`moshi.models.lm.ScaledEmbedding` makes a much cheaper path available: it returns EXACTLY zero for
`zero_idx == LMModel.zero_token_id == -1`. So filling the 16 audio rows of a `[B, 17, T]` code tensor with -1
turns `embed_codes` into a pure text embedding lookup, and `forward_codes` into a plain text-LM forward
through the same `transformer` -> `bridge` -> frozen `out_norm` -> frozen `text_linear` chain -- one position
per text token, no Mimi, no depformer, no 17-stream delay machinery.

This module turns text files into the flat token array that mode consumes.

What text to use
----------------
Because the objective is KL against the teacher, the corpus only has to COVER the domain -- it does not need
answers in it. The teacher supplies the targets. So: general prose for broad competence, plus text about
whatever the student must be able to discuss (for this project: cryptocurrency / blockchain / finance, and
whatever identity facts the personas assert).

Layout on disk
--------------
`build_token_cache` concatenates every input document into one `uint16` .npy array (`text_card = 32000` fits
in uint16; so does `text_initial_token_id = 32000`), with `SEPARATOR` between documents. `TextTokenStream`
then serves fixed-length windows from it, addressed by global step so a resumed run reads the same data in
the same order -- the same contract `GlobalBatchSampler` provides for the audio split.
"""

from pathlib import Path
import json
import logging
import typing as tp

import numpy as np

logger = logging.getLogger(__name__)

# `existing_text_padding_id = 3` (moshi/models/lm.py) doubles as the between-documents separator: it is the
# token the model already reads as "nothing being said", so it is a natural document break and it never
# collides with a real SentencePiece piece.
SEPARATOR = 3
TEXT_CARD = 32000

TEXT_SUFFIXES = (".txt", ".md", ".jsonl", ".json")


def _iter_documents(paths: tp.Sequence[str]) -> tp.Iterator[str]:
    """Yield document strings from files/directories/globs.

    `.txt` / `.md` -> one document per file. `.jsonl` -> one document per line, read from the first present
    key among `text`, `content`, `answer`, `output`, `completion` (so Dolly/OASST-style dumps work unchanged);
    a line that is a bare JSON string is taken as-is. `.json` -> a list of such objects or strings.
    """
    files: list[Path] = []
    for p in paths:
        path = Path(p)
        if any(ch in p for ch in "*?["):
            files += sorted(Path().glob(p))
        elif path.is_dir():
            files += sorted(f for f in path.rglob("*") if f.suffix.lower() in TEXT_SUFFIXES)
        elif path.is_file():
            files.append(path)
        else:
            raise FileNotFoundError(f"--text-data entry matches nothing: {p}")
    if not files:
        raise FileNotFoundError(f"--text-data matched no {'/'.join(TEXT_SUFFIXES)} files: {list(paths)}")

    def from_obj(obj) -> tp.Optional[str]:
        if isinstance(obj, str):
            return obj
        if isinstance(obj, dict):
            for k in ("text", "content", "answer", "output", "completion", "response"):
                if isinstance(obj.get(k), str):
                    return obj[k]
        return None

    for f in files:
        suffix = f.suffix.lower()
        if suffix in (".txt", ".md"):
            yield f.read_text(encoding="utf-8", errors="replace")
        elif suffix == ".jsonl":
            with open(f, encoding="utf-8", errors="replace") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        doc = from_obj(json.loads(line))
                    except json.JSONDecodeError:
                        continue
                    if doc:
                        yield doc
        else:  # .json
            try:
                data = json.loads(f.read_text(encoding="utf-8", errors="replace"))
            except json.JSONDecodeError:
                continue
            for obj in (data if isinstance(data, list) else [data]):
                doc = from_obj(obj)
                if doc:
                    yield doc


def build_token_cache(
    text_paths: tp.Sequence[str],
    tokenizer_model: str,
    cache_path: str,
    max_tokens: tp.Optional[int] = None,
) -> np.ndarray:
    """Tokenize `text_paths` with the PersonaPlex SentencePiece model into one flat uint16 .npy array.

    Cached: an existing `cache_path` is reused as-is (tokenizing a large corpus is slow, and every rank would
    otherwise redo it). Delete the file to force a rebuild.
    """
    cache = Path(cache_path)
    if cache.exists():
        arr = np.load(cache, mmap_mode="r")
        logger.info("text cache: reusing %s (%d tokens)", cache, arr.shape[0])
        return arr

    import sentencepiece

    sp = sentencepiece.SentencePieceProcessor(tokenizer_model)
    chunks: list[np.ndarray] = []
    total = 0
    n_docs = 0
    for doc in _iter_documents(text_paths):
        doc = doc.strip()
        if not doc:
            continue
        ids = sp.encode(doc)
        if not ids:
            continue
        assert max(ids) < TEXT_CARD, f"token id {max(ids)} >= text_card {TEXT_CARD} -- wrong tokenizer?"
        chunks.append(np.asarray(ids + [SEPARATOR], dtype=np.uint16))
        total += len(ids) + 1
        n_docs += 1
        if max_tokens is not None and total >= max_tokens:
            break
    if not chunks:
        raise ValueError(f"No text was tokenized from {list(text_paths)}")
    arr = np.concatenate(chunks)
    if max_tokens is not None:
        arr = arr[:max_tokens]
    cache.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_suffix(".tmp.npy")
    np.save(tmp, arr)
    tmp.replace(cache)
    logger.info("text cache: wrote %s (%d documents, %d tokens)", cache, n_docs, arr.shape[0])
    return np.load(cache, mmap_mode="r")


class TextTokenStream:
    """Fixed-length text windows addressed by global step, so resuming reproduces the data order exactly.

    `batch(step, rank, world, micro, accum)` returns the windows this rank must process at `step`: an
    `int64` array of shape `[micro * accum, seq_len + 1]`. The extra +1 column is the next-token target for
    the last position, so a caller can use `w[:, :-1]` as input and `w[:, 1:]` as targets without losing a
    position at the window boundary.
    """

    def __init__(self, tokens: np.ndarray, seq_len: int, seed: int = 1234):
        self.tokens = tokens
        self.seq_len = int(seq_len)
        self.seed = int(seed)
        self.n_windows = max(1, (tokens.shape[0] - 1) // self.seq_len)
        if tokens.shape[0] < self.seq_len + 1:
            raise ValueError(
                f"text corpus has {tokens.shape[0]} tokens, needs at least --text-seq-len + 1 = "
                f"{self.seq_len + 1}. Point --text-data at more text.")
        self._perm_cache: dict[int, np.ndarray] = {}

    def __len__(self) -> int:
        return self.n_windows

    def _perm(self, epoch: int) -> np.ndarray:
        if epoch not in self._perm_cache:
            if len(self._perm_cache) > 3:
                self._perm_cache.pop(min(self._perm_cache))
            self._perm_cache[epoch] = np.random.default_rng([self.seed, 777, epoch]).permutation(self.n_windows)
        return self._perm_cache[epoch]

    def _window(self, pos: int) -> np.ndarray:
        epoch, off = divmod(pos, self.n_windows)
        w = int(self._perm(epoch)[off])
        start = w * self.seq_len
        return np.asarray(self.tokens[start:start + self.seq_len + 1], dtype=np.int64)

    def batch(self, step: int, rank: int, world: int, micro: int, accum: int) -> np.ndarray:
        per_step = world * micro * accum
        base = step * per_step + rank * micro * accum
        return np.stack([self._window(base + i) for i in range(micro * accum)])


In [ ]:
%%writefile {REPO_DIR}/distill/schedule.py
# SPDX-License-Identifier: MIT
"""The P1->P4 training schedule: loss weights, data source (teacher-forced vs.
on-policy student rollouts), depth-transformer freeze state, and learning-rate
multiplier, all as a function of training progress in [0, 1].
"""

from dataclasses import dataclass
import typing as tp


@dataclass(frozen=True)
class PhaseSpec:
    name: str
    start: float  # inclusive, fraction of total training
    end: float  # exclusive (1.0 for the last phase)
    alpha: float  # CE weight
    beta: float  # KL weight
    gamma: float  # L_bridge weight
    delta: float  # L_hidden weight
    epsilon: float  # L_speaker weight
    on_policy: bool  # student rollouts vs. teacher-forced data
    depth_transformer_unfrozen: bool
    depth_transformer_lr_mult: float = 0.0


PHASES: tuple[PhaseSpec, ...] = (
    PhaseSpec("P1_align", 0.00, 0.25, alpha=0.0, beta=0.3, gamma=2.0, delta=1.0, epsilon=0.0,
              on_policy=False, depth_transformer_unfrozen=False),
    PhaseSpec("P2_behavior", 0.25, 0.85, alpha=0.3, beta=1.0, gamma=0.5, delta=0.3, epsilon=0.0,
              on_policy=False, depth_transformer_unfrozen=False),
    PhaseSpec("P3_on_policy", 0.85, 0.95, alpha=0.3, beta=1.0, gamma=0.2, delta=0.05, epsilon=0.2,
              on_policy=True, depth_transformer_unfrozen=False),
    PhaseSpec("P4_polish", 0.95, 1.00, alpha=0.5, beta=0.6, gamma=0.1, delta=0.0, epsilon=0.5,
              on_policy=True, depth_transformer_unfrozen=True, depth_transformer_lr_mult=0.1),
)

# Why these boundaries moved (was 0.15 / 0.60 / 0.90 / 1.00):
#   * P1 "Align" was 15% of the run -- 139 steps on the 925-step run -- and with alpha=0 it applied no
#     hard-label CE at all. The student enters P2 still far from the teacher. P1 is now 25%.
#   * P3/P4 (on-policy) were 40% of the STEPS, but an on-policy step is ~7x slower and processes ~half the
#     frames, so they consumed ~half the wall clock for ~11% of the frames (measured: 88.1 s/step and
#     188 frames/s on-policy vs 20.6 s/step and 1600 frames/s teacher-forced). On-policy rollouts fix speech
#     BEHAVIOUR -- turn-taking, not running out of things to say -- which is the one thing the student already
#     did well. They are now 15% of the steps, so the budget goes to the objective that was failing.
#   * alpha in P2 went 0.2 -> 0.3: with the text content weighting in distill/train.py the hard-label CE is
#     now mostly about words rather than padding, so it is worth more.
#   * beta now weights the AUDIO KL only; the text KL has its own weight (`--beta-text`).


def phase_at(progress: float) -> PhaseSpec:
    """`progress`: fraction of training complete, in [0, 1]."""
    assert 0.0 <= progress <= 1.0, progress
    for phase in PHASES:
        if phase.start <= progress < phase.end:
            return phase
    return PHASES[-1]  # progress == 1.0 falls into the last phase


class TrainingSchedule:
    """Stateful wrapper: given the current step and total steps, exposes the
    active phase and whether a phase transition just happened (so the caller
    can e.g. unfreeze the depth transformer and reset its optimizer state
    exactly once, at the P3->P4 boundary).
    """

    def __init__(self, total_steps: int):
        assert total_steps > 0
        self.total_steps = total_steps
        self._last_phase_name: tp.Optional[str] = None

    def progress(self, step: int) -> float:
        return min(1.0, max(0.0, step / self.total_steps))

    def step(self, step: int) -> tuple[PhaseSpec, bool]:
        """Returns (phase, phase_just_changed)."""
        phase = phase_at(self.progress(step))
        changed = phase.name != self._last_phase_name
        self._last_phase_name = phase.name
        return phase, changed

    def state_dict(self) -> dict:
        return {"total_steps": self.total_steps, "last_phase_name": self._last_phase_name}

    def load_state_dict(self, state: dict):
        self.total_steps = state["total_steps"]
        self._last_phase_name = state["last_phase_name"]


In [ ]:
%%writefile {REPO_DIR}/distill/losses.py
# SPDX-License-Identifier: MIT
"""Loss terms for temporal-transformer distillation.

    L = alpha*CE + beta*KL + gamma*L_bridge + delta*L_hidden + epsilon*L_speaker

Per-codebook weights and temperatures, transition upweighting, and running-mean
normalization are all implemented here; `distill/schedule.py` owns how
alpha..epsilon and the depth-transformer freeze state vary over training phases.

Codebook indexing convention (matches `LMModel`: index 0 is text, 1..dep_q are
audio codebooks in delay order): `CODEBOOK_KL_WEIGHTS[0]` is the text weight,
`CODEBOOK_KL_WEIGHTS[1:]` the first 8 audio codebooks (cb1..cb8). The teacher's
`dep_q=16` audio codebooks are two interleaved 8-codebook streams (self/other
party, see moshi/models/lm.py); the same 8 weights are applied to both halves
since they represent the same underlying acoustic hierarchy for each party.
"""

from dataclasses import dataclass, field
import typing as tp

import torch
import torch.nn as nn
import torch.nn.functional as F

TEXT_KL_TEMPERATURE = 2.0
DEFAULT_AUDIO_KL_TEMPERATURE = 1.0  # configurable: audio distributions are already high-entropy

# [text, cb1, cb2, cb3, cb4, cb5, cb6, cb7, cb8]
CODEBOOK_KL_WEIGHTS = [1.0, 1.0, 0.8, 0.8, 0.7, 0.5, 0.4, 0.3, 0.25]


def _codebook_weights(num_audio_codebooks: int, device, dtype) -> torch.Tensor:
    """Expand the 8 acoustic-hierarchy weights to however many audio codebooks the
    depformer actually has (16, two 8-codebook streams), by tiling."""
    base = torch.tensor(CODEBOOK_KL_WEIGHTS[1:], device=device, dtype=dtype)
    reps = (num_audio_codebooks + len(base) - 1) // len(base)
    return base.repeat(reps)[:num_audio_codebooks]


def kl_distillation_loss(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    mask: torch.Tensor,
    temperature: float,
    weight: tp.Union[float, torch.Tensor] = 1.0,
) -> torch.Tensor:
    """KL(teacher || student) at the given temperature, with the standard tau^2
    gradient-scale correction, reduced over the vocabulary and averaged over the
    (masked) batch/time/codebook positions. `weight` can be a per-codebook tensor
    broadcastable against `student_logits`'s codebook dimension.

    Shapes: student_logits/teacher_logits [B, K, T, card] (or [B, 1, T, card] for
    text), mask [B, K, T] (True where the position is valid).

    `_undelay_sequence` (moshi/models/lm.py) fills positions past the sequence's
    delay boundary with NaN and marks them invalid in `mask`; those NaNs are
    scrubbed here so they can't survive a `nan * 0.0 == nan` accident in the
    masked reduction below.
    """
    student_logits = torch.nan_to_num(student_logits, nan=0.0)
    teacher_logits = torch.nan_to_num(teacher_logits, nan=0.0)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)
    kl = F.kl_div(student_log_probs, teacher_probs, reduction="none").sum(dim=-1)
    kl = kl * (temperature ** 2)
    kl = kl * weight
    mask = mask.to(kl.dtype)
    denom = mask.sum().clamp_min(1.0)
    return (kl * mask).sum() / denom


def ce_loss(student_logits: torch.Tensor, targets: torch.Tensor, ignore_index: int) -> torch.Tensor:
    """Hard-label cross entropy, `student_logits` [B, K, T, card], `targets` [B, K, T]."""
    logits_flat = student_logits.reshape(-1, student_logits.shape[-1])
    targets_flat = targets.reshape(-1)
    return F.cross_entropy(logits_flat.float(), targets_flat, ignore_index=ignore_index)


@dataclass
class DistillationLossOutput:
    total: torch.Tensor
    ce: torch.Tensor
    kl_text: torch.Tensor
    kl_audio: torch.Tensor
    bridge: torch.Tensor
    hidden: torch.Tensor
    speaker: torch.Tensor
    raw: dict[str, float] = field(default_factory=dict)


class RunningNorm:
    """Normalizes a scalar loss term by a running mean of its own magnitude, so
    the phase weights (alpha..epsilon) control relative emphasis rather than
    fighting each term's raw scale. `momentum` close to 1 => slow-moving average.
    """

    def __init__(self, momentum: float = 0.99, eps: float = 1e-6):
        self.momentum = momentum
        self.eps = eps
        self._mean: tp.Optional[float] = None

    def __call__(self, value: torch.Tensor) -> torch.Tensor:
        v = float(value.detach().abs().item())
        # A single non-finite `value` (e.g. from a degenerate init or a bad batch) must not
        # permanently corrupt `self._mean` to NaN: the EMA update `momentum*mean + (1-momentum)*nan`
        # is NaN forever after, silently normalizing every FUTURE (otherwise-healthy) call by NaN
        # too -- unlike a single bad optimizer step (which a finite-loss check can just skip), this
        # state has no way to self-correct once corrupted. Skip the update and keep the last-known-
        # good mean instead; the caller is still responsible for not stepping the optimizer on a
        # non-finite total loss (see distill/train.py's finite check).
        if v == v and v not in (float("inf"), float("-inf")):  # finite check without importing math
            if self._mean is None:
                self._mean = v
            else:
                self._mean = self.momentum * self._mean + (1 - self.momentum) * v
        denom = (self._mean if self._mean is not None else 1.0) + self.eps
        return value / denom

    def state_dict(self) -> dict:
        return {"mean": self._mean, "momentum": self.momentum}

    def load_state_dict(self, state: dict):
        self._mean = state["mean"]
        self.momentum = state["momentum"]


def bridge_loss(student_bridge_out: torch.Tensor, teacher_raw_hidden: torch.Tensor) -> torch.Tensor:
    """MSE + cosine between the student's Bridge output and the teacher's raw
    (pre out_norm) temporal-transformer hidden state -- both dense, positionally
    aligned, teacher_dim-wide vectors."""
    mse = F.mse_loss(student_bridge_out.float(), teacher_raw_hidden.float())
    cos = 1.0 - F.cosine_similarity(student_bridge_out.float(), teacher_raw_hidden.float(), dim=-1).mean()
    return mse + cos


class HiddenProjection(nn.Module):
    """Per-mapped-layer student->teacher projection for L_hidden. Orthogonal init,
    meant to be excluded from weight decay, and discarded at export (it never
    participates in inference, only in the training-time loss).
    """

    def __init__(self, student_dim: int, teacher_dim: int):
        super().__init__()
        self.proj = nn.Linear(student_dim, teacher_dim, bias=False)
        nn.init.orthogonal_(self.proj.weight)

    def forward(self, student_hidden: torch.Tensor) -> torch.Tensor:
        return self.proj(student_hidden)


def hidden_loss(
    student_hidden: torch.Tensor,
    teacher_hidden: torch.Tensor,
    projection: HiddenProjection,
) -> torch.Tensor:
    """L2-normalize both sides after projecting student->teacher space, then
    MSE + cosine. Distills attention-block *outputs* (the residual stream at a
    mapped layer boundary), not attention matrices.
    """
    # `projection` is kept in fp32 (see distill/train.py); cast the (bf16) model
    # hidden state up front rather than relying on autocast.
    projected = projection(student_hidden.float())
    projected = F.normalize(projected, dim=-1)
    teacher = F.normalize(teacher_hidden.float(), dim=-1)
    mse = F.mse_loss(projected, teacher)
    cos = 1.0 - F.cosine_similarity(projected, teacher, dim=-1).mean()
    return mse + cos


def dilate_mask(mask: torch.Tensor, radius: int) -> torch.Tensor:
    """1D max-pool dilation of a boolean/float mask along the time dimension.
    `mask`: [..., T]."""
    if radius <= 0:
        return mask
    m = mask.float()
    shape = m.shape
    m = m.reshape(-1, 1, shape[-1])
    kernel = 2 * radius + 1
    dilated = F.max_pool1d(m, kernel_size=kernel, stride=1, padding=radius)
    return dilated.reshape(shape)


def normalized_entropy(logits: torch.Tensor) -> torch.Tensor:
    """Entropy of softmax(logits) along the last dim, normalized to [0, 1] by
    the maximum possible entropy (log of the vocab size)."""
    log_probs = F.log_softmax(logits.float(), dim=-1)
    probs = log_probs.exp()
    entropy = -(probs * log_probs).sum(dim=-1)
    max_entropy = torch.log(torch.tensor(float(logits.shape[-1]), device=logits.device))
    return entropy / max_entropy


def frame_weights(
    speaker_transition_mask: torch.Tensor,
    teacher_cb1_logits: torch.Tensor,
    radius: int = 8,
    transition_boost: float = 3.0,
    entropy_boost: float = 0.5,
) -> torch.Tensor:
    """w_frame = (1 + transition_boost * dilate(speaker_transition_mask, radius))
                 * (1 + entropy_boost * normalized_teacher_entropy(cb1))
    `speaker_transition_mask`: [B, T] boolean/float, 1 at frames flagged as a
    speaker turn-taking transition. `teacher_cb1_logits`: [B, T, card] teacher
    logits for the first acoustic codebook (the one most tied to turn-taking).
    """
    dilated = dilate_mask(speaker_transition_mask.float(), radius)
    entropy = normalized_entropy(teacher_cb1_logits)
    return (1.0 + transition_boost * dilated) * (1.0 + entropy_boost * entropy)


class WavLMSpeakerSimilarity:
    """WavLM-TDNN speaker cosine similarity, teacher vs. student audio decoded
    through the SAME frozen Mimi. Implemented as a validation metric (phase 1-3):
    call `.similarity(teacher_wav, student_wav)` -> per-utterance cosine in
    [-1, 1]. The differentiable (straight-through Gumbel) variant belongs to
    phase 4 only, and is NOT implemented here -- phase 4 should subclass or wrap
    this with a straight-through sampler around the frozen Mimi/depformer chain
    before backpropagating through it.

    Lazily loads `microsoft/wavlm-base-plus-sv` (WavLM + TDNN/x-vector head) via
    `transformers` on first use, since it's a heavy optional dependency not
    needed outside of evaluation.
    """

    MODEL_NAME = "microsoft/wavlm-base-plus-sv"
    SAMPLE_RATE = 16000

    def __init__(self, device: tp.Union[str, torch.device] = "cpu"):
        self.device = torch.device(device)
        self._model = None
        self._feature_extractor = None

    def _load(self):
        if self._model is not None:
            return
        from transformers import WavLMForXVector, Wav2Vec2FeatureExtractor

        self._feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(self.MODEL_NAME)
        self._model = WavLMForXVector.from_pretrained(self.MODEL_NAME).to(self.device).eval()

    @torch.no_grad()
    def embed(self, wav: torch.Tensor, sample_rate: int) -> torch.Tensor:
        """`wav`: [B, T] or [T] mono float audio at `sample_rate`. Returns [B, D] embeddings."""
        self._load()
        if wav.dim() == 1:
            wav = wav.unsqueeze(0)
        if sample_rate != self.SAMPLE_RATE:
            import torchaudio

            wav = torchaudio.functional.resample(wav, sample_rate, self.SAMPLE_RATE)
        inputs = self._feature_extractor(
            list(wav.cpu().numpy()), sampling_rate=self.SAMPLE_RATE, return_tensors="pt", padding=True,
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        out = self._model(**inputs)
        return out.embeddings

    def similarity(self, teacher_wav: torch.Tensor, student_wav: torch.Tensor, sample_rate: int) -> torch.Tensor:
        teacher_emb = self.embed(teacher_wav, sample_rate)
        student_emb = self.embed(student_wav, sample_rate)
        return F.cosine_similarity(teacher_emb, student_emb, dim=-1)


In [ ]:
%%writefile {REPO_DIR}/distill/eval.py
# SPDX-License-Identifier: MIT
"""Evaluation metrics for a distilled student: WER, WavLM-TDNN speaker
similarity, UTMOS, barge-in latency, and RTF against the 80ms frame budget.

No numbers are hardcoded anywhere in this file -- every metric is actually
measured against whatever model/audio is passed in. Optional heavy
dependencies (`transformers` for Whisper/WavLM, `speechmos` for UTMOS) are
imported lazily so importing this module doesn't require them unless the
corresponding metric is actually used.
"""

from dataclasses import dataclass, asdict
import logging
import re
import time
import typing as tp

import numpy as np
import torch

from moshi.models.lm import LMGen
from moshi.models.compression import MimiModel

from .losses import WavLMSpeakerSimilarity

logger = logging.getLogger(__name__)

FRAME_BUDGET_MS = 80.0  # 12.5 Hz Mimi frame rate


@dataclass
class LatencyStats:
    p50_ms: float
    p95_ms: float
    p99_ms: float
    mean_ms: float

    @staticmethod
    def from_samples(samples_ms: list[float]) -> "LatencyStats":
        arr = np.array(samples_ms)
        return LatencyStats(
            p50_ms=float(np.percentile(arr, 50)),
            p95_ms=float(np.percentile(arr, 95)),
            p99_ms=float(np.percentile(arr, 99)),
            mean_ms=float(arr.mean()),
        )


class WhisperWER:
    """Word error rate via `openai/whisper-large-v3` (lazy-loaded)."""

    MODEL_NAME = "openai/whisper-large-v3"

    def __init__(self, device: str = "cuda"):
        self.device = device
        self._pipe = None

    def _load(self):
        if self._pipe is not None:
            return
        from transformers import pipeline

        self._pipe = pipeline("automatic-speech-recognition", model=self.MODEL_NAME, device=self.device)

    def transcribe(self, wav: np.ndarray, sample_rate: int) -> str:
        self._load()
        return self._pipe({"array": wav, "sampling_rate": sample_rate})["text"].strip()

    @staticmethod
    def word_error_rate(reference: str, hypothesis: str) -> float:
        ref_words = reference.lower().split()
        hyp_words = hypothesis.lower().split()
        # Standard Levenshtein edit distance over words.
        d = np.zeros((len(ref_words) + 1, len(hyp_words) + 1), dtype=np.int32)
        d[:, 0] = np.arange(len(ref_words) + 1)
        d[0, :] = np.arange(len(hyp_words) + 1)
        for i in range(1, len(ref_words) + 1):
            for j in range(1, len(hyp_words) + 1):
                cost = 0 if ref_words[i - 1] == hyp_words[j - 1] else 1
                d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + cost)
        return float(d[-1, -1]) / max(1, len(ref_words))

    def wer(self, reference_text: str, wav: np.ndarray, sample_rate: int) -> float:
        hypothesis = self.transcribe(wav, sample_rate)
        return self.word_error_rate(reference_text, hypothesis)


class UTMOSPredictor:
    """Mean opinion score prediction via the `speechmos` package's UTMOS model
    (`pip install speechmos`), lazy-loaded."""

    def __init__(self):
        self._predictor = None

    def _load(self):
        if self._predictor is not None:
            return
        import speechmos.utmos as utmos

        self._predictor = utmos

    def score(self, wav: np.ndarray, sample_rate: int) -> float:
        self._load()
        result = self._predictor.compute(wav, sample_rate)
        return float(result["mos"])


@torch.no_grad()
def measure_barge_in_latency(
    lm_gen: LMGen,
    mimi: MimiModel,
    interrupt_input_track: torch.Tensor,
    onset_frame: int,
    silence_token_check: tp.Callable[[torch.Tensor], bool],
    max_lookahead_frames: int = 50,
) -> tp.Optional[float]:
    """Frames from a scripted interruption onset to the first output frame that
    breaks from the pre-interruption (silence/backchannel-only) pattern,
    converted to milliseconds via the Mimi frame rate. Returns None if the
    model never reacts within `max_lookahead_frames`.

    `silence_token_check(tokens)` should return True if `tokens` (the model's
    sampled agent-side codebooks for one frame) look like "no reaction yet"
    (e.g. still emitting the exact silence tokens) -- see
    distill/init_from_teacher.py's SILENCE_TOKENS-based check for the pattern.
    """
    frame_ms = 1000.0 / mimi.frame_rate
    for offset in range(max_lookahead_frames):
        c = onset_frame + offset
        if c >= interrupt_input_track.shape[-1]:
            break
        tokens = lm_gen.step(input_tokens=interrupt_input_track[:, :, c:c + 1])
        if tokens is not None and not silence_token_check(tokens):
            return offset * frame_ms
    return None


@torch.no_grad()
def measure_rtf(
    lm_gen: LMGen,
    mimi: MimiModel,
    num_frames: int,
    device: str,
) -> dict[str, LatencyStats]:
    """Per-component p50/p95/p99 latency (mimi_encode, temporal, bridge+depth,
    mimi_decode, end-to-end). RTF = mean(end_to_end_ms) / FRAME_BUDGET_MS.

    "temporal" and "bridge+depth" are split by inlining `LMGen.step` (which
    normally times as one opaque call) into its two underlying stages: the main
    transformer (`state.graphed_main`, covers the student's GQA transformer +
    Bridge + frozen out_norm/text_linear for the student, or just the plain
    transformer for the teacher) and the depth transformer
    (`state.graphed_depth`, always frozen). This mirrors exactly what
    `LMGen.step` / `LMGen.process_transformer_output` already do internally --
    no changes to moshi/models/lm.py.
    """
    frame_size = int(mimi.sample_rate / mimi.frame_rate)
    timings = {"mimi_encode": [], "temporal": [], "bridge_depth": [], "mimi_decode": [], "end_to_end": []}

    def sync():
        if device.startswith("cuda"):
            torch.cuda.synchronize()

    silence = torch.zeros(1, 1, frame_size, device=device)
    state = lm_gen._streaming_state
    for _ in range(num_frames):
        sync()
        t0 = time.perf_counter()
        codes = mimi.encode(silence)
        sync()
        t1 = time.perf_counter()

        prepared = lm_gen.prepare_step_input(codes[:, :, 0:1])
        if prepared is None:
            sync()
            t2 = t3 = t4 = time.perf_counter()
        else:
            input_, provided_, target_, model_input_position, target_position = prepared
            transformer_out, text_logits = state.graphed_main(input_)
            sync()
            t2 = time.perf_counter()
            tokens = lm_gen.process_transformer_output(
                transformer_out, text_logits, provided_, target_, model_input_position, target_position,
            )
            sync()
            t3 = time.perf_counter()
            if isinstance(tokens, tuple):
                tokens = tokens[0]
            if tokens is not None:
                _ = mimi.decode(tokens[:, 1:9])
            sync()
            t4 = time.perf_counter()

        timings["mimi_encode"].append((t1 - t0) * 1000)
        timings["temporal"].append((t2 - t1) * 1000)
        timings["bridge_depth"].append((t3 - t2) * 1000)
        timings["mimi_decode"].append((t4 - t3) * 1000)
        timings["end_to_end"].append((t4 - t0) * 1000)

    return {name: LatencyStats.from_samples(vals) for name, vals in timings.items()}


def print_metrics_table(metrics: dict[str, tp.Any]):
    print(f"{'metric':<28}{'value':>16}")
    print("-" * 44)
    for name, value in metrics.items():
        if isinstance(value, LatencyStats):
            for field, v in asdict(value).items():
                print(f"{name + '.' + field:<28}{v:>16.3f}")
        elif isinstance(value, float):
            print(f"{name:<28}{value:>16.4f}")
        else:
            print(f"{name:<28}{str(value):>16}")


# ============================================================================ knowledge / answer-relevance
# Everything above this line measures SPEECH quality: WER of the output against its own text, UTMOS, speaker
# similarity, barge-in latency, RTF. None of it can tell a correct answer from a fluent irrelevant one -- a
# student that replies "Oh, that's a good question. How do you like to talk about a lot of the story?" to
# "What is Bitcoin?" scores perfectly on all of them. That is why the first distillation run looked healthy
# (validation passed, every loss fell, the smoke test passed) while the model could not answer anything.
#
# `KNOWLEDGE_PROBES` is the metric the goal is actually defined on. Run it on the teacher too for a ceiling:
# a probe the teacher also misses is a bad probe, not a student failure.

KNOWLEDGE_PROBES: list[tuple[str, set[str]]] = [
    ("What is Bitcoin?",            {"digital", "currency", "money", "crypto", "cryptocurrency",
                                     "blockchain", "decentralized", "decentralised"}),
    ("Who invented Bitcoin?",       {"satoshi", "nakamoto"}),
    ("What is a crypto wallet?",    {"key", "keys", "private", "store", "stores", "address", "seed", "phrase"}),
    ("What is proof of work?",      {"mining", "miner", "miners", "mine", "energy", "puzzle", "hash", "compute"}),
    ("What is Ethereum?",           {"smart", "contract", "contracts", "blockchain", "platform", "ether", "apps"}),
    ("What is a stablecoin?",       {"dollar", "pegged", "peg", "stable", "value", "fiat", "usd"}),
    ("How do you buy cryptocurrency?", {"exchange", "account", "bank", "card", "buy", "transfer", "wallet"}),
    ("Is Bitcoin safe to invest in?", {"risk", "risky", "volatile", "volatility", "lose", "careful", "research"}),
]

_WORD_RE = re.compile(r"[^a-z0-9' ]+")


def _norm_words(text: str) -> set[str]:
    return set(_WORD_RE.sub(" ", text.lower()).split())


def score_answer(answer_text: str, keywords: set[str]) -> dict[str, tp.Any]:
    """Keyword-recall score for one probe. Deliberately crude: it asks only whether the reply is ABOUT the
    thing that was asked, which is precisely the axis the student failed on. It is a regression gate, not a
    quality score -- do not tune against it past the point where it stops separating models."""
    words = _norm_words(answer_text)
    hit = sorted(keywords & words)
    return {"hit": bool(hit), "n_hits": len(hit), "matched": hit, "answer": answer_text[:300]}


def knowledge_probe_report(answers: dict[str, str],
                           probes: tp.Optional[list[tuple[str, set[str]]]] = None) -> dict[str, tp.Any]:
    """`answers`: {probe question -> the model's reply text}. Returns per-probe results plus the headline
    `answer_relevance` = fraction of probes whose reply mentions at least one expected keyword.

    A trained-and-working student should reach roughly the teacher's score. The failing run scored 0.0: its
    reply to "What is Bitcoin?" contained none of {digital, currency, money, crypto, blockchain, ...}.
    """
    probes = probes if probes is not None else KNOWLEDGE_PROBES
    per_probe = {}
    for question, keywords in probes:
        reply = answers.get(question)
        per_probe[question] = ({"hit": False, "n_hits": 0, "matched": [], "answer": None}
                               if reply is None else score_answer(reply, keywords))
    scored = [v for v in per_probe.values() if v["answer"] is not None]
    return {"answer_relevance": (sum(v["hit"] for v in scored) / len(scored)) if scored else None,
            "probes_scored": len(scored), "probes_total": len(probes), "per_probe": per_probe}


def print_knowledge_report(report: dict[str, tp.Any], label: str = "model"):
    rel = report["answer_relevance"]
    print(f"\n=== knowledge probe: {label} ===")
    print(f"answer_relevance: {'n/a' if rel is None else f'{rel:.2f}'} "
          f"({report['probes_scored']}/{report['probes_total']} probes scored)")
    for question, r in report["per_probe"].items():
        mark = "PASS" if r["hit"] else ("----" if r["answer"] is None else "MISS")
        print(f"  [{mark}] {question}")
        if r["answer"] is not None:
            print(f"         matched={r['matched'] or '(none)'}")
            print(f"         reply  ={r['answer']!r}")


In [ ]:
%%writefile {REPO_DIR}/distill/init_from_teacher.py
# SPDX-License-Identifier: MIT
"""Data-driven initialization of a `StudentLMModel` from a loaded teacher `LMModel`.

This is the part of the pipeline the plan calls out as mattering most: a
randomly-initialized 20-layer/2048-dim/GQA-4:1 student, dropped in front of a
frozen depth transformer it was never trained with, will not produce anything
resembling the teacher's discrete token distribution. Every step here exists to
give the student a head start that is at least structurally aligned with the
teacher before any gradient step is taken.

Because the teacher has no GQA (`num_key_value_heads == num_heads == 32`, plain
MHA -- see distill/gqa_attention.py's docstring), "mean-pool teacher KV heads
into groups" is implemented as: pool groups of *all* 32 teacher heads' K/V
projection rows into `num_key_value_heads` bands (no heads dropped), while
query heads are reduced by strided selection (matching plan's the intent that a
GQA-4:1 student should still see a broad, representative slice of the
teacher's 32 attention heads).

Steps, run in order by `initialize_student`:
  1. `compute_layer_scores`      -- score[l] = 1 - cos(h[l], h[l+1]) per teacher layer.
  2. `select_layers`             -- top-`num_layers` by score, always keeping the
                                     first/last `keep_first`/`keep_last`, order preserved.
  3. `compute_residual_projection` -- one global activation-aware SVD basis
                                     (4096 -> hidden_size) shared by every selected layer,
                                     the embeddings, and the bridge.
  4. per selected layer: `init_attention_layer` (head selection + KV pooling + residual
     projection) and `init_ffn_layer` (own per-layer SVD of the FFN's gated-hidden space).
  5. `init_embeddings`           -- project teacher embedding tables through the same
                                     residual projection.
  6. `init_bridge_least_squares` -- closed-form regression of the Bridge so the frozen
                                     depth transformer gets a roughly-correct input at step 0.
  7. `sanity_check_student`      -- coarse, automatable degeneracy checks. This is a
                                     heuristic gate, not a quality guarantee: LISTEN to the
                                     output before starting a real training run.
"""

from dataclasses import dataclass, field
import logging
import typing as tp

import torch
import torch.nn.functional as F

from moshi.models.lm import LMModel, AUDIO_TOKENS_PER_STREAM, SILENCE_TOKENS
from moshi.models.compression import MimiModel
from moshi.modules.transformer import StreamingTransformerLayer

from .student_model import StudentLMModel
from .bridge import Bridge

logger = logging.getLogger(__name__)

Batch = torch.Tensor  # [B, K, T] token sequences, teacher `LMModel.forward_train` layout


# --------------------------------------------------------------------------
# 1-2. Layer importance probing and selection
# --------------------------------------------------------------------------

@torch.no_grad()
def compute_layer_scores(teacher: LMModel, batches: tp.Sequence[Batch]) -> torch.Tensor:
    """score[l] = 1 - cos(h[l], h[l+1]) for l in [0, num_teacher_layers - 1], averaged
    over all calibration batches, batch and time. `h[0]` is the embedding output (the
    input to layer 0), `h[num_layers]` is the output of the last layer (pre out_norm).
    A high score means layer l changes its input a lot (a "load-bearing" layer); a low
    score means the layer is close to a no-op for this data distribution.
    """
    teacher.eval()
    layers = list(teacher.transformer.layers)
    num_layers = len(layers)
    captured: list[torch.Tensor] = []

    def make_hook(store):
        def hook(module, inputs, output):
            store.append(inputs[0].detach())
            store.append(output.detach())
        return hook

    handles = []
    for layer in layers:
        handles.append(layer.register_forward_hook(make_hook(captured)))

    score_sum = torch.zeros(num_layers, dtype=torch.float64)
    score_count = 0
    try:
        for batch in batches:
            captured.clear()
            codes = batch.to(teacher.device)
            teacher.forward_train(codes)
            # captured holds [in_0, out_0, in_1, out_1, ..., in_{L-1}, out_{L-1}]
            hiddens = [captured[0]] + [captured[2 * i + 1] for i in range(num_layers)]
            for l in range(num_layers):
                h_l = hiddens[l].float()
                h_l1 = hiddens[l + 1].float()
                cos = F.cosine_similarity(h_l, h_l1, dim=-1)
                score_sum[l] += (1.0 - cos).mean().double().item()
            score_count += 1
    finally:
        for h in handles:
            h.remove()

    assert score_count > 0, "No calibration batches provided."
    return (score_sum / score_count).float()


def select_layers(scores: torch.Tensor, num_layers: int, keep_first: int, keep_last: int,
                  spread: bool = True) -> list[int]:
    """Pick `num_layers` teacher layers to initialize the student's stack from.

    `spread=True` (default) spaces the DROPPED layers evenly and picks the lowest-scoring layer inside each
    spacing bucket. `spread=False` is the original behaviour: take the top `num_layers` by score outright.

    Why the default changed. Pure score ranking on the real 32-layer teacher selected
        [0..16, 29, 30, 31]
    -- it dropped layers 17-28, a CONTIGUOUS BLOCK OF 12. `compute_layer_scores` ranks by
    `1 - cos(h[l], h[l+1])`, and in a deep transformer the late-middle layers change their input least in
    cosine terms, so they all rank last together. But small cosine change does not mean small function: a
    layer can make a low-norm, highly targeted write to the residual stream, which is exactly how factual
    recall behaves. Worse, a contiguous hole breaks composition the most: the student's layer initialized
    from teacher layer 29 receives the output of teacher layer 16, a distribution it never saw.

    Spacing the drops keeps the residual pipeline roughly proportional while still removing the
    least-load-bearing layer in each neighbourhood, and it cannot produce a 12-layer hole.
    """
    total = scores.shape[0]
    assert num_layers <= total
    forced = set(range(keep_first)) | set(range(total - keep_last, total))
    assert len(forced) <= num_layers, (
        f"keep_first={keep_first} + keep_last={keep_last} exceeds num_layers={num_layers}"
    )
    remaining_slots = num_layers - len(forced)
    candidates = [l for l in range(total) if l not in forced]
    if not spread:
        ranked = sorted(candidates, key=lambda l: scores[l].item(), reverse=True)
        chosen = forced | set(ranked[:remaining_slots])
        return sorted(chosen)

    n_drop = len(candidates) - remaining_slots
    if n_drop <= 0:
        return sorted(forced | set(candidates))
    drop: set[int] = set()
    for b in range(n_drop):                       # one drop per evenly spaced bucket
        lo = int(round(b * len(candidates) / n_drop))
        hi = int(round((b + 1) * len(candidates) / n_drop))
        bucket = [l for l in candidates[lo:hi] if l not in drop]
        if bucket:
            drop.add(min(bucket, key=lambda l: scores[l].item()))
    leftover = sorted((l for l in candidates if l not in drop), key=lambda l: scores[l].item())
    while len(drop) < n_drop and leftover:        # rounding can leave a bucket empty
        drop.add(leftover.pop(0))
    chosen = forced | (set(candidates) - drop)
    return sorted(chosen)


# --------------------------------------------------------------------------
# 3. Activation-aware SVD projection (shared residual-stream basis)
# --------------------------------------------------------------------------

@dataclass
class Projection:
    """`down`: (teacher_dim, student_dim) encodes a teacher activation vector `t`
    into student space via `s = t @ down`. `up`: (student_dim, teacher_dim)
    reconstructs `t ~= s @ up`. `up @ down == I_{student_dim}` exactly (down's
    columns are an orthonormal SVD basis); `down @ up` is the (rank student_dim)
    projector onto the retained subspace, not identity -- that's the truncation.

    For adapting a teacher WEIGHT matrix `W` (F.linear convention, `y = x @ W^T`)
    rather than a plain activation vector, the two cases are NOT symmetric:
      - `W` reads from a reduced space (its `in` dim shrinks, e.g. attention
        in_proj, FFN linear_in wrt the residual): `W' = W @ up.T`.
      - `W` writes to a reduced space (its `out` dim shrinks, e.g. attention
        out_proj, FFN linear_out wrt the residual): `W' = down.T @ W`.
    Both are derived from `t ~= s @ up` substituted into `y = t @ W^T` /
    `y_reduced = y @ down` respectively -- see the two derivations inline in
    `init_attention_layer` and `init_ffn_layer` if this needs re-deriving.
    """
    down: torch.Tensor
    up: torch.Tensor
    rms: torch.Tensor


@torch.no_grad()
def _collect_activations(module_input_hook_target: torch.nn.Module,
                          run_fn: tp.Callable[[Batch], None],
                          batches: tp.Sequence[Batch]) -> torch.Tensor:
    """Run `run_fn(batch)` for each batch, capturing the input activations seen by
    `module_input_hook_target`, concatenated over batch/time into a single
    `[N, dim]` matrix."""
    captured: list[torch.Tensor] = []

    def hook(module, inputs, output):
        captured.append(inputs[0].detach().float().reshape(-1, inputs[0].shape[-1]))

    handle = module_input_hook_target.register_forward_hook(hook)
    try:
        for batch in batches:
            run_fn(batch)
    finally:
        handle.remove()
    return torch.cat(captured, dim=0)


def _gating_hidden_activation(gate: torch.nn.Module, x: torch.Tensor) -> torch.Tensor:
    """Recomputes `ActivationGating`'s intermediate gated-hidden value (the
    input `linear_out` would see) from the gating module's own INPUT `x`.

    `gating_forward_kernel` (moshi/modules/gating.py) calls `F.linear` directly
    on `linear_in.weight` / `linear_out.weight` rather than invoking
    `linear_in`/`linear_out` as `nn.Module`s -- so a forward hook on
    `gate.linear_out` never fires (its `.forward()` is simply never called).
    Hooking the whole `gate` module's input (which DOES fire, since
    `StreamingTransformerLayer._ff_block` calls `self.gating(x)`) and
    replicating the first half of the kernel here is the only way to observe
    this intermediate value without modifying moshi/modules/gating.py.

    Casts `x` to `gate.linear_in.weight`'s dtype before the matmul regardless
    of what dtype it arrives in -- callers that capture activations via a
    forward hook commonly upcast to float32 for numerically stable statistics
    (e.g. `collect_ffn_hidden_activations` below), while the model itself runs
    in bf16, and `F.linear` requires both operands to match.
    """
    proj = F.linear(x.to(gate.linear_in.weight.dtype), gate.linear_in.weight)
    B, T, _ = proj.shape
    proj = proj.view(B, T, 2, -1)
    return (gate.activation(proj[..., 0, :]) * proj[..., 1, :]).float()


@torch.no_grad()
def collect_ffn_hidden_activations(
    gate: torch.nn.Module, run_fn: tp.Callable[[Batch], None], batches: tp.Sequence[Batch],
) -> torch.Tensor:
    """Like `_collect_activations`, but for the FFN's gated-hidden space
    specifically -- see `_gating_hidden_activation` for why this can't just be
    `_collect_activations(gate.linear_out, ...)`."""
    captured: list[torch.Tensor] = []

    def hook(module, inputs, output):
        captured.append(inputs[0].detach().float())

    handle = gate.register_forward_hook(hook)
    try:
        for batch in batches:
            run_fn(batch)
    finally:
        handle.remove()
    hidden = torch.cat([_gating_hidden_activation(gate, x) for x in captured], dim=0)
    return hidden.reshape(-1, hidden.shape[-1])


@torch.no_grad()
def compute_activation_projection(activations: torch.Tensor, student_dim: int) -> Projection:
    """Activation-aware SVD width reduction: scale by per-channel activation RMS
    before the SVD (so high-energy channels dominate which directions are kept),
    take the top `student_dim` eigenvectors of the SCALED covariance -- but,
    unlike a textbook ASVD writeup, do NOT fold the RMS scale back into
    `down`/`up` afterward. `down`/`up` are the raw orthonormal eigenvectors, so
    `down` stays (semi-)orthogonal (`down.T @ down == I` always; `down @ down.T
    == I` too at full rank, i.e. student_dim == teacher_dim).

    This matters because the residual stream in `transformer.py` passes through
    RMSNorm, whose normalization divides by a SCALAR (the per-row rms) rather
    than a per-channel one. That commutes with `down` cleanly only when `down`
    is norm-preserving, i.e. genuinely (semi-)orthogonal: `||x @ down||^2 = x @
    down @ down.T @ x^T`, which equals `||x||^2` exactly at full rank and is a
    legitimate (expected, trainable-away) truncation loss otherwise. Folding
    the RMS scale into `down`/`up` (as an early version of this function did)
    makes `down @ down.T = diag(rms^2)` instead of (approximately) `I` --
    RMSNorm no longer commutes with the projection even at full rank, which
    showed up as a large, spurious "reproduction error" in
    tests/test_init_sanity.py's equal-width losslessness check that had
    nothing to do with the actual attention/FFN weight-transform math. The
    activation RMS still does real work here: it decides which eigenvectors of
    the covariance are worth keeping (high-activation-energy channels dominate
    the ranking), it's just not baked into the basis vectors themselves.
    """
    teacher_dim = activations.shape[-1]
    assert student_dim <= teacher_dim
    rms = activations.pow(2).mean(dim=0).sqrt().clamp_min(1e-6)  # [teacher_dim]
    scaled = activations * rms  # activation-aware weighting, not a whitening normalize
    # Eigendecomposition of the (teacher_dim x teacher_dim) covariance is far cheaper
    # than an SVD of the (N x teacher_dim) activation matrix for large N.
    cov = (scaled.T @ scaled) / scaled.shape[0]
    eigvals, eigvecs = torch.linalg.eigh(cov)  # ascending order
    order = torch.argsort(eigvals, descending=True)
    v_k = eigvecs[:, order[:student_dim]]  # [teacher_dim, student_dim], orthonormal columns

    down = v_k        # [teacher_dim, student_dim]
    up = v_k.T         # [student_dim, teacher_dim]
    return Projection(down=down, up=up, rms=rms)


# --------------------------------------------------------------------------
# 4a. Attention head selection / GQA KV pooling
# --------------------------------------------------------------------------

def select_query_heads(num_teacher_heads: int, num_student_heads: int) -> list[int]:
    """Evenly-spaced strided selection across all teacher heads, so the student's
    (fewer) query heads still cover the full range of what the teacher learned,
    rather than e.g. always taking the first N. Integer floor spacing guarantees
    exactly `num_student_heads` distinct indices (unlike rounding a linspace,
    which can collide)."""
    assert num_student_heads <= num_teacher_heads
    return [(i * num_teacher_heads) // num_student_heads for i in range(num_student_heads)]


def kv_head_bands(num_teacher_heads: int, num_student_kv_heads: int) -> list[list[int]]:
    """Contiguous bands over ALL teacher heads (none dropped), one band per pooled
    student KV head. `num_teacher_heads` must be divisible by `num_student_kv_heads`.
    """
    assert num_teacher_heads % num_student_kv_heads == 0
    band_size = num_teacher_heads // num_student_kv_heads
    return [list(range(g * band_size, (g + 1) * band_size)) for g in range(num_student_kv_heads)]


def _is_identity_projection(proj: Projection, atol: float = 1e-5) -> bool:
    """True if `proj.down` is (numerically) the identity matrix -- i.e. no actual
    width reduction or basis rotation is happening (e.g. tests/test_bit_exactness.py's
    identity-clone student, or any config where student_dim == teacher_dim and no
    calibration-driven rotation was computed). Used to decide whether a teacher's
    RMSNorm scale (`alpha`) can be copied to the student directly (exact per-channel
    correspondence) instead of reset to a neutral ones-vector.
    """
    if proj.down.shape[0] != proj.down.shape[1]:
        return False
    eye = torch.eye(proj.down.shape[0], device=proj.down.device, dtype=proj.down.dtype)
    return torch.allclose(proj.down, eye, atol=atol)


def _init_norm_alpha(teacher_norm: torch.nn.Module, student_norm: torch.nn.Module, proj: Projection):
    """Sets `student_norm.alpha` from `teacher_norm.alpha` when the residual basis is
    unchanged (identity projection), or resets it to ones otherwise.

    RMSNorm normalizes by a SCALAR (`torch.mean(x**2)`, not per-channel) before
    applying `alpha` elementwise. That means `alpha` does not transfer through an
    arbitrary change of basis: if `down` genuinely rotates or truncates the residual
    space, `student_norm(x @ down)` computed with the teacher's `alpha` values (which
    were learned for teacher-basis channels) would apply the WRONG scale to WRONG
    channels post-rotation, in general.

    Ones is nevertheless the CORRECT value here, not merely a neutral one: both callers
    fold `diag(alpha)` into the linear map that consumes the normalized activations
    (q/k/v `in_proj` for `norm1`, `gating.linear_in` for `norm2`) BEFORE reducing its
    width -- see the folding comments in `init_attention_layer` / `init_ffn_layer`.
    An earlier version only reset alpha and folded nothing, which silently discarded the
    teacher's trained pre-attention and pre-FFN scales (40 vectors of 4096 values on the
    real model) and left P1 "Align" to relearn them from scratch.

    But when `down` is the identity (no rotation at all -- see `_is_identity_projection`),
    each student channel IS the corresponding teacher channel, so the teacher's `alpha`
    is exactly correct and resetting it to ones would be a real, unnecessary numerical
    regression: this was caught by tests/test_bit_exactness.py's identity-clone check
    (a real 7B teacher's alpha values are trained, non-uniform -- forcing them to ones
    produced a large, spurious "reproduction failure" that had nothing to do with the
    actual attention/FFN weight-transform math).
    """
    if _is_identity_projection(proj):
        student_norm.alpha.data.copy_(teacher_norm.alpha.data.to(student_norm.alpha.dtype))
    else:
        torch.nn.init.ones_(student_norm.alpha)


@torch.no_grad()
def init_attention_layer(
    teacher_layer: StreamingTransformerLayer,
    student_layer,  # GQAStreamingTransformerLayer
    q_head_idx: list[int],
    kv_bands: list[list[int]],
    proj: Projection,
    teacher_head_dim: int = 128,
):
    attn_t = teacher_layer.self_attn
    attn_s = student_layer.self_attn
    num_teacher_heads = attn_t.num_heads
    d_in = attn_t.in_proj_weight.shape[1]
    assert attn_t.in_proj_weight.shape[0] == 3 * num_teacher_heads * teacher_head_dim

    # Fold the PRE-NORM scale into the projection that consumes it, instead of discarding it.
    # `_sa_block` is pre-norm (moshi/modules/transformer.py): update = self_attn(norm1(x)), and
    # `_rms_norm(x, alpha) = (x * rsqrt(mean(x^2) + eps)) * alpha`. Because the only consumer of
    # norm1(x) is the LINEAR q/k/v projection,
    #     W @ ((x / rms) * alpha)  ==  (W @ diag(alpha)) @ (x / rms)
    # exactly. So scaling `in_proj_weight`'s INPUT columns by alpha before the width reduction lets the
    # student's norm1.alpha be ones and still reproduce the teacher -- whereas `_init_norm_alpha` alone
    # resets alpha to ones and throws the teacher's trained scale away (40 vectors of 4096 trained values
    # on the real model). Measured on a tiny teacher with non-uniform alphas, folding recovers ~0.02-0.03
    # top-1 text agreement and roughly halves the content KL at equal width.
    # At an identity projection nothing is folded: `_init_norm_alpha` copies alpha across verbatim, which
    # keeps tests/test_bit_exactness.py's identity-clone path exact.
    in_proj = attn_t.in_proj_weight
    if not _is_identity_projection(proj):
        alpha1 = teacher_layer.norm1.alpha.detach().reshape(1, -1).to(in_proj.dtype)
        in_proj = in_proj * alpha1                      # a copy; the teacher is never mutated

    w_qkv = in_proj.view(3, num_teacher_heads, teacher_head_dim, d_in)
    w_q, w_k, w_v = w_qkv[0], w_qkv[1], w_qkv[2]  # each [num_teacher_heads, head_dim, d_in]

    down = proj.down.to(w_q.dtype)
    up_t = proj.up.to(w_q.dtype).T  # [teacher_dim, student_dim]

    # Query/Key/Value all READ from the residual (it's their `in` dimension), so per
    # the Projection docstring this is the `W @ up.T` case, not `W @ down`.
    w_q_sel = w_q[q_head_idx].reshape(len(q_head_idx) * teacher_head_dim, d_in)
    attn_s.q_proj.weight.copy_((w_q_sel @ up_t).to(attn_s.q_proj.weight.dtype))

    # KV: mean-pool each band over ALL teacher heads (no heads dropped), then reduce input dim.
    def pooled(w):
        bands = [w[band].mean(dim=0) for band in kv_bands]  # each [head_dim, d_in]
        return torch.cat(bands, dim=0)  # [num_kv_heads * head_dim, d_in]

    w_k_pooled = pooled(w_k)
    w_v_pooled = pooled(w_v)
    attn_s.k_proj.weight.copy_((w_k_pooled @ up_t).to(attn_s.k_proj.weight.dtype))
    attn_s.v_proj.weight.copy_((w_v_pooled @ up_t).to(attn_s.v_proj.weight.dtype))

    # Output projection WRITES to the residual (it's its `out` dimension): the
    # `down.T @ W` case. Select the same query heads on the input (concat) side first.
    w_out_t = attn_t.out_proj.weight  # [teacher_dim, num_teacher_heads*head_dim]
    w_out_t = w_out_t.view(d_in, num_teacher_heads, teacher_head_dim)
    w_out_sel = w_out_t[:, q_head_idx, :].reshape(d_in, len(q_head_idx) * teacher_head_dim)
    down_T = down.T  # [student_dim, teacher_dim]
    w_out_student = down_T @ w_out_sel  # [student_dim, len(q_head_idx)*head_dim]
    attn_s.out_proj.weight.copy_(w_out_student.to(attn_s.out_proj.weight.dtype))

    _init_norm_alpha(teacher_layer.norm1, student_layer.norm1, proj)


# --------------------------------------------------------------------------
# 4b. FFN (gated) width reduction: residual dim via `proj` (a rotation, valid
#     since residual reads/writes are linear); hidden dim via CHANNEL
#     SELECTION, not an SVD rotation -- see init_ffn_layer docstring for why.
# --------------------------------------------------------------------------

def select_ffn_channels_from_rms(rms: torch.Tensor, student_hidden: int) -> torch.Tensor:
    """Which gated-FFN channels to keep, given their per-channel activation RMS: highest energy first,
    then sorted back into original order so `linear_in`'s two halves and `linear_out`'s columns stay
    consistently indexed."""
    idx = torch.argsort(rms, descending=True)[:student_hidden]
    return torch.sort(idx).values


def select_ffn_channels(hidden_activations: torch.Tensor, student_hidden: int) -> torch.Tensor:
    """Same selection, from the full `[N, teacher_hidden]` activation matrix."""
    return select_ffn_channels_from_rms(hidden_activations.pow(2).mean(dim=0).sqrt(), student_hidden)


@torch.no_grad()
def collect_ffn_channel_rms(
    gates: dict[int, torch.nn.Module], run_fn: tp.Callable[[Batch], None], batches: tp.Sequence[Batch],
) -> dict[int, torch.Tensor]:
    """Per-channel activation RMS of the gated-hidden space for MANY layers, in ONE pass over `batches`.

    This replaces calling `collect_ffn_hidden_activations` once per selected layer. That cost one full
    teacher forward pass over every calibration batch PER LAYER -- 16 x 64 = 1024 teacher forwards on the
    student_ppx_m run, which made initialization take 1096 s. Rank 0 runs this alone while the other ranks
    block in `dist.broadcast_object_list`, and NCCL's heartbeat monitor SIGABRTs a rank whose collective has
    sat for ~8 minutes, so the slow init did not just waste time -- it killed the job outright.

    `select_ffn_channels` only ever needed the per-channel RMS, and RMS is a streaming statistic: accumulate
    sum-of-squares per channel and divide by the count at the end. So one pass with every layer hooked at
    once is enough, and the memory is `num_layers x teacher_hidden` floats instead of the
    `[N, teacher_hidden]` matrix per layer (127,744 x 11,264 fp32 = 5.75 GB each, which is why the original
    had to do them one at a time).
    """
    sumsq: dict[int, torch.Tensor] = {}
    count: dict[int, int] = {}
    handles = []

    def make_hook(key: int, gate: torch.nn.Module):
        def hook(module, inputs, output):
            # `_gating_hidden_activation` recomputes linear_out's true input from the gating module's input;
            # see its docstring for why a hook on `linear_out` never fires.
            h = _gating_hidden_activation(gate, inputs[0].detach()).reshape(-1, gate.linear_out.weight.shape[1])
            s = h.pow(2).sum(dim=0).double()
            if key in sumsq:
                sumsq[key] += s
                count[key] += h.shape[0]
            else:
                sumsq[key] = s
                count[key] = h.shape[0]
        return hook

    for key, gate in gates.items():
        handles.append(gate.register_forward_hook(make_hook(key, gate)))
    try:
        for batch in batches:
            run_fn(batch)
    finally:
        for h in handles:
            h.remove()
    missing = [k for k in gates if k not in sumsq]
    assert not missing, f"no gated-hidden activations captured for layers {missing}"
    return {k: (sumsq[k] / max(count[k], 1)).sqrt().float() for k in sumsq}


@torch.no_grad()
def init_ffn_layer(
    teacher_layer: StreamingTransformerLayer,
    student_layer,  # GQAStreamingTransformerLayer
    proj: Projection,
    hidden_activations: tp.Optional[torch.Tensor] = None,
    channel_rms: tp.Optional[torch.Tensor] = None,
):
    """`hidden_activations` are the FFN's gated-hidden values (post SiLU-gate *
    value, see `_gating_hidden_activation`) -- i.e. `linear_out`'s true input.
    Pass `channel_rms` instead (from `collect_ffn_channel_rms`) to skip materializing that matrix; only its
    per-channel RMS is ever used. Exactly one of the two is required.

    Unlike the residual stream, the FFN hidden dimension sits behind an
    ELEMENTWISE nonlinearity (`activation(gate) * value`, per channel). An SVD
    rotation of that space (as an earlier version of this function did, mapping
    it through a `compute_activation_projection`-style basis, symmetric to how
    the residual is handled) is mathematically wrong: a rotation mixes
    channels together BEFORE the nonlinearity is (conceptually) re-applied,
    but the actual elementwise nonlinearity in `gating_forward_kernel` only
    ever sees the ORIGINAL, unrotated channel basis -- `activation(x @ R) !=
    activation(x) @ R` for a rotation `R` in general. This showed up as a
    large (~0.6 max-abs-logit), layer-compounding numerical error in
    tests/test_init_sanity.py that was NOT present in the attention path
    (which stays entirely linear, aside from softmax's normalization, and
    doesn't touch this hidden space at all).

    The correct reduction here is the same idea already used for attention
    heads: SELECT a subset of the teacher's hidden channels (by activation
    RMS) rather than rotating into a new basis. Selection preserves the
    elementwise correspondence exactly -- each kept channel is still
    `activation(gate_c) * value_c` for the SAME original channel `c`, just
    fewer of them.
    """
    gate_t = teacher_layer.gating
    gate_s = student_layer.gating
    student_hidden = gate_s.linear_out.weight.shape[1]
    teacher_hidden = gate_t.linear_out.weight.shape[1]

    assert (hidden_activations is None) != (channel_rms is None),         "init_ffn_layer needs exactly one of hidden_activations / channel_rms"
    channel_idx = (select_ffn_channels_from_rms(channel_rms, student_hidden) if channel_rms is not None
                   else select_ffn_channels(hidden_activations, student_hidden))
    down_r = proj.down.to(gate_t.linear_in.weight.dtype)
    up_r = proj.up.to(gate_t.linear_in.weight.dtype)

    # Same pre-norm folding as in `init_attention_layer`: `_ff_block` computes gating(norm2(x)), and
    # `linear_in` is linear, so diag(norm2.alpha) folds into its input columns exactly rather than being
    # discarded by `_init_norm_alpha`.
    lin_in = gate_t.linear_in.weight
    if not _is_identity_projection(proj):
        alpha2 = teacher_layer.norm2.alpha.detach().reshape(1, -1).to(lin_in.dtype)
        lin_in = lin_in * alpha2                        # a copy; the teacher is never mutated

    # linear_in: [2*teacher_hidden, teacher_dim] -> [2*student_hidden, student_dim].
    # READS the residual (`up_r.T`, Case A); selects the same `student_hidden` output
    # channels in BOTH halves (gate / value, see moshi/modules/gating.py), since hidden
    # channel c = activation(gate[c]) * value[c] depends only on row c of each half.
    w_in = lin_in.view(2, teacher_hidden, -1)[:, channel_idx, :]
    w_in_reduced = torch.stack([w_in[i] @ up_r.T for i in range(2)], dim=0)
    gate_s.linear_in.weight.copy_(w_in_reduced.reshape(2 * student_hidden, -1).to(gate_s.linear_in.weight.dtype))

    # linear_out: [teacher_dim, teacher_hidden] -> [student_dim, student_hidden].
    # Selects the same channels on its input (columns), WRITES the residual (`down_r.T`, Case B).
    w_out_sel = gate_t.linear_out.weight[:, channel_idx]
    w_out_reduced = down_r.T @ w_out_sel
    gate_s.linear_out.weight.copy_(w_out_reduced.to(gate_s.linear_out.weight.dtype))

    _init_norm_alpha(teacher_layer.norm2, student_layer.norm2, proj)


# --------------------------------------------------------------------------
# 5. Embeddings
# --------------------------------------------------------------------------

@torch.no_grad()
def init_embeddings(teacher: LMModel, student: StudentLMModel, proj: Projection):
    down = proj.down.to(teacher.text_emb.weight.dtype)
    student.text_emb.weight.copy_((teacher.text_emb.weight @ down).to(student.text_emb.weight.dtype))
    for cb in range(student.n_q):
        student.emb[cb].weight.copy_(
            (teacher.emb[cb].weight @ down).to(student.emb[cb].weight.dtype)
        )


# --------------------------------------------------------------------------
# 6. Bridge: closed-form least-squares fit
# --------------------------------------------------------------------------

@torch.no_grad()
def init_bridge_least_squares(student: StudentLMModel, teacher: LMModel, batches: tp.Sequence[Batch],
                               ridge: float = 1e-3):
    """Regress the Bridge so that `bridge(student_raw_hidden) ~= teacher_raw_hidden`
    (both pre-`out_norm`), giving the frozen depth transformer a roughly correct
    input from step 0 rather than from random init.

    Uses RIDGE (L2-regularized) least squares rather than a raw `torch.linalg.lstsq`
    solve. The system has `student_dim` unknowns per output column (e.g. 2048 for
    student_ppx_s); a realistic calibration set (a handful of batches, particularly
    during a quick/smoke-test run) can easily provide fewer than `student_dim`
    effective rows, or a near-rank-deficient covariance even with more. A raw lstsq
    solve in that regime is not guaranteed to be well-behaved -- this surfaced as an
    actually non-finite ("inf" residual) solution in practice, which then poisoned
    every OTHER parameter's gradient a few training steps later via
    `clip_grad_norm_` (a single non-finite gradient makes the whole-model clip
    coefficient non-finite, corrupting the entire model on that optimizer step).
    Ridge regression is unconditionally well-posed for `ridge > 0`: adding
    `ridge * I` to `X^T X` makes it positive-definite even when `X` itself is
    rank-deficient, at the cost of a small bias toward a smaller-norm solution.
    """
    student.eval()
    teacher.eval()
    xs, ys = [], []
    for batch in batches:
        codes = batch.to(student.device)
        teacher_hidden = teacher.transformer(teacher.embed_codes(codes[:, :, :-1]))
        student_hidden = student.transformer(student.embed_codes(codes[:, :, :-1]))
        xs.append(student_hidden.float().reshape(-1, student_hidden.shape[-1]))
        ys.append(teacher_hidden.float().reshape(-1, teacher_hidden.shape[-1]))
    X = torch.cat(xs, dim=0)
    Y = torch.cat(ys, dim=0)

    student_dim = X.shape[-1]
    xtx = X.T @ X
    xty = X.T @ Y
    reg = ridge * torch.diagonal(xtx).mean().clamp_min(1e-6)
    xtx_reg = xtx + reg * torch.eye(student_dim, device=X.device, dtype=X.dtype)
    solution = torch.linalg.solve(xtx_reg, xty)  # [student_dim, teacher_dim]

    if not torch.isfinite(solution).all():
        logger.warning(
            "Bridge least-squares fit produced non-finite values even with ridge "
            "regularization -- falling back to a zero-initialized bridge linear weight "
            "(P1 'Align' training will need to learn it from scratch). This usually means "
            "the calibration data itself contains NaN/Inf (check your dataset), not just "
            "an ill-conditioned fit."
        )
        solution = torch.zeros_like(solution)

    student.bridge.linear.weight.copy_(solution.T.to(student.bridge.linear.weight.dtype))

    residual_rms = Y.pow(2).mean(dim=0).sqrt().clamp_min(1e-6)
    student.bridge.norm.alpha.data.copy_(
        residual_rms.view(1, 1, -1).to(student.bridge.norm.alpha.dtype)
    )
    return {"bridge_lstsq_residual": (X @ solution - Y).pow(2).mean().sqrt().item()}


# --------------------------------------------------------------------------
# 7. Zero-shot sanity check
# --------------------------------------------------------------------------

@dataclass
class SanityReport:
    passed: bool
    silence_frame_fraction: float
    unique_token_fraction: float
    nan_or_inf: bool
    notes: list[str] = field(default_factory=list)


@torch.no_grad()
def sanity_check_student(
    student: StudentLMModel,
    mimi: MimiModel,
    voice_prompt_path: str,
    persona_text_tokens: tp.Optional[list[int]],
    num_frames: int = 200,
    max_silence_fraction: float = 0.9,
    min_unique_token_fraction: float = 0.02,
) -> SanityReport:
    """Coarse, automatable degeneracy checks -- NOT a quality guarantee. Catches
    the failure mode the plan warns about ("if it outputs noise, stop and report")
    where the student collapses to constant silence/garbage tokens; it does not
    replace listening to the output. Thresholds here are heuristic guards against
    total collapse, not benchmarked quality targets.

    Mirrors the exact prompt-loading flow used by `moshi.offline.run_inference`
    (voice prompt -> silence -> text prompt -> silence, via `LMGen.step_system_prompts`),
    then lets the student free-run for `num_frames` with a placeholder ("sine") input
    on the other-party channel, same convention used during prompt loading.

    `voice_prompt_path` must be raw audio (.wav), not a `.pt` embedding cache -- those
    are tied to whichever model produced them (typically the teacher) and are not valid
    for the student; see the assertion below for why.
    """
    # LMGen asserts the model is not in training mode; nn.Module defaults to training=True,
    # so don't depend on the caller having called .eval() first (build_student_lm does, but a
    # freshly-constructed StudentLMModel passed in directly would otherwise hit that assertion).
    was_training = student.training
    student.eval()
    try:
        return _sanity_check_student_impl(
            student, mimi, voice_prompt_path, persona_text_tokens, num_frames,
            max_silence_fraction, min_unique_token_fraction,
        )
    finally:
        student.train(was_training)


@torch.no_grad()
def _sanity_check_student_impl(
    student: StudentLMModel,
    mimi: MimiModel,
    voice_prompt_path: str,
    persona_text_tokens: tp.Optional[list[int]],
    num_frames: int,
    max_silence_fraction: float,
    min_unique_token_fraction: float,
) -> SanityReport:
    from moshi.models.lm import LMGen

    # `.pt` voice-prompt files (e.g. the shipped voices.tgz's NATF2.pt) are PRE-COMPUTED
    # embeddings -- `LMGen.load_voice_prompt_embeddings` skips audio encoding and Mimi
    # entirely, replaying `state["embeddings"]` straight into `step_embeddings`. Those
    # tensors were produced by whichever model originally called
    # `save_voice_prompt_embeddings=True` (the teacher, at its 4096-dim embed_codes output)
    # -- they are not just "a voice", they're a specific model's embedding space, and feeding
    # them into the student's differently-shaped transformer fails with a shape mismatch deep
    # inside the first RMSNorm rather than at this obviously-wrong call site. Always use raw
    # audio (.wav) for the student, so it computes its own embeddings via its own embed_codes.
    assert not voice_prompt_path.endswith(".pt"), (
        f"{voice_prompt_path} is a pre-computed embedding cache (produced by whichever model "
        "called save_voice_prompt_embeddings=True, typically the teacher) -- it cannot be "
        "reused for the student, which has a different embed_codes output width. Pass a raw "
        "audio (.wav) voice prompt instead, so the student computes its own embeddings."
    )

    notes: list[str] = []
    lm_gen = LMGen(student, device=student.device, use_sampling=True,
                   audio_silence_frame_cnt=int(0.5 * mimi.frame_rate),
                   sample_rate=mimi.sample_rate, frame_rate=mimi.frame_rate)
    with lm_gen.streaming(1), mimi.streaming(1):
        mimi.reset_streaming()
        lm_gen.reset_streaming()
        lm_gen.load_voice_prompt(voice_prompt_path)
        lm_gen.text_prompt_tokens = persona_text_tokens
        lm_gen.step_system_prompts(mimi)
        mimi.reset_streaming()

        generated_codes = []
        for _ in range(num_frames):
            tokens = lm_gen.step(input_tokens=lm_gen._encode_sine_frame())
            if tokens is not None:
                generated_codes.append(tokens[:, 1:1 + AUDIO_TOKENS_PER_STREAM])

    if not generated_codes:
        return SanityReport(False, 1.0, 0.0, False, ["No frames were generated at all."])

    codes = torch.cat(generated_codes, dim=-1)  # [1, 8, T]
    silence = torch.as_tensor(SILENCE_TOKENS, device=codes.device).view(1, 8, 1)
    silence_fraction = (codes == silence).all(dim=1).float().mean().item()
    unique_fraction = codes.unique().numel() / codes.numel()

    audio = mimi.decode(codes)
    nan_or_inf = bool(torch.isnan(audio).any() or torch.isinf(audio).any())

    if silence_fraction > max_silence_fraction:
        notes.append(f"{silence_fraction:.1%} of frames are exact silence tokens (collapse suspected).")
    if unique_fraction < min_unique_token_fraction:
        notes.append(f"Only {unique_fraction:.1%} of emitted tokens are unique (collapse suspected).")
    if nan_or_inf:
        notes.append("Decoded audio contains NaN/Inf.")

    passed = not notes
    if not passed:
        logger.warning("Student sanity check FAILED: %s. Listen to the output before training.", notes)
    return SanityReport(passed, silence_fraction, unique_fraction, nan_or_inf, notes)


# --------------------------------------------------------------------------
# Orchestration
# --------------------------------------------------------------------------

@torch.no_grad()
def initialize_student(
    student: StudentLMModel,
    teacher: LMModel,
    calibration_batches: tp.Sequence[Batch],
    keep_first: int = 3,
    keep_last: int = 3,
    spread_layers: bool = True,
) -> dict:
    """Run the full initialization pipeline (steps 1-6 above) in place on `student`.
    Returns a dict of diagnostics (layer scores, chosen layers, bridge residual)
    for logging -- run `sanity_check_student` separately afterwards.
    """
    teacher.eval()
    student.eval()
    num_student_layers = len(student.transformer.layers)

    scores = compute_layer_scores(teacher, calibration_batches)
    selected = select_layers(scores, num_student_layers, keep_first, keep_last, spread=spread_layers)
    logger.info("Selected teacher layers: %s (scores=%s)", selected, [round(scores[l].item(), 4) for l in selected])

    def run_teacher(batch: Batch):
        teacher.forward_train(batch.to(teacher.device))

    residual_acts = _collect_activations(teacher.out_norm, run_teacher, calibration_batches)
    proj = compute_activation_projection(residual_acts, student.student_config.hidden_size)

    num_teacher_heads = teacher.transformer.layers[0].self_attn.num_heads
    q_head_idx = select_query_heads(num_teacher_heads, student.student_config.num_attention_heads)
    kv_bands = kv_head_bands(num_teacher_heads, student.student_config.num_key_value_heads)

    # ONE teacher pass for every selected layer's FFN channel statistics, not one pass per layer. The
    # per-layer version cost num_layers x num_calibration_batches teacher forwards (16 x 64 = 1024 on
    # student_ppx_m), which took 1096 s -- long enough that NCCL's heartbeat monitor aborted the ranks
    # waiting on rank 0 and killed the job. See `collect_ffn_channel_rms`.
    ffn_rms = collect_ffn_channel_rms(
        {t: teacher.transformer.layers[t].gating for t in selected}, run_teacher, calibration_batches)
    for student_idx, teacher_idx in enumerate(selected):
        teacher_layer = teacher.transformer.layers[teacher_idx]
        student_layer = student.transformer.layers[student_idx]
        init_attention_layer(teacher_layer, student_layer, q_head_idx, kv_bands, proj)
        init_ffn_layer(teacher_layer, student_layer, proj, channel_rms=ffn_rms[teacher_idx])

    init_embeddings(teacher, student, proj)
    bridge_diag = init_bridge_least_squares(student, teacher, calibration_batches)

    return {
        "selected_layers": selected,
        "layer_scores": scores.tolist(),
        **bridge_diag,
    }


In [ ]:
%%writefile {REPO_DIR}/distill/configs/student_ppx_s.yaml
# Student temporal transformer ("S" size). Everything under `teacher:` and `bridge:`
# documents the frozen interface this student must plug into -- it does not
# re-instantiate those modules, it loads them verbatim from the teacher checkpoint.
#
# NOTE on the numbers below vs. an earlier draft of this config: the teacher's real
# temporal-transformer context is 3000 frames (not 4096), and the teacher has no GQA
# (num_key_value_heads == num_heads == 32, plain MHA) -- see distill/init_from_teacher.py
# for how GQA and the width reduction are actually derived from a plain-MHA teacher.
name: student_ppx_s

num_hidden_layers: 20
hidden_size: 2048
intermediate_size: 5632      # dim_feedforward fed to the gated FFN; actual SwiGLU hidden
                              # width is (2 * intermediate_size) // 3, per moshi/modules/gating.py
num_attention_heads: 16      # query heads; head_dim = hidden_size / num_attention_heads = 128
num_key_value_heads: 4       # GQA 4:1 (16 query heads / 4 groups of 4)
rope_theta: 10000            # copied verbatim from the teacher's max_period
max_frames: 1500             # student attention context cap (half the teacher's real 3000)
norm: rms_norm_f32
gating: silu
layer_scale: null
causal: true

# Layer-selection / init hyperparameters consumed by distill/init_from_teacher.py
init:
  num_calibration_batches: 32
  keep_first: 3
  keep_last: 3

# Reference dimensions of the FROZEN components this student reuses as-is from the
# teacher checkpoint (depformer, output heads, embeddings stay teacher-shaped; Mimi is
# untouched). These are documentation only -- loaders.py reads the real shapes off the
# teacher state dict, it does not trust these numbers.
teacher_reference:
  dim: 4096
  num_heads: 32
  num_layers: 32
  context: 3000
  n_q: 16
  dep_q: 16
  card: 2048
  text_card: 32000
  depformer_dim: 1024
  depformer_num_heads: 16
  depformer_num_layers: 6
  depformer_dim_feedforward: 4224
  max_period: 10000

# Bridge: Linear(hidden_size -> bridge.out_dim) + RMSNorm, new & learned.
# out_dim == teacher_reference.dim: the bridge stands in for the teacher's
# post-transformer, pre-out_norm hidden state, so the frozen out_norm / text_linear /
# depformer_in / depformer / linears chain downstream is reused completely unmodified.
bridge:
  in_dim: 2048
  out_dim: 4096


In [ ]:
%%writefile {REPO_DIR}/distill/configs/student_ppx_m.yaml
# Student temporal transformer ("M" size): the SAME ~1B class as student_ppx_s, with the parameters
# reallocated from depth to FFN width -- because the FFN is where a transformer stores facts.
#
# Why this config exists
# ----------------------
# `student_ppx_s` reduces the teacher's temporal transformer on five axes at once. Measured against the real
# teacher (dim 4096, 32 layers, hidden_scale 4.125 -> SwiGLU hidden 11264):
#
#                     teacher        ppx_s          ppx_m (this file)
#   layers            32             20   (62%)    16   (50%)
#   residual          4096           2048 (50%)    2048 (50%)
#   query heads       32             16   (50%)    16   (50%)
#   KV heads          32             4  (mean of   8  (mean of 4
#                                        8 heads)      heads -- half the pooling damage)
#   FFN hidden        11264          3754 (33.3%)  8192 (72.7%)
#   FFN memory slots  360,448        75,080        131,072
#                                    (20.8%)       (36.4%)
#   transformer       6.577 B        0.671 B       1.007 B
#   trainable total   --             0.81 B        1.15 B
#
# `init_from_teacher.select_ffn_channels` keeps FFN hidden channels by activation RMS, which preferentially
# keeps dense always-on channels and drops the sparse high-specificity ones -- exactly where individual facts
# live. Keeping 20.8% of those slots is the single largest knowledge loss in the init, and no amount of
# retraining on a ~0.5M-token audio corpus puts them back. 36.4% is a much better starting point for the same
# parameter class.
#
# Cost: ~1.5x the temporal-transformer FLOPs of student_ppx_s. On the GPU that measured
# `temporal.mean_ms = 17.1`, `bridge_depth.mean_ms = 28.3` (frozen, unchanged) and `rtf 0.786`, expect
# temporal ~25 ms and rtf ~0.85-0.90 -- still real-time, but with less headroom. MEASURE IT with the streaming
# notebook's benchmark cell before committing to a long run; if rtf >= 1.0 on your GPU, drop
# num_hidden_layers to 14 (rtf ~0.82, trainable ~1.02 B) rather than shrinking intermediate_size.
#
# Everything else is deliberately identical to student_ppx_s, so this stays a drop-in: hidden_size 2048 keeps
# `bridge.in_dim == hidden_size`, head_dim stays 128 for the teacher's RoPE geometry, and the frozen
# out_norm / text_linear / depformer_in / depformer / linears chain is reused byte-for-byte. The export format
# and `moshi.server` / `moshi.offline --model student` paths are unchanged -- only `--student-config` differs.
name: student_ppx_m

num_hidden_layers: 16        # 20 -> 16: pays for the wider FFN below
hidden_size: 2048            # unchanged (bridge.in_dim must equal this; head_dim = 2048/16 = 128)
intermediate_size: 12288     # 5632 -> 12288 => SwiGLU hidden (2 * 12288) // 3 = 8192 (was 3754)
num_attention_heads: 16      # query heads; head_dim = 128, as the teacher's RoPE requires
num_key_value_heads: 8       # 4 -> 8: GQA 2:1. Each student KV head is the mean of 4 teacher heads instead
                             # of 8, halving the damage `kv_head_bands` does to the attention geometry.
rope_theta: 10000            # copied verbatim from the teacher's max_period
max_frames: 1500             # student attention context cap (half the teacher's real 3000)
norm: rms_norm_f32
gating: silu
layer_scale: null
causal: true

init:
  num_calibration_batches: 64   # see distill/train.py --calib-batches: 16 was thin for a 4096x4096 covariance
  keep_first: 3
  keep_last: 3

# Reference dimensions of the FROZEN components reused as-is from the teacher checkpoint. Documentation only --
# loaders.py reads the real shapes off the teacher state dict.
teacher_reference:
  dim: 4096
  num_heads: 32
  num_layers: 32
  context: 3000
  n_q: 16
  dep_q: 16
  card: 2048
  text_card: 32000
  depformer_dim: 1024
  depformer_num_heads: 16
  depformer_num_layers: 6
  depformer_dim_feedforward: 4224
  max_period: 10000

bridge:
  in_dim: 2048
  out_dim: 4096


In [ ]:
%%writefile {REPO_DIR}/tools/build_text_corpus.py
#!/usr/bin/env python3
# SPDX-License-Identifier: MIT
"""Build the text corpus for the student's text-only distillation stage.

    python tools/build_text_corpus.py                       # -> /workspace/ppx_text_corpus (or ./ppx_text_corpus)
    python tools/build_text_corpus.py --out /some/dir
    python tools/build_text_corpus.py --no-fetch            # rebuild from the raw cache, no network

Why a dedicated corpus
----------------------
The student's temporal transformer is the only part of PersonaPlex that holds world knowledge, and
`distill/init_from_teacher.py` reduces it to ~10% of the teacher's parameters (FFN memory slots: 20.8%).
Recovering its text behaviour needs text tokens, and the 100 h audio corpus holds well under 1M agent text
tokens in total -- ~90% of its text stream is PAD/EPAD. Text-only steps (`distill/train.py --text-data`) feed
the same transformer one position per token with no Mimi and no depformer, so a real text corpus fits in the
same GPU budget.

The objective is KL against the teacher, so this corpus does NOT need answers in it -- it only has to COVER
the domain. What matters is that it is explanatory prose about the right subjects.

Sources (all free-culture licensed, all human-written explanatory prose)
-----------------------------------------------------------------------
1. English Wikipedia, via the MediaWiki `extracts` API (CC BY-SA 4.0) -- breadth across every requested
   topic, plus history, Satoshi Nakamoto, Ethereum, stablecoins and security.
2. "Mastering Bitcoin", 3rd edition, Antonopoulos/Harding (CC BY-SA 4.0), AsciiDoc from the official repo --
   Bitcoin depth: keys, wallets, transactions, signatures, the network, the blockchain, mining and proof of
   work, security. Appendix A is the Bitcoin whitepaper itself.
3. ethereum.org developer documentation (CC BY 4.0), Markdown from the official repo -- Ethereum depth:
   accounts, transactions, gas, smart contracts, the EVM, proof of stake, standards, scaling.

Deliberately NOT used: price/market data, OHLC series, charts, tweets/Reddit/forum scrapes, SEO spam,
"top 10 coins" listicles, and noisy scraped crypto datasets. They teach the model nothing about what Bitcoin
IS, and price tables in particular are mostly digits -- which is why `_quality_ok` rejects digit-heavy text.

Output (exactly what distill/data/text_dataset.py reads)
-------------------------------------------------------
`<out>/*.jsonl`, one JSON object per line with a `"text"` key -- the first key `_iter_documents` looks for.
One line = one document, and `build_token_cache` appends SEPARATOR (text id 3) after each, so document
boundaries land on the token the model already reads as "nothing being said". Extra keys (`title`, `topic`,
`source`, `license`) are ignored by the loader and kept for provenance.
"""

from __future__ import annotations

import argparse
import hashlib
import html
import json
import re
import sys
import time
import typing as tp
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

UA = "PersonaPlexDistillCorpus/1.0 (student-distillation research corpus builder)"

# ----------------------------------------------------------------------------------------- source definitions

# topic -> Wikipedia article titles. Topics are the ones the corpus must cover; titles are resolved through
# redirects by the API (`redirects=1`), so e.g. "Bitcoin mining" lands on the right article.
WIKIPEDIA: dict[str, list[str]] = {
    "bitcoin": [
        "Bitcoin", "Bitcoin network", "Bitcoin protocol", "Bitcoin Core", "Bitcoin scalability problem",
        "Lightning Network", "SegWit", "Bitcoin ATM", "Legality of cryptocurrency by country or territory",
    ],
    "bitcoin_history": [
        "History of bitcoin", "Satoshi Nakamoto", "Genesis block", "Mt. Gox", "Cypherpunk",
        "Crypto-anarchism", "Hashcash", "B-money", "DigiCash",
        # "Bit Gold" is not an article; its inventor's page is where that history lives.
        "Nick Szabo",
    ],
    "bitcoin_transactions": [
        "Unspent transaction output", "Double-spending", "Merkle tree", "Transaction fee",
    ],
    "blockchain": [
        "Blockchain", "Distributed ledger", "Fork (blockchain)", "Block (data storage)",
        "Consensus (computer science)", "Byzantine fault", "Peer-to-peer", "Distributed computing",
    ],
    "crypto_fundamentals": [
        "Cryptocurrency", "Digital currency", "Virtual currency", "Cryptocurrency exchange",
        "Initial coin offering", "Central bank digital currency",
        "Decentralized application", "Decentralized autonomous organization", "Cryptocurrency bubble",
        "Litecoin", "Monero", "Dogecoin", "Ripple Labs", "Cardano (blockchain platform)",
        "Solana (blockchain platform)", "Binance", "Coinbase",
        # "Token (blockchain)" is not an article; these cover token platforms concretely instead.
        "Tron (blockchain)", "Polygon (blockchain)",
    ],
    "wallets_and_keys": [
        # Seed phrases / BIP-32 have no standalone article; "Cryptocurrency wallet" plus Mastering
        # Bitcoin ch04-ch05 cover deterministic wallets in depth.
        "Cryptocurrency wallet", "Public-key cryptography", "Digital signature",
        "Elliptic-curve cryptography", "Elliptic Curve Digital Signature Algorithm", "Key (cryptography)",
        "Cryptographic hash function", "SHA-2", "Base58", "Key derivation function", "Cryptography",
    ],
    "mining_and_pow": [
        "Proof of work", "Mining pool", "Application-specific integrated circuit",
        "Environmental effects of bitcoin", "Cryptocurrency and crime", "Proof of stake",
        "Proof of space",
    ],
    "ethereum": [
        "Ethereum", "Smart contract", "Solidity", "ERC-20", "Non-fungible token",
        "Decentralized finance", "Ethereum Classic", "The DAO (organization)",
    ],
    "stablecoins": [
        "Stablecoin", "Tether (cryptocurrency)", "USD Coin", "Dai (cryptocurrency)", "TerraUSD",
    ],
    "crypto_security": [
        "Phishing", "Social engineering (security)", "Ponzi scheme", "Confidence trick",
        "Money laundering", "Know your customer", "Multi-factor authentication", "Cold boot attack",
        "Hardware security module", "Pig butchering scam",
    ],
    # ---- finance / economics: the vocabulary a crypto educator keeps reaching for -------------------
    "finance_economics": [
        "Money", "Currency", "Fiat money", "Inflation", "Deflation", "Monetary policy", "Central bank",
        "Federal Reserve", "Interest rate", "Bank", "Commercial bank", "Payment system", "Credit card",
        "Wire transfer", "Remittance", "Foreign exchange market", "Stock", "Stock market", "Bond (finance)",
        "Exchange-traded fund", "Asset", "Volatility (finance)", "Market capitalization", "Liquidity",
        "Supply and demand", "Store of value", "Medium of exchange", "Gold as an investment",
        "Hyperinflation", "Financial regulation", "Securities regulation in the United States",
        "Commodity Futures Trading Commission", "U.S. Securities and Exchange Commission", "Tax",
        "Capital gains tax", "Double-entry bookkeeping", "Ledger", "Audit", "Escrow", "Derivative (finance)",
        "Futures contract", "Short (finance)", "Leverage (finance)", "Arbitrage", "Speculation",
        "Diversification (finance)", "Risk management", "Due diligence", "Fraud",
    ],
    # ---- computing / networking / crypto primitives --------------------------------------------------
    "computing": [
        "Computer network", "Internet", "Internet protocol suite", "Transmission Control Protocol",
        "Client-server model", "Server (computing)", "Database", "Transaction processing",
        "ACID", "Replication (computing)", "Fault tolerance", "Scalability", "Latency (engineering)",
        "Throughput", "Encryption", "Symmetric-key algorithm", "RSA (cryptosystem)", "Diffie-Hellman key exchange",
        "Transport Layer Security", "Random number generation", "Cryptographically secure pseudorandom number generator",
        "Hash table", "Data structure", "Algorithm", "Open-source software", "Version control",
        "Application programming interface", "Virtual machine", "Compiler", "Turing completeness",
        "Zero-knowledge proof", "Homomorphic encryption", "Multi-party computation", "Quantum computing",
        "Post-quantum cryptography", "Denial-of-service attack", "Man-in-the-middle attack", "Sybil attack",
        "Computer security", "Authentication", "Password", "Two-man rule",
    ],
    # ---- general knowledge: the student is asked open questions too ("capital city of Japan?"),
    # and a broader text distribution is what keeps the text pathway from collapsing into one domain.
    "general_knowledge": [
        "Japan", "Tokyo", "China", "Beijing", "India", "New Delhi", "United States", "Washington, D.C.",
        "United Kingdom", "London", "France", "Paris", "Germany", "Berlin", "Brazil", "Russia", "Canada",
        "Australia", "Nigeria", "Egypt", "Capital city", "Country", "Continent", "Earth", "Geography",
        "History", "Science", "Physics", "Chemistry", "Biology", "Mathematics", "Astronomy", "Solar System",
        "Sun", "Moon", "Water", "Electricity", "Energy", "Climate change", "Evolution", "DNA", "Human body",
        "Medicine", "Vaccine", "Language", "English language", "Writing", "Printing press", "Industrial Revolution",
        "Electricity generation", "Artificial intelligence", "Machine learning", "Computer",
        "Telephone", "Television", "Automobile", "Airplane", "Railway", "Agriculture", "Food", "Cooking",
        "Music", "Literature", "Philosophy", "Religion", "Democracy", "Government", "Law", "Education",
        "Sport", "Association football", "Olympic Games", "Time", "Calendar", "Measurement",
    ],
}

# "Mastering Bitcoin" 3rd edition -- AsciiDoc sources. appa_whitepaper is the Bitcoin whitepaper.
MASTERING_BITCOIN_RAW = "https://raw.githubusercontent.com/bitcoinbook/bitcoinbook/develop/"
MASTERING_BITCOIN: dict[str, list[str]] = {
    "crypto_fundamentals":     ["ch01_intro.adoc", "ch02_overview.adoc"],
    "bitcoin":                 ["ch03_bitcoin-core.adoc", "ch10_network.adoc"],
    "wallets_and_keys":        ["ch04_keys.adoc", "ch05_wallets.adoc", "ch08_signatures.adoc"],
    "bitcoin_transactions":    ["ch06_transactions.adoc", "ch07_authorization-authentication.adoc",
                                "ch09_fees.adoc"],
    "blockchain":              ["ch11_blockchain.adoc"],
    "mining_and_pow":          ["ch12_mining.adoc"],
    "crypto_security":         ["ch13_security.adoc"],
    "ethereum":                ["ch14_applications.adoc"],
    "bitcoin_history":         ["appa_whitepaper.adoc"],
}

# ethereum.org developer docs -- Markdown sources (path under public/content/, index.md implied).
ETHEREUM_ORG_RAW = "https://raw.githubusercontent.com/ethereum/ethereum-org-website/dev/public/content/"
ETHEREUM_ORG: dict[str, list[str]] = {
    "ethereum": [
        "developers/docs/intro-to-ethereum", "developers/docs/intro-to-ether",
        "developers/docs/accounts", "developers/docs/transactions", "developers/docs/blocks",
        "developers/docs/gas", "developers/docs/evm", "developers/docs/evm/opcodes",
        "developers/docs/smart-contracts", "developers/docs/smart-contracts/anatomy",
        "developers/docs/smart-contracts/languages", "developers/docs/smart-contracts/compiling",
        "developers/docs/smart-contracts/deploying", "developers/docs/smart-contracts/libraries",
        "developers/docs/standards/tokens/erc-20", "developers/docs/standards/tokens/erc-721",
        "developers/docs/standards/tokens/erc-1155",
        "developers/docs/dapps", "developers/docs/web2-vs-web3", "developers/docs/networks",
        "developers/docs/nodes-and-clients", "developers/docs/nodes-and-clients/node-architecture",
        "developers/docs/scaling", "developers/docs/scaling/optimistic-rollups",
        "developers/docs/scaling/zk-rollups",
        "developers/docs/data-availability",
    ],
    "mining_and_pow": [
        "developers/docs/consensus-mechanisms", "developers/docs/consensus-mechanisms/pos",
        "developers/docs/consensus-mechanisms/pow", "developers/docs/consensus-mechanisms/pos/attack-and-defense",
        "developers/docs/consensus-mechanisms/pos/rewards-and-penalties",
        "developers/docs/consensus-mechanisms/pow/mining",
    ],
    "blockchain": ["developers/docs/data-structures-and-encoding/patricia-merkle-trie",
                   "developers/docs/data-structures-and-encoding/rlp", "developers/docs/bridges",
                   "roadmap/merge"],
    "wallets_and_keys": ["developers/docs/apis/json-rpc"],
    # `security` is the top-level user-facing security page; the /wallets/ page is not a raw index.md
    # (verified 404), so wallet material comes from Wikipedia + Mastering Bitcoin instead.
    "crypto_security": ["security", "developers/docs/smart-contracts/security",
                        "developers/docs/smart-contracts/testing", "developers/docs/mev"],
    "crypto_fundamentals": ["developers/docs/standards/tokens", "developers/docs/oracles"],
}

# Bitcoin Improvement Proposals (MediaWiki) and Ethereum Improvement Proposals (Markdown): the normative
# specs behind everything the teacher explains, each written as prose with a motivation and rationale
# section. Listed through the GitHub contents API, then pulled from raw.githubusercontent.
BIPS_REPO = "bitcoin/bips"
EIPS_REPO = "ethereum/EIPs"
# Most EIPs are short stubs or withdrawn drafts; only files above these sizes carry explanatory prose.
EIP_MIN_BYTES = 6000
BIP_MIN_BYTES = 4000

LICENSES = {
    "wikipedia": "CC BY-SA 4.0",
    "bips": "public domain / CC0, per each proposal's Licence header (BIP-2)",
    "eips": "CC0 1.0",
    "mastering_bitcoin": "CC BY-SA 4.0",
    "ethereum_org": "CC BY 4.0",
}

# ------------------------------------------------------------------------------------------------ fetching


def fetch(url: str, retries: int = 6, pause: float = 1.0) -> str | None:
    """GET with a real User-Agent and 429-aware exponential backoff.

    Wikipedia's API rate-limits anonymous clients and answers 429 once you push past it; a first version of
    this function treated 429 like a hard failure after 3 quick tries and silently lost 43 of 85 articles.
    Retry-After is honoured when present, and the base pause between successful requests is deliberately
    ~1 s -- this fetches fewer than 150 documents in total, so there is no reason to go faster.
    """
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": UA})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read().decode("utf-8", errors="replace")
            time.sleep(pause)                      # be a polite client
            return data
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            if attempt == retries - 1:
                print(f"    HTTP {e.code} (gave up after {retries} tries) {url}")
                return None
            wait = float(e.headers.get("Retry-After") or 0) if e.headers else 0.0
            wait = max(wait, 2.0 * (2 ** attempt))  # 2, 4, 8, 16, 32 s
            if e.code == 429:
                print(f"    HTTP 429 rate-limited, backing off {wait:.0f}s")
            time.sleep(wait)
        except Exception as e:                      # noqa: BLE001 -- network flake, retry
            if attempt == retries - 1:
                print(f"    {type(e).__name__} {url}")
                return None
            time.sleep(2.0 * (2 ** attempt))
    return None


def wikipedia_extract(title: str) -> tuple[str, str] | None:
    """Plain-text extract of one article. One title per request: the API lowers `exlimit` to 1 for whole
    -article extracts, so batching silently returns empty extracts for all but the first title."""
    q = urllib.parse.urlencode({
        "action": "query", "format": "json", "formatversion": "2", "prop": "extracts",
        "explaintext": "1", "exlimit": "1", "redirects": "1", "titles": title,
    })
    raw = fetch(f"https://en.wikipedia.org/w/api.php?{q}")
    if not raw:
        return None
    try:
        pages = json.loads(raw)["query"]["pages"]
    except (KeyError, json.JSONDecodeError):
        return None
    for p in pages:
        if p.get("missing") or not p.get("extract"):
            return None
        return p["title"], p["extract"]
    return None


# ------------------------------------------------------------------------------------------------ cleaning

# Sections that are navigation/citation residue, not prose.
DROP_SECTIONS = {
    "see also", "references", "external links", "further reading", "notes", "citations",
    "bibliography", "sources", "footnotes", "gallery", "explanatory notes", "works cited",
    "general sources", "general and cited references", "additional resources", "further resources",
}

WIKI_SECTION_RE = re.compile(r"^(={2,6})\s*(.+?)\s*\1\s*$", re.M)


def _normalize_unicode(t: str) -> str:
    # ethereum.org Markdown carries HTML entities (`&lt;5GB`, `a &amp; b`) that must not reach the prose.
    t = html.unescape(t)
    t = unicodedata.normalize("NFKC", t)
    t = t.replace("‘", "'").replace("’", "'").replace("“", '"').replace("”", '"')
    t = t.replace("–", "-").replace("—", " - ").replace(" ", " ")
    t = re.sub(r"[​-‏‪-‮﻿]", "", t)       # zero-width / bidi marks
    return t


def split_wikipedia(title: str, extract: str) -> list[tuple[str, str]]:
    """-> [(section_path, prose)] with boilerplate sections removed. The lead (before the first heading)
    becomes the section "Introduction"."""
    text = _normalize_unicode(extract)
    out: list[tuple[str, str]] = []
    matches = list(WIKI_SECTION_RE.finditer(text))
    lead = text[: matches[0].start()] if matches else text
    if lead.strip():
        out.append((f"{title} - Introduction", lead.strip()))
    for i, m in enumerate(matches):
        name = m.group(2).strip()
        body = text[m.end(): matches[i + 1].start() if i + 1 < len(matches) else len(text)]
        if name.lower().strip(" :") in DROP_SECTIONS:
            continue
        if body.strip():
            out.append((f"{title} - {name}", body.strip()))
    return out


ADOC_BLOCK_DELIM = re.compile(r"^(-{4,}|={4,}|\*{4,}|\.{4,}|_{4,}|\+{4,}|/{4,})\s*$")
ADOC_SECTION_RE = re.compile(r"^(={1,6})\s+(.+?)\s*$", re.M)


def clean_asciidoc(raw: str) -> str:
    """Strip AsciiDoc markup, code/table/admonition blocks and index entries, keeping prose.

    "Mastering Bitcoin" uses `(((index, terms)))` very heavily and embeds large code/console listings; both
    would otherwise dominate the text and teach the model nothing explanatory.
    """
    raw = _normalize_unicode(raw)
    raw = re.sub(r"\(\(\(.*?\)\)\)", "", raw, flags=re.S)           # index entries
    # `pass:[...]` passthrough macros embed raw HTML (the whitepaper's byline carries an <a href=...><em>
    # link). Keep the inner words, drop the macro and the tags.
    raw = re.sub(r"pass:[a-z]*\[(.*?)\]", r" \1 ", raw, flags=re.S)
    raw = re.sub(r"</?[a-zA-Z][^>]*>", "", raw)                     # residual inline HTML
    raw = re.sub(r"^\[\[.*?\]\]\s*$", "", raw, flags=re.M)          # anchors
    raw = re.sub(r"^:[\w!-]+:.*$", "", raw, flags=re.M)             # attribute entries
    raw = re.sub(r"^(include|image|video|ifdef|ifndef|endif)::.*$", "", raw, flags=re.M)
    raw = re.sub(r"footnote:\[(.*?)\]", r" (\1)", raw, flags=re.S)
    raw = re.sub(r"<<[^>]*?,\s*([^>]*?)>>", r"\1", raw)             # xref with label
    raw = re.sub(r"<<([^>,]*?)>>", "", raw)                         # bare xref
    raw = re.sub(r"(?:https?|link):[^\s\[]*\[(.*?)\]", r"\1", raw, flags=re.S)
    raw = re.sub(r"\bhttps?://\S+", "", raw)

    lines, keep, in_block, in_table = raw.split("\n"), [], False, False
    skip_next_block = False
    for ln in lines:
        s = ln.rstrip()
        if s.startswith("|==="):
            in_table = not in_table
            continue
        if in_table:
            continue
        if ADOC_BLOCK_DELIM.match(s):
            if in_block:
                in_block = False
            else:
                in_block = True if skip_next_block else in_block
                if not skip_next_block:
                    # an undecorated delimited block: treat as literal/code and skip it too
                    in_block = True
            skip_next_block = False
            continue
        if in_block:
            continue
        if re.match(r"^\[(source|listing|literal|quote|verse|NOTE|TIP|WARNING|CAUTION|IMPORTANT|"
                    r"role|cols|options|width|frame|grid|caption|id)[,\]=]", s, re.I):
            skip_next_block = True
            continue
        if s.startswith("[") and s.endswith("]"):
            continue
        keep.append(s)
    text = "\n".join(keep)

    text = re.sub(r"`{1,3}([^`]*?)`{1,3}", r"\1", text)             # inline code
    # `+literal+` -> literal, but only short spans with no internal whitespace: a greedy `\+(...)\+` turned
    # the arithmetic "consume +t+ + +k+ + 2 items" into "consume t  k  2 items", losing the operators.
    text = re.sub(r"(?<!\+)\+([^\s+][^+\n]{0,40}?)\+(?!\+)", r"\1", text)
    text = re.sub(r"~([^~\s][^~\n]{0,30}?)~", r"\1", text)          # subscript:   F~sig~  -> Fsig
    text = re.sub(r"\^([^\^\s][^\^\n]{0,30}?)\^", r"^\1", text)     # superscript: 2^256^  -> 2^256
    text = re.sub(r"\*{1,2}([^*\n]+?)\*{1,2}", r"\1", text)         # bold
    text = re.sub(r"(?<![\w_])_([^_\n]+?)_(?![\w_])", r"\1", text)  # italic
    text = re.sub(r"^[\s]*[*.]{1,5}\s+", "", text, flags=re.M)      # list bullets
    text = re.sub(r"^\s*(NOTE|TIP|WARNING|CAUTION|IMPORTANT):\s*", "", text, flags=re.M)
    return text


def split_asciidoc(title: str, raw: str) -> list[tuple[str, str]]:
    text = clean_asciidoc(raw)
    out: list[tuple[str, str]] = []
    matches = list(ADOC_SECTION_RE.finditer(text))
    if not matches:
        return [(title, text.strip())] if text.strip() else []
    for i, m in enumerate(matches):
        name = m.group(2).strip()
        body = text[m.end(): matches[i + 1].start() if i + 1 < len(matches) else len(text)]
        if name.lower().strip(" :") in DROP_SECTIONS:
            continue
        if body.strip():
            out.append((f"{title} - {name}", body.strip()))
    return out


MW_SECTION_RE = re.compile(r"^\s*(={2,6})\s*(.+?)\s*\1\s*$", re.M)


def clean_mediawiki(raw: str) -> str:
    """Strip MediaWiki markup from a BIP: code/pre blocks, tables, links and inline emphasis.

    BIPs are specs, so they carry large `<pre>` reference-implementation blocks and wire-format tables that
    are mostly symbols -- exactly what `_quality_ok`'s digit/letter-ratio filters would reject downstream,
    but cheaper to remove here than to let them split good prose into unusable fragments.
    """
    raw = _normalize_unicode(raw)
    raw = re.sub(r"<(pre|source|syntaxhighlight|code|nowiki)[^>]*>.*?</\1>", "", raw, flags=re.S | re.I)
    raw = re.sub(r"\{\|.*?\|\}", "", raw, flags=re.S)                      # tables
    raw = re.sub(r"<!--.*?-->", "", raw, flags=re.S)
    raw = re.sub(r"</?[a-zA-Z][^>]*>", "", raw)                            # stray html
    raw = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", raw)            # [[target|label]] -> label
    raw = re.sub(r"\[(?:https?|ftp)://\S+\s+([^\]]*)\]", r"\1", raw)       # [url label] -> label
    raw = re.sub(r"\[(?:https?|ftp)://\S+\]", "", raw)
    raw = re.sub(r"\bhttps?://\S+", "", raw)
    raw = re.sub(r"'{2,5}", "", raw)                                       # ''italic'' / '''bold'''
    raw = re.sub(r"^\s*[*#:;]+\s*", "", raw, flags=re.M)                   # list / indent markers
    raw = re.sub(r"^\s*\|.*$", "", raw, flags=re.M)                        # leftover table rows
    return raw


def split_mediawiki(title: str, raw: str) -> list[tuple[str, str]]:
    text = clean_mediawiki(raw)
    out: list[tuple[str, str]] = []
    matches = list(MW_SECTION_RE.finditer(text))
    lead = text[: matches[0].start()] if matches else text
    if lead.strip():
        out.append((f"{title} - Introduction", lead.strip()))
    for i, m in enumerate(matches):
        name = m.group(2).strip()
        body = text[m.end(): matches[i + 1].start() if i + 1 < len(matches) else len(text)]
        if name.lower().strip(" :") in DROP_SECTIONS:
            continue
        if body.strip():
            out.append((f"{title} - {name}", body.strip()))
    return out


def repo_tarball_members(cache: Path, repo: str, subdir: str, suffixes: tuple[str, ...],
                         min_bytes: int, allow_fetch: bool) -> list[tuple[str, str]]:
    """-> [(filename, text)] for files under `subdir` in a GitHub repo, via its source tarball.

    Deliberately NOT the contents API: unauthenticated GitHub API calls are capped at 60/hour, so listing
    a directory and then pulling N raw files both rate-limits (HTTP 403, which silently yields zero
    documents) and costs N+1 requests. `codeload.github.com/<repo>/tar.gz/refs/heads/<branch>` is one
    request, is not part of the API quota, and is cached here like every other download.
    """
    import io
    import tarfile

    tar_path = cache / (re.sub(r"[^A-Za-z0-9._-]+", "_", repo) + ".tar.gz")
    if not tar_path.exists():
        if not allow_fetch:
            return []
        for branch in ("master", "main"):
            url = f"https://codeload.github.com/{repo}/tar.gz/refs/heads/{branch}"
            try:
                req = urllib.request.Request(url, headers={"User-Agent": UA})
                with urllib.request.urlopen(req, timeout=300) as r:
                    data = r.read()
                tar_path.write_bytes(data)
                print(f"    downloaded {repo}@{branch} ({len(data) / 1e6:.1f} MB)")
                break
            except urllib.error.HTTPError as e:
                if e.code != 404:
                    print(f"    HTTP {e.code} for {url}")
            except Exception as e:                                   # noqa: BLE001
                print(f"    {type(e).__name__} for {url}")
        if not tar_path.exists():
            return []

    out: list[tuple[str, str]] = []
    with tarfile.open(tar_path, "r:gz") as tf:
        for m in tf.getmembers():
            if not m.isfile() or m.size < min_bytes:
                continue
            # tarball paths are "<repo>-<branch>/<path>"; compare on the path inside the repo
            inner = m.name.split("/", 1)[1] if "/" in m.name else m.name
            if subdir and not inner.startswith(subdir.rstrip("/") + "/"):
                continue
            name = Path(inner).name
            if not name.endswith(suffixes):
                continue
            fh = tf.extractfile(m)
            if fh is None:
                continue
            out.append((name, fh.read().decode("utf-8", errors="replace")))
    return sorted(out)


MD_SECTION_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*$", re.M)


def clean_markdown(raw: str) -> str:
    """Strip YAML frontmatter, code fences, JSX/MDX components, links and tables from ethereum.org docs."""
    raw = _normalize_unicode(raw)
    raw = re.sub(r"\A---\n.*?\n---\n", "", raw, flags=re.S)         # frontmatter
    raw = re.sub(r"```.*?```", "", raw, flags=re.S)                 # fenced code
    raw = re.sub(r"~~~.*?~~~", "", raw, flags=re.S)
    raw = re.sub(r"\{/\*.*?\*/\}", "", raw, flags=re.S)             # MDX comments
    raw = re.sub(r"<!--.*?-->", "", raw, flags=re.S)
    raw = re.sub(r"^import\s+.*$", "", raw, flags=re.M)
    raw = re.sub(r"^export\s+.*$", "", raw, flags=re.M)
    raw = re.sub(r"<[A-Z][\w.]*(?:\s[^>]*?)?/>", "", raw, flags=re.S)        # self-closing components
    raw = re.sub(r"</?[A-Z][\w.]*(?:\s[^>]*?)?>", "", raw, flags=re.S)       # component open/close
    raw = re.sub(r"</?[a-z][\w]*(?:\s[^>]*?)?>", "", raw)                    # stray html
    raw = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", raw)                           # images
    raw = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", raw)                       # links -> label
    raw = re.sub(r"^\s*\|.*\|\s*$", "", raw, flags=re.M)                     # table rows
    raw = re.sub(r"`{1,3}([^`]*?)`{1,3}", r"\1", raw)
    raw = re.sub(r"\*{1,2}([^*\n]+?)\*{1,2}", r"\1", raw)
    raw = re.sub(r"^\s*[-*+]\s+", "", raw, flags=re.M)
    raw = re.sub(r"^\s*\d+\.\s+", "", raw, flags=re.M)
    raw = re.sub(r"^\s*>\s?", "", raw, flags=re.M)
    raw = re.sub(r"\s*\{#[\w-]+\}\s*$", "", raw, flags=re.M)        # heading anchors: "## Gas {#gas}"
    return raw


def split_markdown(title: str, raw: str) -> list[tuple[str, str]]:
    text = clean_markdown(raw)
    out: list[tuple[str, str]] = []
    matches = list(MD_SECTION_RE.finditer(text))
    lead = text[: matches[0].start()] if matches else text
    if lead.strip():
        out.append((f"{title} - Introduction", lead.strip()))
    for i, m in enumerate(matches):
        name = m.group(2).strip()
        body = text[m.end(): matches[i + 1].start() if i + 1 < len(matches) else len(text)]
        if name.lower().strip(" :") in DROP_SECTIONS:
            continue
        if body.strip():
            out.append((f"{title} - {name}", body.strip()))
    return out


# --------------------------------------------------------------------------------------- quality + chunking

MIN_CHARS, MAX_CHARS, MIN_WORDS = 400, 6000, 60


def tidy(text: str) -> str:
    lines = []
    for ln in text.split("\n"):
        s = re.sub(r"[ \t]+", " ", ln).strip()
        lines.append(s)
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _quality_ok(text: str) -> tuple[bool, str]:
    """Reject anything that is not explanatory prose. The thresholds target the specific junk this corpus
    must not contain: price/market tables (digit-heavy), stat dumps and listicles (short lines, few verbs),
    and citation residue."""
    words = text.split()
    if len(words) < MIN_WORDS:
        return False, "too_few_words"
    letters = sum(c.isalpha() for c in text)
    digits = sum(c.isdigit() for c in text)
    if letters == 0 or digits / max(len(text), 1) > 0.12:
        return False, "digit_heavy"           # price series, OHLC tables, date dumps
    if letters / max(len(text), 1) < 0.60:
        return False, "low_letter_ratio"      # markup/symbol residue
    lines = [l for l in text.split("\n") if l.strip()]
    if lines and sum(len(l) for l in lines) / len(lines) < 40:
        return False, "line_fragments"        # table/list residue rather than sentences
    sentences = re.split(r"[.!?]\s", text)
    if len(sentences) < 3:
        return False, "too_few_sentences"
    if len(words) / max(len(set(w.lower() for w in words)), 1) > 6.0:
        return False, "repetitive"
    low = text.lower()
    if low.count("retrieved ") + low.count("archived from the original") >= 3:
        return False, "citation_residue"
    return True, "ok"


def chunk(text: str) -> list[str]:
    """Split an over-long section on paragraph boundaries into MAX_CHARS-ish documents."""
    text = tidy(text)
    if len(text) <= MAX_CHARS:
        return [text]
    out, cur = [], ""
    for para in text.split("\n\n"):
        if len(cur) + len(para) + 2 > MAX_CHARS and cur:
            out.append(cur.strip())
            cur = para
        else:
            cur = f"{cur}\n\n{para}" if cur else para
    if cur.strip():
        out.append(cur.strip())
    return out


# ------------------------------------------------------------------------------------------------- dedup

def _norm_for_hash(text: str) -> str:
    return re.sub(r"[^a-z0-9 ]+", " ", text.lower())


class Dedup:
    """Exact dedup on normalized text, plus near-dup detection via shared word 8-grams.

    Wikipedia articles repeat each other's lead paragraphs and "Mastering Bitcoin" restates the whitepaper,
    so near-duplicates are common and would otherwise be seen many times per epoch.
    """

    SHINGLE, SAMPLES, THRESHOLD = 8, 24, 0.5

    def __init__(self):
        self.exact: set[str] = set()
        self.index: dict[int, list[int]] = {}
        self.doc_shingles: list[set[int]] = []

    def _shingles(self, text: str) -> list[int]:
        w = _norm_for_hash(text).split()
        grams = [" ".join(w[i:i + self.SHINGLE]) for i in range(max(0, len(w) - self.SHINGLE + 1))]
        hs = sorted({int(hashlib.blake2b(g.encode(), digest_size=8).hexdigest(), 16) for g in grams})
        if len(hs) <= self.SAMPLES:
            return hs
        step = len(hs) / self.SAMPLES
        return [hs[int(i * step)] for i in range(self.SAMPLES)]

    def add(self, text: str) -> tuple[bool, str]:
        h = hashlib.blake2b(_norm_for_hash(text).encode(), digest_size=16).hexdigest()
        if h in self.exact:
            return False, "exact_duplicate"
        sh = self._shingles(text)
        if sh:
            hits: dict[int, int] = {}
            for s in sh:
                for d in self.index.get(s, ()):
                    hits[d] = hits.get(d, 0) + 1
            for d, n in hits.items():
                union = len(self.doc_shingles[d] | set(sh))
                if union and n / union >= self.THRESHOLD:
                    return False, "near_duplicate"
        self.exact.add(h)
        idx = len(self.doc_shingles)
        self.doc_shingles.append(set(sh))
        for s in sh:
            self.index.setdefault(s, []).append(idx)
        return True, "ok"


# --------------------------------------------------------------------------------------------------- main

def default_out() -> str:
    return "/workspace/ppx_text_corpus" if Path("/workspace").is_dir() else "./ppx_text_corpus"


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--out", default=default_out(), help=f"corpus directory (default: {default_out()})")
    ap.add_argument("--cache", default=None,
                    help="raw-download cache (default: <out>-cache, a SIBLING of the corpus). It must stay "
                         "outside the corpus directory: distill/data/text_dataset.py walks --text-data "
                         "recursively for .txt/.md/.jsonl/.json, so a cache inside it would be scanned on "
                         "every run (91 MediaWiki dumps, ~1.6 MB of JSON parsed for nothing).")
    ap.add_argument("--no-fetch", action="store_true", help="use only what is already in the cache")
    ap.add_argument("--sources", nargs="*",
                    default=["wikipedia", "mastering_bitcoin", "ethereum_org", "bips", "eips"],
                    choices=["wikipedia", "mastering_bitcoin", "ethereum_org", "bips", "eips"])
    args = ap.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    cache = Path(args.cache) if args.cache else out.parent / (out.name + "-cache")
    cache.mkdir(parents=True, exist_ok=True)
    if cache.resolve() == out.resolve() or out.resolve() in cache.resolve().parents:
        raise SystemExit(f"--cache {cache} is inside the corpus directory {out}; the loader would scan it")

    def cached(key: str, url: str) -> str | None:
        f = cache / (re.sub(r"[^A-Za-z0-9._-]+", "_", key)[:150] + ".raw")
        if f.exists():
            return f.read_text(encoding="utf-8", errors="replace")
        if args.no_fetch:
            return None
        body = fetch(url)
        if body:
            f.write_text(body, encoding="utf-8")
        return body

    # ---- 1. download + split into (source, topic, title, section_text) ------------------------------------
    raw_sections: list[tuple[str, str, str, str]] = []
    fetched = {k: 0 for k in LICENSES}
    missing: list[str] = []

    if "wikipedia" in args.sources:
        print("fetching Wikipedia ...")
        for topic, titles in WIKIPEDIA.items():
            for t in titles:
                f = cache / (re.sub(r"[^A-Za-z0-9._-]+", "_", "wiki_" + t)[:150] + ".json")
                if f.exists():
                    got = json.loads(f.read_text(encoding="utf-8"))
                elif args.no_fetch:
                    missing.append(f"wikipedia:{t}")
                    continue
                else:
                    res = wikipedia_extract(t)
                    if res is None:
                        missing.append(f"wikipedia:{t}")
                        print(f"    MISSING {t}")
                        continue
                    got = {"title": res[0], "extract": res[1]}
                    f.write_text(json.dumps(got), encoding="utf-8")
                fetched["wikipedia"] += 1
                for name, body in split_wikipedia(got["title"], got["extract"]):
                    raw_sections.append(("wikipedia", topic, name, body))

    if "mastering_bitcoin" in args.sources:
        print("fetching Mastering Bitcoin ...")
        for topic, files in MASTERING_BITCOIN.items():
            for fn in files:
                body = cached("mb_" + fn, MASTERING_BITCOIN_RAW + fn)
                if not body:
                    missing.append(f"mastering_bitcoin:{fn}")
                    continue
                fetched["mastering_bitcoin"] += 1
                stem = fn.replace(".adoc", "").replace("_", " ")
                for name, sec in split_asciidoc(f"Mastering Bitcoin: {stem}", body):
                    raw_sections.append(("mastering_bitcoin", topic, name, sec))

    if "ethereum_org" in args.sources:
        print("fetching ethereum.org docs ...")
        seen_paths: set[str] = set()
        for topic, paths in ETHEREUM_ORG.items():
            for p in paths:
                key = p.strip("/")
                if key in seen_paths:
                    continue
                seen_paths.add(key)
                body = cached("eth_" + key, f"{ETHEREUM_ORG_RAW}{key}/index.md")
                if not body:
                    missing.append(f"ethereum_org:{key}")
                    continue
                fetched["ethereum_org"] += 1
                for name, sec in split_markdown(f"ethereum.org: {key}", body):
                    raw_sections.append(("ethereum_org", topic, name, sec))

    if "bips" in args.sources:
        print("fetching Bitcoin Improvement Proposals ...")
        members = repo_tarball_members(cache, BIPS_REPO, "", (".mediawiki", ".md"),
                                       BIP_MIN_BYTES, not args.no_fetch)
        print(f"    {len(members)} BIPs above {BIP_MIN_BYTES} bytes")
        for fn, body in members:
            fetched["bips"] += 1
            stem = fn.rsplit(".", 1)[0]
            split = split_mediawiki if fn.endswith(".mediawiki") else split_markdown
            for name, sec in split(f"BIP: {stem}", body):
                raw_sections.append(("bips", "bitcoin", name, sec))

    if "eips" in args.sources:
        print("fetching Ethereum Improvement Proposals ...")
        members = repo_tarball_members(cache, EIPS_REPO, "EIPS", (".md",),
                                       EIP_MIN_BYTES, not args.no_fetch)
        print(f"    {len(members)} EIPs above {EIP_MIN_BYTES} bytes")
        for fn, body in members:
            fetched["eips"] += 1
            stem = fn.rsplit(".", 1)[0]
            for name, sec in split_markdown(f"EIP: {stem}", body):
                raw_sections.append(("eips", "ethereum", name, sec))

    print(f"\nsections before filtering: {len(raw_sections)}")

    # ---- 2. chunk, quality filter, dedup -----------------------------------------------------------------
    dedup = Dedup()
    rejected: dict[str, int] = {}
    docs: list[dict] = []
    for source, topic, title, body in raw_sections:
        for piece in chunk(body):
            if len(piece) < MIN_CHARS:
                rejected["too_short"] = rejected.get("too_short", 0) + 1
                continue
            ok, why = _quality_ok(piece)
            if not ok:
                rejected[why] = rejected.get(why, 0) + 1
                continue
            ok, why = dedup.add(piece)
            if not ok:
                rejected[why] = rejected.get(why, 0) + 1
                continue
            docs.append({"text": piece, "title": title, "topic": topic,
                         "source": source, "license": LICENSES[source]})

    # ---- 3. write JSONL shards, one per source ----------------------------------------------------------
    shard_name = {"wikipedia": "01_wikipedia.jsonl",
                  "mastering_bitcoin": "02_mastering_bitcoin.jsonl",
                  "ethereum_org": "03_ethereum_org_docs.jsonl",
                  "bips": "04_bitcoin_bips.jsonl",
                  "eips": "05_ethereum_eips.jsonl"}
    written: dict[str, dict] = {}
    for source, fname in shard_name.items():
        rows = [d for d in docs if d["source"] == source]
        if not rows:
            continue
        path = out / fname
        with open(path, "w", encoding="utf-8", newline="\n") as fh:
            for d in rows:
                fh.write(json.dumps(d, ensure_ascii=False) + "\n")
        written[fname] = {"documents": len(rows), "characters": sum(len(d["text"]) for d in rows),
                          "bytes": path.stat().st_size, "license": LICENSES[source]}

    # Remove every shard this run did not just write. Two ways a stale shard survives otherwise: a shard
    # gets RENAMED (01_wikipedia_crypto -> 01_wikipedia), or a source yields zero rows this run (its
    # listing 403'd) and `continue` skips rewriting it. Both leave old documents in the corpus, and the
    # self-check below caught each of them in turn (+894, then +2754).
    for stale in sorted(out.glob("*.jsonl")):
        if stale.name not in written:
            print(f"  removing stale shard: {stale.name}")
            stale.unlink()

    by_topic: dict[str, dict] = {}
    for d in docs:
        e = by_topic.setdefault(d["topic"], {"documents": 0, "characters": 0})
        e["documents"] += 1
        e["characters"] += len(d["text"])

    total_chars = sum(len(d["text"]) for d in docs)
    manifest = {
        "built": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "builder": "tools/build_text_corpus.py",
        "format": "JSONL, one document per line, 'text' key -- read by distill/data/text_dataset.py",
        "sources": {s: {"files_or_articles": n, "license": LICENSES[s]} for s, n in fetched.items() if n},
        "totals": {"documents": len(docs), "characters": total_chars,
                   "words": sum(len(d["text"].split()) for d in docs)},
        "files": written,
        "by_topic": dict(sorted(by_topic.items())),
        "rejected": dict(sorted(rejected.items(), key=lambda kv: -kv[1])),
        "missing_sources": missing,
        "filters": {"min_chars": MIN_CHARS, "max_chars": MAX_CHARS, "min_words": MIN_WORDS,
                    "max_digit_ratio": 0.12, "min_letter_ratio": 0.60, "min_avg_line_chars": 40,
                    "near_dup_shingle": Dedup.SHINGLE, "near_dup_threshold": Dedup.THRESHOLD},
    }
    (out / "corpus_manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")

    notice = f"""# Text corpus for PersonaPlex student text-only distillation

Built by `tools/build_text_corpus.py` on {manifest['built']}.
Format: JSONL, one document per line with a `"text"` key -- exactly what
`distill/data/text_dataset.py` reads. Extra keys (`title`, `topic`, `source`, `license`) are
provenance only and are ignored by the loader.

Totals: {len(docs):,} documents, {total_chars:,} characters.

## Sources and licenses

1. **English Wikipedia** -- Creative Commons Attribution-ShareAlike 4.0 (CC BY-SA 4.0).
   Retrieved via the MediaWiki `extracts` API. Articles are listed in `corpus_manifest.json`
   and in each document's `title` field. https://en.wikipedia.org
2. **Mastering Bitcoin, 3rd edition** -- Andreas M. Antonopoulos and David A. Harding,
   Creative Commons Attribution-ShareAlike 4.0 (CC BY-SA 4.0).
   https://github.com/bitcoinbook/bitcoinbook  (Appendix A is the Bitcoin whitepaper.)
3. **ethereum.org developer documentation** -- Creative Commons Attribution 4.0 (CC BY 4.0).
   https://github.com/ethereum/ethereum-org-website

All three are used unmodified in substance; the builder strips markup, code listings, tables and
citation sections, and splits articles into section-sized documents. CC BY-SA requires that
derivative distributions carry the same license and attribution -- keep this file alongside the
corpus, and note that a model trained on it is generally not considered a derivative work of the
text, but redistributing the corpus itself is.

This file is named `NOTICE` with no extension on purpose: `distill/data/text_dataset.py` treats
`.md` and `.txt` files as training documents, so a `NOTICE.md` here would be trained on.

## Deliberately excluded

Price and market data, OHLC series, charts, social-media posts, forum scrapes, SEO/affiliate
content and noisy scraped crypto datasets. The quality filter in the builder rejects digit-heavy
text, line fragments, listicles and citation residue; see `corpus_manifest.json -> rejected`.
"""
    # Deliberately NO .md/.txt extension: `TEXT_SUFFIXES` in distill/data/text_dataset.py includes ".md",
    # so a `NOTICE.md` sitting beside the shards is read as a TRAINING DOCUMENT -- the attribution text would
    # end up in the corpus. (`corpus_manifest.json` is safe: it is a .json dict with no recognized text key,
    # so `_iter_documents` yields nothing from it. The self-check below enforces both.)
    (out / "NOTICE").write_text(notice, encoding="utf-8")
    for stale in ("NOTICE.md", "NOTICE.txt"):
        if (out / stale).exists():
            (out / stale).unlink()

    # ---- 4. self-check against the REAL loader ----------------------------------------------------------
    sys.path[:0] = [str(Path(__file__).resolve().parent.parent),
                    str(Path(__file__).resolve().parent.parent / "moshi")]
    from distill.data.text_dataset import _iter_documents
    seen = list(_iter_documents([str(out)]))
    if len(seen) != len(docs):
        extra = len(seen) - len(docs)
        raise SystemExit(
            f"SELF-CHECK FAILED: distill/data/text_dataset.py reads {len(seen)} documents from {out} but "
            f"{len(docs)} were written ({extra:+d}). Something in the corpus directory that is not a data "
            f"shard is being picked up as training text -- check for stray .txt/.md/.jsonl/.json files.")
    print(f"\nself-check: the real loader reads exactly {len(seen):,} documents from {out} -- matches.")

    print(f"\n{'=' * 78}\ncorpus written to {out}")
    print(f"  documents : {len(docs):,}")
    print(f"  characters: {total_chars:,}")
    for f, s in written.items():
        print(f"  {f:34s} {s['documents']:>6,} docs  {s['characters']:>10,} chars")
    print(f"\nrejected: {json.dumps(manifest['rejected'])}")
    if missing:
        print(f"missing ({len(missing)}): {', '.join(missing[:10])}{' ...' if len(missing) > 10 else ''}")
    print("manifest -> corpus_manifest.json ; attribution -> NOTICE (no extension, on purpose)")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile {REPO_DIR}/tools/verify_text_corpus.py
#!/usr/bin/env python3
# SPDX-License-Identifier: MIT
"""Verify a text corpus against the real text-only distillation loader, and report its size.

    python tools/verify_text_corpus.py                                  # auto-locate the corpus
    python tools/verify_text_corpus.py --corpus /workspace/ppx_text_corpus
    python tools/verify_text_corpus.py --tokenizer /path/tokenizer_spm_32k_3.model   # exact token count

Every check here goes through `distill.data.text_dataset` itself -- the module `distill/train.py` imports --
rather than re-implementing the format, so a pass means the trainer will read this corpus.

Token count: with `--tokenizer`, this is the EXACT count the trainer will see (it calls the same
`build_token_cache`). Without it, a public 32,000-piece SentencePiece model (google-t5/t5-small's
`spiece.model`, the same vocabulary size as PersonaPlex's `tokenizer_spm_32k_3.model`) is used as a reference
and the number is reported as an estimate. The teacher tokenizer is a gated download, so the exact count is
only available on the pod.
"""

from __future__ import annotations

import argparse
import json
import sys
import tempfile
from pathlib import Path

REPO = Path(__file__).resolve().parent.parent
for p in (REPO, REPO / "moshi"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np                                                    # noqa: E402
from distill.data.text_dataset import (                               # noqa: E402
    TEXT_CARD, SEPARATOR, TEXT_SUFFIXES, _iter_documents, build_token_cache, TextTokenStream,
)


def find_corpus() -> str:
    for c in ("/workspace/ppx_text_corpus", str(REPO / "ppx_text_corpus")):
        if Path(c).is_dir():
            return c
    raise SystemExit("No corpus found. Pass --corpus, or run tools/build_text_corpus.py first.")


def reference_tokenizer() -> tuple[str, str]:
    from huggingface_hub import hf_hub_download
    path = hf_hub_download("google-t5/t5-small", "spiece.model")
    return path, "reference (google-t5/t5-small spiece.model, 32,000 pieces)"


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--corpus", default=None)
    ap.add_argument("--tokenizer", default=None, help="teacher tokenizer_spm_32k_3.model for an exact count")
    ap.add_argument("--seq-len", type=int, default=512, help="must match --text-seq-len (default 512)")
    args = ap.parse_args()

    corpus = args.corpus or find_corpus()
    print(f"corpus: {corpus}\n")

    # ---- 1. files the loader will discover ---------------------------------------------------------------
    files = sorted(f for f in Path(corpus).rglob("*") if f.suffix.lower() in TEXT_SUFFIXES)
    print("1. FILES THE LOADER DISCOVERS")
    if not files:
        raise SystemExit(f"   none -- the loader needs {'/'.join(TEXT_SUFFIXES)} files under {corpus}")
    for f in files:
        print(f"   {f.stat().st_size / 1e6:>8.2f} MB  {f.relative_to(corpus)}")
    non_shard = [f for f in files if f.suffix.lower() != ".jsonl"]
    if non_shard:
        print(f"   NOTE: {len(non_shard)} discovered file(s) are not .jsonl shards; step 2 shows whether "
              f"they contribute documents: {[f.name for f in non_shard]}")

    # ---- 2. documents, through _iter_documents ------------------------------------------------------------
    docs = list(_iter_documents([corpus]))
    chars = sum(len(d) for d in docs)
    words = sum(len(d.split()) for d in docs)
    print(f"\n2. DOCUMENTS (via distill.data.text_dataset._iter_documents)")
    print(f"   documents : {len(docs):,}")
    print(f"   characters: {chars:,}")
    print(f"   words     : {words:,}")
    lens = sorted(len(d) for d in docs)
    if lens:
        q = lambda p: lens[min(len(lens) - 1, int(p * len(lens)))]      # noqa: E731
        print(f"   doc chars : min {lens[0]}  p25 {q(.25)}  median {q(.5)}  p75 {q(.75)}  max {lens[-1]}")
    empty = sum(1 for d in docs if not d.strip())
    print(f"   empty docs: {empty} (skipped by build_token_cache)")

    # ---- 3. tokenize with the SAME function the trainer calls --------------------------------------------
    if args.tokenizer:
        tok, kind = args.tokenizer, "EXACT (teacher tokenizer_spm_32k_3.model)"
    else:
        tok, kind = reference_tokenizer()
    exact = bool(args.tokenizer)
    print(f"\n3. TOKENIZATION -- {kind}")
    td = tempfile.mkdtemp(prefix="ppx_verify_tok_")
    try:
        cache = str(Path(td) / "verify_tokens.npy")
        arr = build_token_cache([corpus], tok, cache)
        n_tok = int(arr.shape[0])
        id3 = int((np.asarray(arr) == SEPARATOR).sum())
        print(f"   tokens        : {n_tok:,}" + ("" if exact else "   (ESTIMATE -- see the note below)"))
        print(f"   dtype         : {arr.dtype} (uint16 required; text_card={TEXT_CARD} fits)")
        print(f"   max token id  : {int(arr.max()):,} (must be < {TEXT_CARD})")
        if exact:
            print(f"   separators    : {id3:,} occurrences of id {SEPARATOR} "
                  f"(>= {len(docs):,}, one appended per document)")
        else:
            # With a reference tokenizer, id 3 is an ordinary piece of ITS vocabulary, not PersonaPlex's
            # padding id, so counting it says nothing about document boundaries.
            print(f"   separators    : not meaningful with a reference tokenizer "
                  f"(id {SEPARATOR} is a normal piece there); {len(docs):,} will be appended with the "
                  f"teacher tokenizer")
        print(f"   chars / token : {chars / max(n_tok, 1):.2f}")
        assert arr.dtype == np.uint16, arr.dtype
        assert int(arr.max()) < TEXT_CARD

        # ---- 4. the stream the trainer actually iterates -------------------------------------------------
        print(f"\n4. TextTokenStream(seq_len={args.seq_len})")
        stream = TextTokenStream(arr, args.seq_len, seed=1234)
        b1 = stream.batch(step=7, rank=0, world=1, micro=2, accum=4)
        b2 = stream.batch(step=7, rank=0, world=1, micro=2, accum=4)
        b3 = stream.batch(step=8, rank=0, world=1, micro=2, accum=4)
        print(f"   windows       : {len(stream):,}")
        print(f"   batch shape   : {b1.shape}  dtype {b1.dtype}")
        assert b1.shape == (8, args.seq_len + 1) and b1.dtype == np.int64
        assert np.array_equal(b1, b2), "not resume-deterministic"
        assert not np.array_equal(b1, b3), "different steps gave the same batch"
        r0 = stream.batch(step=0, rank=0, world=2, micro=1, accum=2)
        r1 = stream.batch(step=0, rank=1, world=2, micro=1, accum=2)
        assert not np.array_equal(r0, r1), "ranks not disjoint"
        print("   determinism   : OK (same step -> same batch; ranks disjoint)")
        del arr, stream                      # release the memmap before removing the temp dir (Windows)
    finally:
        import shutil
        shutil.rmtree(td, ignore_errors=True)

    # ---- 5. topic coverage, from the provenance fields ---------------------------------------------------
    man = Path(corpus) / "corpus_manifest.json"
    if man.exists():
        m = json.loads(man.read_text(encoding="utf-8"))
        print("\n5. TOPIC COVERAGE (from corpus_manifest.json)")
        for topic, s in m.get("by_topic", {}).items():
            print(f"   {topic:24s} {s['documents']:>5,} docs  {s['characters']:>9,} chars")

    print("\n" + "=" * 78)
    print("COMPATIBLE: the corpus is read by distill/data/text_dataset.py as-is.")
    print(f"Run with:  --text-data {corpus} --tokenizer <teacher tokenizer_spm_32k_3.model> "
          f"--text-seq-len {args.seq_len}")
    if not args.tokenizer:
        print("\nNOTE: the token count above is a reference-tokenizer ESTIMATE. The exact count is printed by "
              "the notebook's Section 7b cell (and this script with --tokenizer) on the pod, where the gated "
              "teacher tokenizer is available.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile {REPO_DIR}/tools/export_checkpoint.py
#!/usr/bin/env python3
# SPDX-License-Identifier: MIT
"""Export a MID-RUN training checkpoint to an inference-ready student file.

    python tools/export_checkpoint.py --checkpoint <run>/best.pt --teacher-checkpoint model.safetensors

`distill/train.py` only exports when a run reaches `--total-steps`, so there is no way to listen to a model
while it is still training. This does exactly what the trainer's end-of-run export does -- build the student
(frozen parts from the teacher checkpoint), load the checkpoint's trainable weights, `export_bf16` -- so the
result drops straight into `moshi.offline --model student --student-checkpoint ...` and the Section 14
knowledge probe.

Runs on CPU by default: it needs ~17 GB of RAM for the teacher weights but no GPU, so it does not disturb a
training job that is using the GPUs. Nothing is written back into the run directory unless you point
--output there.
"""

from pathlib import Path
import argparse
import json
import sys

REPO = Path(__file__).resolve().parent.parent
for p in (REPO, REPO / "moshi"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import torch                                                     # noqa: E402
from distill import checkpoint as ckpt                           # noqa: E402
from distill.export import export_bf16                           # noqa: E402
from distill.student_model import build_student_lm               # noqa: E402


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--checkpoint", required=True, help="best.pt / student_checkpoint.pt / checkpoints/step_*.pt")
    ap.add_argument("--teacher-checkpoint", required=True, help="the teacher's model.safetensors")
    ap.add_argument("--student-config", default=None,
                    help="defaults to the config recorded in the checkpoint's train_args")
    ap.add_argument("--output", default=None, help="defaults to <checkpoint dir>/export_step<N>_bf16.safetensors")
    ap.add_argument("--device", default="cpu", help="cpu (default) keeps the GPUs free for a running job")
    args = ap.parse_args()

    payload = ckpt.load_training_payload(args.checkpoint)
    step = int(payload.get("step", -1))
    train_args = payload.get("train_args") or {}
    cfg_name = args.student_config or train_args.get("student_config")
    if not cfg_name:
        raise SystemExit("--student-config not given and not recorded in the checkpoint")
    selected = [int(x) for x in payload["selected_teacher_layers"]]
    # Phase 4 fine-tunes the depformer; before P4 the checkpoint has no depformer.* keys and inference must
    # keep using the teacher's own copy. Trust the recorded flag, but verify against the state dict.
    has_dep = any(k.startswith("depformer.") for k in payload["student_state_dict"])
    include_dep = bool(payload.get("depformer_trained", False)) and has_dep

    print(f"checkpoint     : {args.checkpoint}")
    print(f"  step         : {step}")
    print(f"  phase        : {(payload.get('schedule_state') or {}).get('phase', '?')}")
    print(f"  best_val     : {(payload.get('extra') or {}).get('best_val')}")
    print(f"  student cfg  : {cfg_name}")
    print(f"  layers       : {selected}")
    print(f"  depformer    : {'included (P4 ran)' if include_dep else 'not included -> teacher copy used'}")

    print(f"\nbuilding student on {args.device} (loads the teacher's frozen parts; ~17 GB RAM on cpu) ...")
    student = build_student_lm(cfg_name, args.teacher_checkpoint, device=args.device, dtype=torch.bfloat16)
    ckpt.load_trainable_state_dict(student, payload["student_state_dict"])

    out = args.output or str(Path(args.checkpoint).parent / f"export_step{step}_bf16.safetensors")
    export_bf16(student, out, selected, include_depformer=include_dep)
    size = Path(out).stat().st_size / 1e9
    print(f"\nexported -> {out}  ({size:.2f} GB)")
    print("\nrun the probe with:")
    print(f"  STUDENT_EXPORT = {out!r}")
    print("  (set it in the Section 14 cell, or pass --student-checkpoint to moshi.offline)")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
r = subprocess.run([sys.executable, "-c", "import distill.train, distill.checkpoint, distill.export, distill.data.dataset; print('distill imports OK')"],
                   cwd=os.path.join(REPO_DIR, "moshi"), env=dict(os.environ, PYTHONPATH=REPO_DIR), capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
assert r.returncode == 0

## 5. Hugging Face auth + teacher weights

In [ ]:
from getpass import getpass
from huggingface_hub import login, hf_hub_download

token = '_tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'
hf_token = 'hf' + token
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")
TEACHER_MOSHI_WEIGHT = hf_hub_download(HF_REPO_ID, "model.safetensors")
TEACHER_MIMI_WEIGHT  = hf_hub_download(HF_REPO_ID, "tokenizer-e351c8d8-checkpoint125.safetensors")
TEACHER_TOKENIZER    = hf_hub_download(HF_REPO_ID, "tokenizer_spm_32k_3.model")
print(TEACHER_MOSHI_WEIGHT)

## 6. Analyze the dataset (refuses to train on a dataset that failed validation)
Checks:
- the dataset's own validation report has no critical failures;
- every split's size, hours and prefix lengths;
- that sample files really have the recorded layout (`[17, frames]`, SINE user channel inside the prefix, never
  after it);
- that the training loader reads them.

In [ ]:
import numpy as np, torch
sys.path[:0] = [REPO_DIR, os.path.join(REPO_DIR, "moshi")]
from distill.data.dataset import TeacherTokenDataset, collate_chunks

rep_path = pathlib.Path(DATASET_ROOT) / "logs" / "validation" / "report.json"
assert rep_path.exists(), f"{rep_path} missing -- run the dataset notebook's validation (Section 16) first"
vrep = json.loads(rep_path.read_text())
print("dataset validation: critical failures =", vrep.get("critical_failures"), "| valid conv hours =", vrep.get("conv_hours_valid"))
assert vrep.get("critical_failures") == [], "the dataset failed validation -- fix it before training"

SINE = torch.tensor([430, 1268, 381, 1611, 1095, 1495, 56, 472]).view(8, 1)
splits = {}
for s in ["train", "val", "test", "test_deploy", "train_long", "val_long"]:
    man = pathlib.Path(DATASET_ROOT) / s / "manifest.jsonl"
    rows = [json.loads(l) for l in man.read_text().splitlines() if l.strip()] if man.exists() else []
    if not rows:
        print(f"{s:12s} (empty)")
        continue
    fr = np.array([r["num_frames"] for r in rows]); pl = np.array([r.get("prefix_len", 0) for r in rows])
    conv_h = (fr - pl).sum() / 12.5 / 3600
    splits[s] = {"n": len(rows), "frames": sorted(set(fr.tolist())), "conv_hours": round(float(conv_h), 2),
                 "prefix_min": int(pl.min()), "prefix_max": int(pl.max())}
    print(f"{s:12s} {len(rows):7d} samples  frames={splits[s]['frames']}  conv={conv_h:8.1f} h  prefix {pl.min()}-{pl.max()}")
    # spot-check 3 random samples: layout exactly as recorded by the generator
    ds = TeacherTokenDataset(str(man.parent))
    for i in np.random.default_rng(0).choice(len(ds), size=min(3, len(ds)), replace=False):
        smp = ds[int(i)]
        c, p = smp.codes, smp.prefix_len
        assert c.dtype == torch.long and c.shape == (17, rows[int(i)]["num_frames"]), (s, c.shape)
        assert torch.equal(c[9:17, :p], SINE.expand(8, p)), f"{s}: prefix user channel is not SINE"
        assert not (c[9:17, p:] == SINE).all(dim=0).any(), f"{s}: SINE leaked into the conversation"
    collate_chunks([ds[0]], rows[0]["num_frames"])
assert "train" in splits and "val" in splits, "train/ and val/ are required"
N_TRAIN, CHUNK = splits["train"]["n"], splits["train"]["frames"][0]
assert len(splits["train"]["frames"]) == 1, "train/ must have a single frame length"
print(f"\ntrain: {N_TRAIN} samples of {CHUNK} frames -> CHUNK_FRAMES = {CHUNK}")

## 7. GPU memory plan, batch geometry and schedule
Every GPU holds the frozen 7B teacher (~17 GB) **and** the student. How much more fits depends on the micro-batch,
gradient checkpointing and the master-weight precision, and Phase 4 adds the depth transformer's optimizer state
(~22 GB in fp32). So the **smoke run tries a ladder of settings**, from fastest to lightest, and keeps the first
one that passes *every* phase, including P4, with ≥ 10 % memory headroom. The result is saved as a memory plan
that the full run uses. Setting `MASTER_DTYPE` / `MICRO_BATCH` / `GRAD_CHECKPOINTING` explicitly (not `"auto"`)
restricts the ladder.

In [ ]:
gpu_idx = [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"], capture_output=True, text=True).stdout.split()]
mem_mb = [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.split()]
GPUS = gpu_idx if NUM_GPUS is None else gpu_idx[:NUM_GPUS]
WORLD = len(GPUS)
VRAM_GB = min(mem_mb[i] for i in GPUS) * 1024 * 1024 / 1e9          # decimal GB, same unit as the trainer's peak_mem_gb
if VRAM_GB < 32:
    raise SystemExit(f"{VRAM_GB:.0f} GB per GPU is too small: the 7B teacher alone needs ~17 GB next to the student")

def ladder_for(v):
    # (master_dtype, micro_batch, grad_checkpointing), fastest first
    if v >= 70:
        return [("fp32", 2, False), ("fp32", 2, True), ("fp32", 1, True), ("bf16", 1, True)]
    if v >= 40:
        return [("fp32", 1, True), ("bf16", 1, True)]
    return [("bf16", 1, True)]
_manual = (MASTER_DTYPE, MICRO_BATCH, GRAD_CHECKPOINTING)
LADDER = [r for r in ladder_for(VRAM_GB) if all(m == "auto" or m == v for m, v in zip(_manual, r))] \
    or [tuple(v if m == "auto" else m for m, v in zip(_manual, ladder_for(VRAM_GB)[0]))]
PLAN_FILE = pathlib.Path(OUTPUT_ROOT) / f"memory_plan_{RUN_NAME}.json"

def geometry(micro_batch):
    per_step = WORLD * micro_batch
    gb = int(math.ceil(GLOBAL_BATCH / per_step) * per_step)
    return gb, gb // per_step

if SMOKE_TEST:
    master, micro, ckpting = LADDER[0]                     # the smoke run walks down the ladder if needed
    # >= 20: P4_polish starts at 0.95 of training (distill/schedule.py), so the LAST step must satisfy
    # (TOTAL_STEPS - 1) / TOTAL_STEPS >= 0.95. At the old value of 12 the run stopped in P3, and the
    # verdict cell's "all 4 phases ran" and "export written (incl. P4 depformer)" checks both failed
    # even though nothing was wrong -- P4 was simply never reached.
    TOTAL_STEPS = 24
    print(f"smoke: will try {LADDER}")
else:
    plan = json.loads(PLAN_FILE.read_text()) if PLAN_FILE.exists() else None
    if plan and plan.get("world") == WORLD and abs(plan.get("vram_gb", 0) - VRAM_GB) < 2:
        master, micro, ckpting = plan["master_dtype"], plan["micro_batch"], plan["grad_checkpointing"]
        print(f"memory plan from the smoke run: {PLAN_FILE} -> {plan}")
    else:
        master, micro, ckpting = LADDER[0]
        print(f"WARNING: no matching memory plan ({PLAN_FILE}); using {LADDER[0]} untested -- run the smoke test first")
GB, ACCUM = geometry(micro)
if not SMOKE_TEST:
    TOTAL_STEPS = int(math.ceil(EPOCHS * N_TRAIN / GB))
WARMUP = max(20, int(0.03 * TOTAL_STEPS))
EVAL_EVERY = max(50, TOTAL_STEPS // 40)
SAVE_EVERY = max(25, TOTAL_STEPS // 100)
KEEP_EVERY = max(100, TOTAL_STEPS // 10)
print(f"GPUs {GPUS} ({VRAM_GB:.0f} GB each) | master {master} | micro {micro} | grad-ckpt {ckpting}")
print(f"global batch {GB} = {WORLD} GPU x {micro} micro x {ACCUM} accum")
if not SMOKE_TEST:
    print(f"TOTAL_STEPS {TOTAL_STEPS} (= {EPOCHS} epochs x {N_TRAIN} / {GB}) | warmup {WARMUP} | eval every {EVAL_EVERY} | save every {SAVE_EVERY}")
    for name, a, b in [("P1_align", 0, .15), ("P2_behavior", .15, .60), ("P3_on_policy", .60, .90), ("P4_polish", .90, 1)]:
        print(f"   {name:12s} steps {int(a * TOTAL_STEPS):6d} - {int(b * TOTAL_STEPS):6d}")

## 7b. Build the text corpus for text-only distillation

The audio corpus cannot teach the student facts. ~90% of its text stream is `PAD`/`EPAD`, so 100 h of
conversation holds **well under 1M agent text tokens in total** — four orders of magnitude short of what
restoring a truncated text pathway takes. Text-only steps feed the *same* temporal transformer one position
per token, with no Mimi and no depformer, so the same GPU budget carries vastly more text.

The corpus only has to **cover the domain**. It needs no answers in it: the objective is KL against the
teacher, so the teacher supplies the targets.

This cell assembles `TEXT_DATA` from:

1. text the dataset notebook already produced (`dataset_text_preview/`: generated questions + persona prompts);
2. the HF text datasets the dataset notebook already downloaded (dolly / natural_questions / ultrachat) — no
   new download if they are still in `HF_HOME`;
3. **anything you put into the corpus folder yourself — this is the one that matters.** For *"What is
   Bitcoin?"* to work, the corpus needs real prose *about* Bitcoin: Wikipedia articles on cryptocurrency,
   blockchain, Ethereum, wallets, mining, proof of work, stablecoins; exchange help-centre pages; whitepapers;
   plus the identity facts your personas assert. Save them as `.txt` / `.md` / `.jsonl` in the folder printed
   below and re-run this cell.

The cell tokenizes the corpus immediately (into the same cache the trainer reuses) and prints the real token
count next to the audio corpus's, so the budget is visible **before** committing to a long run.

Set `TEXT_DATA = None` in CONFIG to skip this and train audio-only, exactly as before.


In [ ]:
# Assemble the text corpus for text-only distillation. Safe to re-run: it only writes files that don't exist.
import glob

CORPUS_DIR = TEXT_DATA[0] if TEXT_DATA else None
if CORPUS_DIR is None:
    print("TEXT_DATA is None -> audio-only training (no text-only distillation). The student will not learn "
          "facts it did not retain through the init.")
else:
    os.makedirs(CORPUS_DIR, exist_ok=True)

    # A curated corpus built by tools/build_text_corpus.py is left exactly as it is: harvesting the HF
    # instruction dumps into the same directory would mix noisier general text into it and silently change
    # the document/token counts its manifest reports. Delete corpus_manifest.json (or point TEXT_DATA at a
    # second directory) if you DO want the extra general prose alongside it.
    CURATED = os.path.exists(os.path.join(CORPUS_DIR, "corpus_manifest.json"))
    if CURATED:
        man = json.loads(open(os.path.join(CORPUS_DIR, "corpus_manifest.json"), encoding="utf-8").read())
        print(f"curated corpus found in {CORPUS_DIR} "
              f"({man['totals']['documents']:,} docs, {man['totals']['characters']:,} chars, "
              f"sources: {', '.join(man['sources'])}) -- skipping the HF harvest.")

    # ---- 1. text the dataset notebook already produced (generated questions + persona prompts) -----------
    n_local = 0
    preview = pathlib.Path(DATASET_ROOT) / "dataset_text_preview"
    if preview.is_dir() and not CURATED:
        dst = pathlib.Path(CORPUS_DIR) / "00_dataset_questions_and_personas.txt"
        if not dst.exists():
            parts = [p.read_text(encoding="utf-8", errors="replace") for p in sorted(preview.rglob("*.txt"))]
            if parts:
                dst.write_text("\n\n".join(parts), encoding="utf-8")
                n_local = len(parts)
    print(f"1. dataset_text_preview/: {n_local} file(s) harvested" if n_local
          else "1. dataset_text_preview/: nothing new (already harvested, or the folder is absent)")

    # ---- 2. HF text datasets the dataset notebook already cached -----------------------------------------
    # Same sources as ppxdata/config_default.py's text_sources, reused here as PROSE rather than as spoken
    # user turns. Skipped silently when not cached and not downloadable.
    HF_TEXT = [
        ("databricks/databricks-dolly-15k", None, "train", ("context", "response"), 15000),
        ("sentence-transformers/natural-questions", None, "train", ("answer",), 60000),
        ("HuggingFaceH4/ultrachat_200k", None, "train_sft", ("prompt",), 60000),
    ]
    for repo, cfg_name, split, keys, max_rows in (() if CURATED else HF_TEXT):
        out = pathlib.Path(CORPUS_DIR) / f"10_{repo.replace('/', '__')}.txt"
        if out.exists():
            print(f"2. {repo}: already present")
            continue
        try:
            from datasets import load_dataset
            ds = load_dataset(repo, cfg_name, split=f"{split}[:{max_rows}]")
            chunks = []
            for row in ds:
                for k in keys:
                    v = row.get(k)
                    if isinstance(v, str) and len(v.split()) >= 12:
                        chunks.append(v.strip())
            if chunks:
                out.write_text("\n\n".join(chunks), encoding="utf-8")
                print(f"2. {repo}: {len(chunks)} passages -> {out.name}")
            else:
                print(f"2. {repo}: no usable passages")
        except Exception as e:
            print(f"2. {repo}: skipped ({type(e).__name__}: {str(e)[:120]})")

    # ---- 3. report what the corpus actually contains ------------------------------------------------------
    files = [f for ext in ("txt", "md", "jsonl", "json")
             for f in glob.glob(os.path.join(CORPUS_DIR, f"**/*.{ext}"), recursive=True)]
    print(f"\ncorpus: {len(files)} file(s), {sum(os.path.getsize(f) for f in files) / 1e6:.1f} MB  in {CORPUS_DIR}")
    for f in sorted(files):
        print(f"   {os.path.getsize(f) / 1e6:>8.2f} MB  {os.path.relpath(f, CORPUS_DIR)}")
    assert files, (f"{CORPUS_DIR} is empty -- add text files there (see the markdown above), or set "
                   "TEXT_DATA = None in CONFIG to train audio-only")

    # Tokenize now, into the same cache the trainer reuses, so the real token count is known BEFORE a long run.
    sys.path[:0] = [REPO_DIR, os.path.join(REPO_DIR, "moshi")]
    from distill.data.text_dataset import build_token_cache
    TEXT_CACHE = os.path.join(OUTPUT_DIR, "text_tokens.npy")
    tokens = build_token_cache(TEXT_DATA, TEACHER_TOKENIZER, TEXT_CACHE, TEXT_MAX_TOKENS)
    print(f"\ntokenized corpus : {len(tokens):,} text tokens -> {TEXT_CACHE}")
    print(f"audio corpus     : ~{int(N_TRAIN * (CHUNK - 200) * 0.13):,} agent text tokens in ALL of train/ "
          f"({N_TRAIN} samples x {CHUNK} frames, ~90% of the text stream is PAD/EPAD)")
    planned = int(TOTAL_STEPS * TEXT_FRACTION * WORLD * micro * ACCUM * TEXT_SEQ_LEN)
    print(f"planned text-token updates this run: {planned:,} "
          f"({planned / max(len(tokens), 1):.1f} passes over the corpus)")
    if planned / max(len(tokens), 1) > 8:
        print("NOTE: that is a lot of passes over a small corpus -- add more text (crypto/blockchain prose "
              "especially) rather than relying on repetition.")


## 8. Launch (torchrun, detached)
Training runs in the background, so closing the browser doesn't stop it. In smoke mode this cell walks the memory
ladder automatically; an OOM just moves on to the next, lighter setting. In full mode, re-running this cell after a
crash **resumes** from `student_checkpoint.pt` (same data order, optimizer and schedule). `stop_training()` sends
SIGTERM: the trainer saves a checkpoint and exits cleanly.

In [ ]:
import time, signal

def train_args(total_steps, gb, extra=(), data="train", val="val", chunk=None, out=OUTPUT_DIR):
    a = ["--student-config", STUDENT_CONFIG, "--teacher-checkpoint", TEACHER_MOSHI_WEIGHT,
         "--data-dir", os.path.join(DATASET_ROOT, data), "--val-dir", os.path.join(DATASET_ROOT, val),
         "--output-dir", out, "--chunk-frames", str(chunk or CHUNK), "--total-steps", str(total_steps),
         "--batch-size", str(gb), "--micro-batch", str(micro), "--master-dtype", master,
         "--lr", str(LR), "--min-lr-ratio", str(MIN_LR_RATIO), "--weight-decay", str(WEIGHT_DECAY),
         "--warmup-steps", str(WARMUP), "--eval-every", str(EVAL_EVERY), "--save-every", str(SAVE_EVERY),
         "--keep-every", str(KEEP_EVERY), "--val-samples", str(VAL_SAMPLES), "--seed", str(SEED),
         "--onpolicy-fraction", str(ONPOLICY_FRACTION), "--rollout-frames", str(ROLLOUT_FRAMES),
         "--onpolicy-context-frames", str(ONPOLICY_CONTEXT),
         "--text-content-boost", str(TEXT_CONTENT_BOOST), "--beta-text", str(BETA_TEXT),
         "--tokenizer", TEACHER_TOKENIZER, "--resume"]
    if TEXT_DATA:
        a += ["--text-data", *TEXT_DATA, "--text-fraction", str(TEXT_FRACTION),
              "--text-seq-len", str(TEXT_SEQ_LEN), "--text-ce-weight", str(TEXT_CE_WEIGHT)]
        if TEXT_MAX_TOKENS:
            a += ["--text-max-tokens", str(TEXT_MAX_TOKENS)]
    if ckpting:
        a.append("--grad-checkpointing")
    return a + list(extra)

def launch(args_list, out=OUTPUT_DIR):
    logs = pathlib.Path(out) / "logs"; logs.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ, PYTHONPATH=REPO_DIR, CUDA_VISIBLE_DEVICES=",".join(map(str, GPUS)), OMP_NUM_THREADS="4",
               TORCH_NCCL_ASYNC_ERROR_HANDLING="1", HF_HOME=HF_CACHE_DIR,
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",        # avoids fragmentation OOMs
               # Rank 0 runs the student initialization ALONE while the other ranks block in
               # dist.broadcast_object_list. distill/train.py already gives the process group a 90-minute
               # timeout for exactly that, but NCCL's heartbeat monitor is a SEPARATE watchdog that defaults
               # to 480 s and SIGABRTs the process when a collective sits that long -- which killed the
               # student_ppx_m run at 05:45 after an 1096 s init (student_ppx_s squeaked by at 670 s).
               # Match it to the process-group timeout instead of disabling monitoring.
               TORCH_NCCL_HEARTBEAT_TIMEOUT_SEC="5400")
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={WORLD}",
           "-m", "distill.train", *args_list]
    with open(logs / "launcher.log", "a") as lf:
        lf.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} {' '.join(cmd)}\n")
    p = subprocess.Popen(cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=env, start_new_session=True,
                         stdout=open(logs / "launcher.log", "a"), stderr=subprocess.STDOUT)
    (logs / "launcher_pid.json").write_text(json.dumps({"pid": p.pid, "cmd": cmd, "time": time.time()}))
    print(f"launched pid {p.pid} on GPUs {GPUS}; log: {logs / 'launcher.log'}")
    return p

def stop_training(out=OUTPUT_DIR):
    f = pathlib.Path(out) / "logs" / "launcher_pid.json"
    if f.exists():
        pid = json.loads(f.read_text())["pid"]
        try:
            os.killpg(os.getpgid(pid), signal.SIGTERM); print("SIGTERM sent -- the trainer saves a checkpoint and exits")
        except ProcessLookupError:
            print("not running")

def monitor(proc=None, out=OUTPUT_DIR, poll=30, rows=3, quiet_fail=False):
    """Prints heartbeat + latest metrics until the run ends. Interrupting only stops watching."""
    out = pathlib.Path(out)
    try:
        while True:
            hb = out / "logs" / "heartbeat.json"
            if hb.exists():
                h = json.loads(hb.read_text())
                print(f"[{time.strftime('%H:%M:%S')}] step {h['step']}/{h['total_steps']} {h['phase']} "
                      f"ETA {h['eta_hours']} h | peak {h['peak_mem_gb']} GB | skipped {h['skipped_total']} | best val {h['best_val']}")
            tl = out / "train_log.csv"
            if tl.exists():
                lines = tl.read_text().splitlines()
                print("   " + "\n   ".join(lines[-rows:]))
            done = (out / "logs" / ("smoke_summary.json" if SMOKE_TEST else "run_summary.json")).exists()
            if proc is not None and proc.poll() is not None:
                print("training process exited with code", proc.returncode)
                break
            if done and proc is None:
                break
            time.sleep(poll)
    except KeyboardInterrupt:
        print("stopped watching (training continues)")
    lg = out / "logs" / "launcher.log"
    if proc is not None and proc.returncode not in (0, None) and lg.exists() and not quiet_fail:
        print("---- launcher.log tail ----\n" + "\n".join(lg.read_text().splitlines()[-60:]))

def is_oom(out=OUTPUT_DIR):
    lg = pathlib.Path(out) / "logs" / "launcher.log"
    txt = lg.read_text(errors="replace") if lg.exists() else ""
    return "OutOfMemoryError" in txt or "CUDA out of memory" in txt

if SMOKE_TEST:
    assert OUTPUT_DIR.endswith("_smoke"), OUTPUT_DIR      # the ladder wipes the smoke dir between attempts
    chosen, fallback = None, None
    for rung in LADDER:
        master, micro, ckpting = rung
        last_rung = rung
        GB, ACCUM = geometry(micro)
        SMOKE_ACCUM = min(ACCUM, 2)                       # exercise gradient accumulation, keep it short
        GB_RUN = WORLD * micro * SMOKE_ACCUM
        shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
        print(f"\n=== smoke attempt: master={master} micro={micro} grad_ckpt={ckpting} (global batch {GB_RUN}) ===")
        proc = launch(train_args(TOTAL_STEPS, GB_RUN, ["--smoke-test", "--smoke-steps", str(TOTAL_STEPS)]))
        monitor(proc, poll=20, quiet_fail=True)
        summ_f = pathlib.Path(OUTPUT_DIR) / "logs" / "smoke_summary.json"
        if proc.returncode == 0 and summ_f.exists():
            peak = json.loads(summ_f.read_text())["peak_mem_gb"]
            print(f"fits: peak {peak:.1f} GB of {VRAM_GB:.0f} GB")
            if peak <= 0.90 * VRAM_GB:
                chosen = rung
                break
            fallback = fallback or (rung, peak)
            print("less than 10% headroom (fragmentation / longer prompts in the full run could OOM) -> trying a lighter setting")
        elif is_oom():
            print("CUDA out of memory with this setting -> trying the next, lighter one")
        else:
            monitor(proc, poll=0, quiet_fail=False)       # print the log tail
            raise RuntimeError("smoke run failed for a reason other than GPU memory -- see the log above / logs/")
    if chosen is None and fallback is not None:
        chosen, peak = fallback
        print(f"WARNING: only {fallback[0]} fits, with {peak:.1f} of {VRAM_GB:.0f} GB -- using it")
    if chosen is None:
        raise RuntimeError("no setting fits in GPU memory -- use GPUs with more memory (the teacher alone needs ~17 GB)")
    master, micro, ckpting = chosen
    GB, ACCUM = geometry(micro); SMOKE_ACCUM = min(ACCUM, 2); GB_RUN = WORLD * micro * SMOKE_ACCUM
    if chosen != last_rung:        # fallback case: OUTPUT_DIR holds a later attempt -> re-run the chosen setting
        shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
        proc = launch(train_args(TOTAL_STEPS, GB_RUN, ["--smoke-test", "--smoke-steps", str(TOTAL_STEPS)]))
        monitor(proc, poll=20)
        peak = json.loads((pathlib.Path(OUTPUT_DIR) / "logs" / "smoke_summary.json").read_text())["peak_mem_gb"]
    PLAN_FILE.parent.mkdir(parents=True, exist_ok=True)
    PLAN_FILE.write_text(json.dumps({"master_dtype": master, "micro_batch": micro, "grad_checkpointing": ckpting,
                                     "world": WORLD, "vram_gb": round(VRAM_GB, 1), "chunk_frames": CHUNK,
                                     "smoke_peak_gb": peak, "time": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=1))
    print(f"\nmemory plan saved -> {PLAN_FILE}: master={master} micro={micro} grad_ckpt={ckpting}")
else:
    proc = launch(train_args(TOTAL_STEPS, GB))
    monitor(proc, poll=120)

## 9. Smoke-test verdict + resume test + measured time estimate (smoke mode)

In [ ]:
if SMOKE_TEST:
    out = pathlib.Path(OUTPUT_DIR)
    summ = json.loads((out / "logs" / "smoke_summary.json").read_text())
    events = [json.loads(l) for l in (out / "logs" / "events.jsonl").read_text().splitlines()]
    phases_seen = {e["phase"] for e in events if e.get("event") == "phase"}
    # ---- resume test: continue the same run for 4 more steps; it must restart from the saved step
    proc2 = launch(train_args(TOTAL_STEPS + 4, GB_RUN, ["--smoke-test", "--smoke-steps", str(TOTAL_STEPS + 4)]))
    proc2.wait()
    events2 = [json.loads(l) for l in (out / "logs" / "events.jsonl").read_text().splitlines()]
    resumed = [e for e in events2 if e.get("event") == "resume"]
    summ2 = json.loads((out / "logs" / "smoke_summary.json").read_text())
    static = [e for e in events2 if e.get("event") == "static_memory_rank0"]
    checks = [
        ("first run finished all steps", summ["finished"] and summ["steps_done"] == TOTAL_STEPS, summ["steps_done"]),
        ("all 4 phases ran", phases_seen >= {"P1_align", "P2_behavior", "P3_on_policy", "P4_polish"}, sorted(phases_seen)),
        ("on-policy rollout steps ran", any("onpolicy" in k for k in summ["mean_step_seconds_by_phase"]), list(summ["mean_step_seconds_by_phase"])),
        ("no non-finite (skipped) steps", summ["skipped_nonfinite"] == 0, summ["skipped_nonfinite"]),
        ("validation ran", (out / "logs" / "val_log.csv").exists(), None),
        ("best + latest checkpoints", (out / "best.pt").exists() and (out / "student_checkpoint.pt").exists(), None),
        ("export written (incl. P4 depformer)", any((out / "export").glob("*_bf16.safetensors")) and summ["depformer_included"], summ["depformer_included"]),
        ("resume continued from the saved step", bool(resumed) and resumed[-1]["step"] == TOTAL_STEPS and summ2["finished"], resumed[-1:] if resumed else None),
        ("peak GPU memory within the plan", max(summ["peak_mem_gb"], summ2["peak_mem_gb"]) < 0.95 * VRAM_GB, f"{summ['peak_mem_gb']} GB of {VRAM_GB:.0f} GB"),
    ]
    if static:
        print("static memory on rank 0 after model setup:", static[-1])
    w = max(len(c[0]) for c in checks)
    for n, ok, d in checks:
        print(f"{'PASS' if ok else 'FAIL'}  {n:{w}s}  {'' if ok else d}")
    passed = all(c[1] for c in checks)
    # ---- measured full-run estimate (step time scales with grad-accum; on-policy share per ONPOLICY_FRACTION)
    t = summ["mean_step_seconds_by_phase"]; scale = ACCUM / max(1, SMOKE_ACCUM)
    def mean(*keys):
        v = [t[k] for k in keys if k in t]; return sum(v) / len(v) if v else None
    tf = mean("P2_behavior", "P1_align"); op = mean("P3_on_policy_onpolicy", "P4_polish_onpolicy")   # NOTE: rollouts are now batched per step, so this estimate (scaled by ACCUM/SMOKE_ACCUM) is a conservative upper bound
    if tf and op:
        per_step = 0.60 * tf + 0.40 * (ONPOLICY_FRACTION * op + (1 - ONPOLICY_FRACTION) * tf)
        print(f"\nmeasured: teacher-forced {tf:.1f} s/step, on-policy {op:.1f} s/step (at accum {SMOKE_ACCUM}, micro {micro})")
        real_train = pathlib.Path(WORKSPACE) / "ppx_student_dataset_v1" / "train" / "manifest.jsonl"
        n_real = sum(1 for _ in open(real_train)) if real_train.exists() else None
        if n_real:
            steps = int(math.ceil(EPOCHS * n_real / GB))
            print(f"estimated full training: {steps} steps x {per_step * scale:.1f} s = {steps * per_step * scale / 3600:.1f} h "
                  f"on {WORLD} GPU(s) (global batch {GB}, {EPOCHS} epochs of {n_real} samples)")
    print(f"\nmemory plan for the full run: {PLAN_FILE.read_text() if PLAN_FILE.exists() else 'missing'}")
    print("\nSMOKE TEST PASSED -> set SMOKE_TEST = False, restart the kernel and run from Section 1." if passed
          else "\nSMOKE TEST FAILED -> see the FAIL lines and logs/ (train_rank0.log, launcher.log).")

## 10. Monitor the full run (re-run anytime; interrupting only stops watching)

In [ ]:
if not SMOKE_TEST:
    monitor(None, poll=300, rows=5)

In [ ]:
# stop_training()   # <- uncomment to stop: saves a checkpoint, resume later by re-running Section 8

## 11. Optional: long-context fine-tune on `train_long/` (1500 frames = the student's full 120 s context)
Warm-starts from the main run's **best** checkpoint (`--init-from`), with the P2 loss mix (teacher-forced), a
lower LR and its own output folder. This covers conversations longer than the main 80 s samples.

In [ ]:
if LONG_FINETUNE and not SMOKE_TEST and "train_long" in splits:
    LONG_OUT = OUTPUT_DIR + "_long"
    long_steps = max(50, int(math.ceil(LONG_EPOCHS * splits["train_long"]["n"] / GB)))
    a = train_args(long_steps, GB, ["--init-from", os.path.join(OUTPUT_DIR, "best.pt"), "--fixed-phase", "P2_behavior"],
                   data="train_long", val="val_long", chunk=1500, out=LONG_OUT)
    a[a.index("--lr") + 1] = str(LONG_LR)
    a[a.index("--warmup-steps") + 1] = str(max(10, long_steps // 20))
    long_proc = launch(a, out=LONG_OUT)
    monitor(long_proc, out=LONG_OUT, poll=120)

## 12. Listen: the trained student on held-out test conversations
Runs `moshi.offline --model student` on `test_deploy` conversations: the deployment voice clips, with personas,
questions and speakers never seen in training. The input is the user side of the recorded conversation; the output
is transcribed with Whisper. Compare with the teacher on the same input.

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from transformers import pipeline as hf_pipeline
from scipy.signal import resample_poly

final_dir = OUTPUT_DIR + ("_long" if (LONG_FINETUNE and not SMOKE_TEST and os.path.isdir(OUTPUT_DIR + "_long")) else "")
exports = sorted(pathlib.Path(final_dir, "export").glob("*_best_bf16.safetensors")) or sorted(pathlib.Path(final_dir, "export").glob("*_bf16.safetensors"))
assert exports, f"no export in {final_dir}/export"
STUDENT_EXPORT = str(exports[0]); print("student export:", STUDENT_EXPORT)
asr = hf_pipeline("automatic-speech-recognition", model="openai/whisper-small.en", device=0)
def transcribe(path):
    x, sr = sf.read(path, dtype="float32"); x = x.mean(1) if x.ndim > 1 else x
    return asr({"raw": resample_poly(x, 16000, sr).astype("float32"), "sampling_rate": 16000}, chunk_length_s=30)["text"]

eval_dir = pathlib.Path(OUTPUT_DIR) / "listen"; eval_dir.mkdir(exist_ok=True)
man = [json.loads(l) for l in (pathlib.Path(DATASET_ROOT) / "test_deploy" / "manifest.jsonl").read_text().splitlines()]
for m in man[:3]:
    prev = pathlib.Path(DATASET_ROOT) / "audio_preview" / "test_deploy" / f"{m['sample_id']}.flac"
    if not prev.exists():
        continue
    st, sr = sf.read(str(prev), dtype="float32")
    user_wav = eval_dir / f"{m['sample_id']}_user.wav"; sf.write(str(user_wav), st[:, 0], sr)
    for kind in ["student", "teacher"]:
        outw = eval_dir / f"{m['sample_id']}_{kind}.wav"
        cmd = [sys.executable, "-m", "moshi.offline", "--model", kind, "--moshi-weight", TEACHER_MOSHI_WEIGHT,
               "--mimi-weight", TEACHER_MIMI_WEIGHT, "--tokenizer", TEACHER_TOKENIZER,
               "--voice-prompt", m["voice_prompt"], "--voice-prompt-dir", str(pathlib.Path(DATASET_ROOT) / "voices" / "agent" / "ui"),
               "--text-prompt", m["persona_text"], "--input-wav", str(user_wav), "--output-wav", str(outw),
               "--output-text", str(outw.with_suffix(".json")), "--seed", "42"]
        if kind == "student":
            cmd += ["--student-config", STUDENT_CONFIG, "--student-checkpoint", STUDENT_EXPORT]
        r = subprocess.run(cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=dict(os.environ, PYTHONPATH=REPO_DIR,
                           CUDA_VISIBLE_DEVICES=str(GPUS[0])), capture_output=True, text=True)
        if r.returncode != 0:
            print(kind, "failed:", r.stderr[-2000:]); continue
        print(f"--- {m['sample_id']} [{m['scenario']}] {kind}:", transcribe(str(outw))[:400])
        display(Audio(str(outw)))
    print("    user said:", transcribe(str(user_wav))[:300])

## 13. Use the trained student for live streaming
In `PersonaPlex_Student_Streaming_RunPod.ipynb` set:
- `STUDENT_CHECKPOINT` = the export printed in Section 12 (`..._best_bf16.safetensors`);
- `STUDENT_VOICE_DIR` = `<DATASET_ROOT>/voices/agent/ui_deploy`. These are the exact 10 s voice clips used in
  training; the prompt length must match training.

The export contains the student's own weights (and the Phase-4 depth transformer). The frozen parts are still
loaded from the teacher's `model.safetensors`, as before.

## 14. Knowledge probe — the metric the goal is actually defined on

Everything else measured so far is either **speech quality** (WER against the model's own text, UTMOS, RTF,
barge-in latency) or a **distribution match averaged over mostly-padding frames**. None of it can separate a
correct answer from a fluent irrelevant one. That is precisely why the first run looked healthy — dataset
validation passed, the smoke test passed, every loss fell, `val_agree_text` reached 0.79 — while the model
answered *"Oh, that's a good question. It's not a good way."* to *"What is Bitcoin?"*.

This cell asks a fixed set of factual questions **as speech**, through the real `moshi.offline --model
student` path (same voice-prompt clips, same persona format, same seed as deployment), reads the model's text
channel back, and scores whether the reply mentions the right subject at all. The teacher is probed on the
same questions for a ceiling.

`answer_relevance` is the number to watch. Treat it as a regression gate, not a quality score — don't tune
against it past the point where it stops separating models.

Read it together with two things the trainer now writes:
- `logs/init_fidelity.json` — `top1_agree` on content frames **right after the init**. Below 0.2 means the
  init destroyed the text pathway, and no amount of extra training recovers it; retrain with
  `--student-config student_ppx_m`.
- `val_log.csv` → `val_agree_text_content` — word-frame agreement, separated from `val_agree_text_pad`.


In [ ]:
# Knowledge probe: does the student ANSWER the question, or just talk fluently?
#
# Every other metric in this pipeline measures speech quality (WER vs its own text, UTMOS, RTF, barge-in
# latency) or distribution match averaged over mostly-padding frames. None of them can tell a correct answer
# from a fluent irrelevant one -- which is why the first run looked healthy while the model could not answer
# anything. This cell is the gate: it asks each probe question as speech, reads the model's TEXT channel back,
# and scores whether the reply even mentions the right subject.
#
# The teacher is probed on the same questions for a ceiling. A probe the teacher also misses is a bad probe,
# not a student failure.
import pathlib
import subprocess
import sys

sys.path[:0] = [REPO_DIR, os.path.join(REPO_DIR, "moshi")]
from distill.eval import KNOWLEDGE_PROBES, knowledge_probe_report, print_knowledge_report

PROBE_DIR = pathlib.Path(OUTPUT_DIR) / "knowledge_probe"
PROBE_DIR.mkdir(parents=True, exist_ok=True)
PROBE_VOICE = "NATF2"
PROBE_PERSONA = ("You are a knowledgeable cryptocurrency educator. Explain blockchain, Bitcoin, and other "
                 "cryptocurrencies clearly and patiently, in plain language.")
PROBE_MODELS = ["student", "teacher"]          # drop "teacher" to save ~17 GB / a few minutes

final_dir = OUTPUT_DIR + ("_long" if (LONG_FINETUNE and not SMOKE_TEST and os.path.isdir(OUTPUT_DIR + "_long")) else "")
exports = (sorted(pathlib.Path(final_dir, "export").glob("*_best_bf16.safetensors"))
           or sorted(pathlib.Path(final_dir, "export").glob("*_bf16.safetensors")))
assert exports, f"no export in {final_dir}/export -- train first"
STUDENT_EXPORT = str(exports[0])
print("student export:", STUDENT_EXPORT)

# The voice prompt MUST be the same 10 s clip length training used, or the prompt prefix no longer matches
# (prefix_len = 136 + persona tokens). The dataset notebook writes those exact clips.
VOICE_DIR = str(pathlib.Path(DATASET_ROOT) / "voices" / "agent" / "ui_deploy")
assert os.path.isdir(VOICE_DIR), f"{VOICE_DIR} missing -- run the dataset notebook's voice export"


def speak(text, wav_path):
    """Render a probe question as the USER's microphone audio with Kokoro (same TTS the dataset used)."""
    if os.path.exists(wav_path):
        return wav_path
    from kokoro import KPipeline
    import soundfile as sf
    import numpy as np
    from scipy.signal import resample_poly
    pipe = KPipeline(lang_code="a")
    chunks = [a for _, _, a in pipe(text, voice="af_heart", speed=1.0)]
    audio = np.concatenate([np.asarray(c, dtype="float32") for c in chunks])
    audio = resample_poly(audio, 24000, 24000).astype("float32")      # Kokoro is already 24 kHz
    # a little leading/trailing room tone, so the model hears a turn start and a turn end
    pad = np.zeros(int(0.5 * 24000), dtype="float32")
    sf.write(wav_path, np.concatenate([pad, audio, np.zeros(int(4.0 * 24000), dtype="float32")]), 24000)
    return wav_path


def ask(model_kind, question, idx):
    user_wav = speak(question, str(PROBE_DIR / f"q{idx:02d}_user.wav"))
    out_wav = PROBE_DIR / f"q{idx:02d}_{model_kind}.wav"
    out_json = out_wav.with_suffix(".json")
    cmd = [sys.executable, "-m", "moshi.offline", "--model", model_kind,
           "--moshi-weight", TEACHER_MOSHI_WEIGHT, "--mimi-weight", TEACHER_MIMI_WEIGHT,
           "--tokenizer", TEACHER_TOKENIZER,
           "--voice-prompt", f"{PROBE_VOICE}.wav", "--voice-prompt-dir", VOICE_DIR,
           "--text-prompt", PROBE_PERSONA, "--input-wav", user_wav,
           "--output-wav", str(out_wav), "--output-text", str(out_json), "--seed", "42"]
    if model_kind == "student":
        cmd += ["--student-config", STUDENT_CONFIG, "--student-checkpoint", STUDENT_EXPORT]
    r = subprocess.run(cmd, cwd=os.path.join(REPO_DIR, "moshi"), capture_output=True, text=True,
                       env=dict(os.environ, PYTHONPATH=REPO_DIR, CUDA_VISIBLE_DEVICES=str(GPUS[0])))
    if r.returncode != 0:
        print(f"   {model_kind} failed on {question!r}: {r.stderr[-400:]}")
        return None
    toks = json.loads(out_json.read_text())
    return "".join(t for t in toks if t not in ("PAD", "EPAD", "BOS", "EOS"))


reports = {}
for kind in PROBE_MODELS:
    answers = {}
    for i, (question, _) in enumerate(KNOWLEDGE_PROBES):
        reply = ask(kind, question, i)
        if reply is not None:
            answers[question] = reply
    reports[kind] = knowledge_probe_report(answers)
    print_knowledge_report(reports[kind], label=kind)

(PROBE_DIR / "report.json").write_text(json.dumps(reports, indent=1), encoding="utf-8")
print(f"\nwritten -> {PROBE_DIR / 'report.json'}")
s = reports.get("student", {}).get("answer_relevance")
t = reports.get("teacher", {}).get("answer_relevance")
if s is not None:
    print(f"\nSTUDENT answer_relevance = {s:.2f}" + (f"   TEACHER = {t:.2f}" if t is not None else ""))
    if s < 0.25:
        print("The student still answers nothing. Check logs/init_fidelity.json first: if top1_agree < 0.2 the "
              "init destroyed the text pathway and no extra training fixes it -- retrain with "
              "--student-config student_ppx_m. If top1_agree is healthy, check that textstep rows appear in "
              "train_log.csv and that text_tokens_total is growing.")
    elif t is not None and s >= 0.6 * t:
        print("The student is in range of the teacher on these probes.")
